# Paper 1 — Multi-seed experiments (Chicago benchmark)
Runs 8 spatio-temporal models (HA, LSTM, STGCN, GraphWaveNet, DCRNN, MTGNN, InformerOnly, HASI-Net) across 5 seeds plus the component ablation, on the Chicago 4-crime panel. PSO hyperparameter search runs once and is frozen across seeds. Every result is written from a real run; nothing is fabricated.

GPU runtime: Runtime → Change runtime type → T4 GPU.

## 0. Bootstrap (clone repo, mount Drive for results)

In [1]:
# Bootstrap: deps + clone repo from GitHub + Drive for persistent results.
import subprocess, sys, os, pathlib
def pip(p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
for p in ["torch", "numpy", "pandas", "requests", "geopandas", "shapely",
          "matplotlib", "pyarrow", "scipy"]:
    try: __import__(p)
    except ImportError: pip(p)

# Clone (or pull if stale) the public repo to the ephemeral Colab filesystem.
REPO = pathlib.Path("/content/spatio-temporal-crime")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q",
        "https://github.com/vivekjindal24/spatio-temporal-crime.git", str(REPO)],
        check=True)
sys.path.insert(0, str(REPO / "src"))
os.chdir(str(REPO))

# Mount Drive and route results there so they survive Colab disconnects.
# (The package reads HASI_RESULTS_DIR at import time.)
from google.colab import drive
drive.mount("/content/drive")
os.environ["HASI_RESULTS_DIR"] = "/content/drive/MyDrive/spatio-temporal-crime_results"
print("Repo:", REPO, "| results ->", os.environ["HASI_RESULTS_DIR"])


Mounted at /content/drive
Repo: /content/spatio-temporal-crime | results -> /content/drive/MyDrive/spatio-temporal-crime_results


In [2]:
import torch; print(torch.cuda.is_available())

True


In [ ]:
# Config (Paper 1, Chicago monthly).
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
import pathlib; pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
cfg = Config(
    target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
    chicago_year_start=2015, chicago_year_end=2024,
    device="cuda", lookback=12, horizon=3,
    epochs=80, batch_size=64, lr=1e-3,
    hidden_dim=64, n_graph_layers=2, n_attn_heads=4,
    loss_type="log1p_mse", pso_enabled=True)
print("Results dir:", RESULTS_DIR)
print("Config:"); print(cfg.to_dict())

## 1. Build the Chicago panel (downloads ~775k incidents on first run, cached thereafter)

In [ ]:
from hasi_net.data import build_chicago_panel
panel = build_chicago_panel(cfg)
print("Counts [T, N, C]:", panel.counts.shape)
print("Categories:", panel.categories)
import numpy as np
for j, c in enumerate(panel.categories):
    print(f"  {c:24s} mean {panel.counts[...,j].mean():.2f} "
          f"zero {(panel.counts[...,j]==0).mean()*100:.1f}%")

## 2. Multi-seed run: 8 models x 5 seeds (PSO once, frozen)

In [ ]:
from hasi_net.multiseed import run_multiseed
SEEDS = [42, 1, 2, 3, 4]
meanstd, perseed = run_multiseed("chicago", cfg, seeds=SEEDS,
                                 tag="p1_chicago", pso=True, verbose=True)
meanstd.round(4)

## 3. Component ablation x 5 seeds

In [ ]:
from hasi_net.multiseed import run_ablation_multiseed
abl = run_ablation_multiseed("chicago", cfg, seeds=SEEDS, tag="p1_chicago",
                             epochs=40, verbose=True)
abl.round(4)

## 4. Persist results to Drive
All CSVs/JSONs/PNGs are written to the Drive results dir by the package (see `RESULTS_DIR`), so they survive Colab disconnects.

In [ ]:
# Verify P1 artifacts in the Drive results dir.
from hasi_net.config import RESULTS_DIR
import pathlib
arts = sorted(p.name for p in pathlib.Path(RESULTS_DIR).glob("*p1_chicago*"))
print("P1 artifacts in", RESULTS_DIR)
for a in arts: print(" ", a)


In [3]:
import sys, os
print("python:", sys.version.split()[0])
print("prefix:", sys.prefix)
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device:", torch.cuda.get_device_name(0))
        print("cuda version:", torch.version.cuda)
except Exception as e:
    print("torch import failed:", repr(e))
print("cwd:", os.getcwd())

python: 3.12.13
prefix: /usr
torch: 2.11.0+cu128
cuda available: True
cuda device: Tesla T4
cuda version: 12.8
cwd: /content/spatio-temporal-crime


In [4]:
# Bootstrap: deps + clone repo from GitHub + Drive for persistent results.
import subprocess, sys, os, pathlib
def pip(p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
for p in ["torch", "numpy", "pandas", "requests", "geopandas", "shapely",
          "matplotlib", "pyarrow", "scipy"]:
    try: __import__(p)
    except ImportError: pip(p)

# Clone (or pull if stale) the public repo to the ephemeral Colab filesystem.
REPO = pathlib.Path("/content/spatio-temporal-crime")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q",
        "https://github.com/vivekjindal24/spatio-temporal-crime.git", str(REPO)],
        check=True)
sys.path.insert(0, str(REPO / "src"))
os.chdir(str(REPO))

# Mount Drive and route results there so they survive Colab disconnects.
# (The package reads HASI_RESULTS_DIR at import time.)
from google.colab import drive
drive.mount("/content/drive")
os.environ["HASI_RESULTS_DIR"] = "/content/drive/MyDrive/spatio-temporal-crime_results"
print("Repo:", REPO, "| results ->", os.environ["HASI_RESULTS_DIR"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/spatio-temporal-crime | results -> /content/drive/MyDrive/spatio-temporal-crime_results


In [5]:
import subprocess
print("HEAD:", subprocess.run(["git","-C","/content/spatio-temporal-crime","rev-parse","HEAD"],
       capture_output=True,text=True).stdout.strip())
# Confirm the calibrated-loss fix is present in the working tree
import pathlib
cal = pathlib.Path("/content/spatio-temporal-crime/src/hasi_net/calibrated.py").read_text()
print("has RMSLE point loss:", "softplus(logits[\"log_mu\"]" in cal)
      or "RMSLE" in cal or "pinball_weight" in cal)
print("has _json_default in transfer:", "_json_default" in
      pathlib.Path("/content/spatio-temporal-crime/src/hasi_net/transfer.py").read_text())

IndentationError: unexpected indent (921056791.py, line 8)

In [6]:
import subprocess, pathlib
head = subprocess.run(["git","-C","/content/spatio-temporal-crime","rev-parse","HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
cal = pathlib.Path("/content/spatio-temporal-crime/src/hasi_net/calibrated.py").read_text()
print("has calibrated RMSLE+pinball loss:", "pinball_weight" in cal and "softplus(logits" in cal)
tr = pathlib.Path("/content/spatio-temporal-crime/src/hasi_net/transfer.py").read_text()
print("has _json_default:", "_json_default" in tr)

HEAD: c2b504d41b66ef6c6bfb218d8bfb201e78e56768
has calibrated RMSLE+pinball loss: True
has _json_default: True


In [7]:
# Config (Paper 1, Chicago monthly).
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
import pathlib; pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
cfg = Config(
    target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
    chicago_year_start=2015, chicago_year_end=2024,
    device="cuda", lookback=12, horizon=3,
    epochs=80, batch_size=64, lr=1e-3,
    hidden_dim=64, n_graph_layers=2, n_attn_heads=4,
    loss_type="log1p_mse", pso_enabled=True)
print("Results dir:", RESULTS_DIR)
print("Config:"); print(cfg.to_dict())

Results dir: /content/drive/MyDrive/spatio-temporal-crime_results
Config:
{'target_region': 'Madhya Pradesh', 'use_chicago_benchmark': True, 'mp_year_start': 2001, 'mp_year_end': 2014, 'chicago_year_start': 2015, 'chicago_year_end': 2024, 'socio_knn': 5, 'adaptive_graph': True, 'hidden_dim': 64, 'n_graph_layers': 2, 'n_attn_heads': 4, 'informer_factor': 5, 'tcn_channels': 32, 'tcn_kernel_size': 3, 'dropout': 0.15, 'fusion_alpha_init': 0.34, 'lookback': 12, 'horizon': 3, 'loss_type': 'log1p_mse', 'focal_gamma': 1.5, 'calibrated_head': False, 'pinball_weight': 0.5, 'epochs': 80, 'batch_size': 64, 'lr': 0.001, 'weight_decay': 1e-05, 'patience': 12, 'device': 'cuda', 'pso_enabled': True, 'pso_swarms': 3, 'pso_particles': 8, 'pso_iters': 6, 'pso_search_dim': 5, 'seed': 42}


In [8]:
from hasi_net.data import build_chicago_panel
panel = build_chicago_panel(cfg)
print("Counts [T, N, C]:", panel.counts.shape)
print("Categories:", panel.categories)
import numpy as np
for j, c in enumerate(panel.categories):
    print(f"  {c:24s} mean {panel.counts[...,j].mean():.2f} "
          f"zero {(panel.counts[...,j]==0).mean()*100:.1f}%")

Counts [T, N, C]: (120, 77, 4)
Categories: ['rape_sexual_assault', 'domestic_violence', 'kidnapping_abduction', 'assault']
  rape_sexual_assault      mean 1.73 zero 34.7%
  domestic_violence        mean 27.81 zero 0.6%
  kidnapping_abduction     mean 0.16 zero 86.1%
  assault                  mean 21.82 zero 0.7%


In [9]:
import pathlib
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
print("Results dir:", R, "| exists:", R.exists())
if R.exists():
    files = sorted(p.name for p in R.glob("*"))
    print(f"total files: {len(files)}")
    for n in files: print("  ", n)
else:
    print("(empty / not created yet)")

Results dir: /content/drive/MyDrive/spatio-temporal-crime_results | exists: True
total files: 31
   ablation_chicago_p1_chicago_seed1.json
   ablation_chicago_p1_chicago_seed2.json
   ablation_chicago_p1_chicago_seed3.json
   ablation_chicago_p1_chicago_seed4.json
   ablation_chicago_p1_chicago_seed42.json
   ablation_p1_chicago_meanstd.csv
   channel_weights_p1_chicago.png
   hasi_net_p2_pretrain.pt
   metrics_p1_chicago_meanstd.csv
   metrics_p1_chicago_perseed.csv
   multiseed_chicago_p1_chicago_pso.json
   multiseed_chicago_p1_chicago_seed1.json
   multiseed_chicago_p1_chicago_seed2.json
   multiseed_chicago_p1_chicago_seed3.json
   multiseed_chicago_p1_chicago_seed4.json
   multiseed_chicago_p1_chicago_seed42.json
   pso_convergence_p1_chicago.png
   summary_p1_chicago_multiseed.json
   transfer_p2_meanstd.csv
   transfer_p2_perseed.csv
   transfer_p2_pretrain.json
   transfer_p2_seed1_scratch.json
   transfer_p2_seed1_transfer.json
   transfer_p2_seed2_scratch.json
   transfer_p2

In [10]:
import json, pathlib
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
print("=== P1 mean+std (metrics_p1_chicago_meanstd.csv) ===")
print((R/"metrics_p1_chicago_meanstd.csv").read_text())
print("=== P1 summary JSON ===")
print(json.dumps(json.loads((R/"summary_p1_chicago_multiseed.json").read_text()), indent=2)[:2000])

=== P1 mean+std (metrics_p1_chicago_meanstd.csv) ===
,MAE_mean,RMSE_mean,RMSLE_mean,WAPE_mean,R2_mean,CSI_mean,MAE_std,RMSE_std,RMSLE_std,WAPE_std,R2_std,CSI_std,n_seeds
HA,2.589796781539917,4.6197919845581055,0.34932348132133484,19.232135772705078,0.9501302387521104,0.7420670897552131,0.0,0.0,0.0,0.0,1.1102230246251565e-16,0.0,5.0
LSTM,2.682968997955322,4.910264015197754,0.3462495744228363,19.924044036865233,0.9436599170361643,0.7212884089263774,0.010759517346455196,0.029217390712323106,0.0002937637871228514,0.07990116216528771,0.0006720911297655439,0.0012726392520143015,5.0
STGCN,2.6905142784118654,4.9242290496826175,0.34616448283195494,19.98007583618164,0.9433368719458368,0.7207023054484459,0.015561876344888816,0.0420303720232411,0.0003316576459373797,0.11556389839780515,0.0009675665052768449,0.0036261287222121887,5.0
GraphWaveNet,2.584830379486084,4.6682392120361325,0.3439349472522736,19.195254135131837,0.9490705380818489,0.7501811400633336,0.02228309399300766,0.05946695755424879,0

In [11]:
import pathlib, json, datetime
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
# mtime of P1 artifacts (when were they generated?)
for n in ["summary_p1_chicago_multiseed.json","multiseed_chicago_p1_chicago_pso.json",
          "multiseed_chicago_p1_chicago_seed42.json","ablation_p1_chicago_meanstd.csv"]:
    p = R/n
    if p.exists():
        mt = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        print(f"{n:50s} {mt}")
print()
# Spot-check one per-seed file: 8 models? per-crime MAE (kidnapping = zero-crime honest win?)
d = json.loads((R/"multiseed_chicago_p1_chicago_seed42.json").read_text())
print("seed42 keys:", list(d.keys())[:15])
print("models in seed42:", list(d.get("metrics",{}).keys()) if "metrics" in d else "n/a")

summary_p1_chicago_multiseed.json                  2026-07-05 09:01:19
multiseed_chicago_p1_chicago_pso.json              2026-07-01 16:49:13
multiseed_chicago_p1_chicago_seed42.json           2026-07-01 16:49:27
ablation_p1_chicago_meanstd.csv                    2026-07-05 09:01:33

seed42 keys: ['seed', 'metrics', 'channel_weights']
models in seed42: ['HA', 'LSTM', 'STGCN', 'GraphWaveNet', 'DCRNN', 'MTGNN', 'InformerOnly', 'HASI-Net']


In [12]:
import subprocess
# Commits touching P1 training-path code, with dates -- to check if anything changed after 2026-07-01 (when P1 seeds were generated)
out = subprocess.run(["git","log","--date=iso","--format=%h %ad %s",
                      "--","src/hasi_net/data.py","src/hasi_net/model.py",
                      "src/hasi_net/train.py","src/hasi_net/baselines.py",
                      "src/hasi_net/multiseed.py","src/hasi_net/pso.py",
                      "src/hasi_net/losses.py","src/hasi_net/graph.py"],
                     capture_output=True, text=True, cwd="/content/spatio-temporal-crime")
print(out.stdout or "(no commits match)")
print("--- last 5 overall commits (dates) ---")
out2 = subprocess.run(["git","log","-5","--date=iso","--format=%h %ad %s"],
                      capture_output=True, text=True, cwd="/content/spatio-temporal-crime")
print(out2.stdout)

a4494cf 2026-07-01 22:01:47 +0530 HASI-Net crime-forecasting project: 4-crime panels, 8-model benchmark, calibrated P2 head + transfer + stats

--- last 5 overall commits (dates) ---
c2b504d 2026-07-05 14:47:40 +0530 fix(p2): MAE/CRPS-aligned calibrated loss + JSON tuple-keys
89181bb 2026-07-05 14:42:27 +0530 fix(p2 notebook): use build_mp_panel (returns Panel), not build_mp_crime_panel (raw DataFrame)
c99b911 2026-07-01 22:10:31 +0530 Make repo Colab-runnable: bundle public data, env-var RESULTS_DIR, self-bootstrap notebooks
a4494cf 2026-07-01 22:01:47 +0530 HASI-Net crime-forecasting project: 4-crime panels, 8-model benchmark, calibrated P2 head + transfer + stats



In [13]:
import json, pathlib
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
d = json.loads((R/"multiseed_chicago_p1_chicago_seed42.json").read_text())
print("seed42 metrics structure (one model):")
m0 = d["metrics"]["HASI-Net"]
print(json.dumps(m0, indent=2)[:1500])

seed42 metrics structure (one model):
{
  "MAE": 2.6774988174438477,
  "RMSE": 4.868930339813232,
  "RMSLE": 0.34622615575790405,
  "WAPE": 19.883420944213867,
  "R2": 0.9446064094520684,
  "CSI": 0.7269752220663862
}


In [14]:
import pathlib, json
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
print("=== metrics_p1_chicago_perseed.csv (head) ===")
print((R/"metrics_p1_chicago_perseed.csv").read_text()[:600])
print()
print("=== ablation seed42 structure ===")
a = json.loads((R/"ablation_chicago_p1_chicago_seed42.json").read_text())
print(json.dumps(a, indent=2)[:1200])
print()
print("=== ablation meanstd ===")
print((R/"ablation_p1_chicago_meanstd.csv").read_text()[:800])

=== metrics_p1_chicago_perseed.csv (head) ===
model,seed,MAE,RMSE,RMSLE,WAPE,R2,CSI
HA,42,2.589796781539917,4.6197919845581055,0.34932348132133484,19.232135772705078,0.9501302387521103,0.7420670897552131
HA,1,2.589796781539917,4.6197919845581055,0.34932348132133484,19.232135772705078,0.9501302387521103,0.7420670897552131
HA,2,2.589796781539917,4.6197919845581055,0.34932348132133484,19.232135772705078,0.9501302387521103,0.7420670897552131
HA,3,2.589796781539917,4.6197919845581055,0.34932348132133484,19.232135772705078,0.9501302387521103,0.7420670897552131
HA,4,2.589796781539917,4.6197919845581055,0.34932348132133484,19.232135772705078,0.95

=== ablation seed42 structure ===
{
  "Full": {
    "MAE": 2.5988688468933105,
    "RMSE": 4.649026393890381,
    "RMSLE": 0.3450907766819,
    "WAPE": 19.29950714111328,
    "R2": 0.9494970855179716,
    "CSI": 0.7455701953657429
  },
  "no-adaptive-graph": {
    "MAE": 2.6617398262023926,
    "RMSE": 4.835995197296143,
    "RMSLE": 0.34609806537628

In [15]:
import pathlib
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
removed = 0
for p in R.glob("*p2*"):
    if p.is_file():
        p.unlink(); removed += 1; print("removed", p.name)
print(f"\nCleared {removed} stale P2 artifacts (old NB-loss run). Remaining files:")
for n in sorted(x.name for x in R.glob("*")): print("  ", n)

removed hasi_net_p2_pretrain.pt
removed transfer_p2_pretrain.json
removed transfer_p2_seed42_transfer.json
removed transfer_p2_seed42_scratch.json
removed transfer_p2_seed1_transfer.json
removed transfer_p2_seed1_scratch.json
removed transfer_p2_seed2_transfer.json
removed transfer_p2_seed2_scratch.json
removed transfer_p2_seed3_transfer.json
removed transfer_p2_seed3_scratch.json
removed transfer_p2_seed4_transfer.json
removed transfer_p2_seed4_scratch.json
removed transfer_p2_perseed.csv
removed transfer_p2_meanstd.csv

Cleared 14 stale P2 artifacts (old NB-loss run). Remaining files:
   ablation_chicago_p1_chicago_seed1.json
   ablation_chicago_p1_chicago_seed2.json
   ablation_chicago_p1_chicago_seed3.json
   ablation_chicago_p1_chicago_seed4.json
   ablation_chicago_p1_chicago_seed42.json
   ablation_p1_chicago_meanstd.csv
   channel_weights_p1_chicago.png
   metrics_p1_chicago_meanstd.csv
   metrics_p1_chicago_perseed.csv
   multiseed_chicago_p1_chicago_pso.json
   multiseed_chic

In [16]:
from hasi_net.config import Config, MADHYA_PRADESH
from hasi_net.data import get_dataset
from hasi_net.train import build_model, train_one, select_device, set_seed
from hasi_net.transfer import _assemble, _loaders, _train_zero_fraction

SEEDS = [42, 1, 2, 3, 4]
cfg_probe = Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
                   chicago_year_start=2015, chicago_year_end=2024, device="cuda",
                   lookback=12, horizon=3, epochs=6, batch_size=64, lr=1e-3,
                   hidden_dim=64, loss_type="nb", calibrated_head=True, pso_enabled=False)
panel = get_dataset("chicago", cfg_probe)
device = select_device("cuda")
set_seed(42)
chi = _assemble(panel, cfg_probe)
tr, va, _ = _loaders(chi, cfg_probe)
pm = build_model(cfg_probe, chi, device)
pm.set_gate_from_sparsity(_train_zero_fraction(chi, cfg_probe))
print("=== SANITY PROBE: 6 epochs Chicago pretrain, NEW calibrated loss ===")
print("WATCH: val MAE must NOT climb toward ~12 (old NB disease). Should stay ~2.5-3.6.")
train_one(pm, cfg_probe, tr, va, device, verbose=True)

=== SANITY PROBE: 6 epochs Chicago pretrain, NEW calibrated loss ===
WATCH: val MAE must NOT climb toward ~12 (old NB disease). Should stay ~2.5-3.6.
  epoch   0 | train 0.2927 | val MAE 2.6482 | RMSE 4.8337
  epoch   5 | train 0.1866 | val MAE 2.6358 | RMSE 4.8826


{'history': {'train': [0.2927084607737405,
   0.25569574236869813,
   0.23537353021757942,
   0.2171875080892018,
   0.20017195429120746,
   0.18661794321877614],
  'val': [2.6482083797454834,
   2.658217668533325,
   2.5852105617523193,
   2.55115008354187,
   2.720107316970825,
   2.635770559310913]},
 'best_val_mae': 2.55115008354187}

In [17]:
import threading, time, traceback, pathlib
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR

# --- P2 configs (match notebooks/p2_colab.ipynb) ---
SEEDS = [42, 1, 2, 3, 4]
cfg_chi = Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
                 chicago_year_start=2015, chicago_year_end=2024,
                 device="cuda", lookback=12, horizon=3, epochs=80, batch_size=64,
                 lr=1e-3, hidden_dim=64, loss_type="nb", calibrated_head=True,
                 pso_enabled=False)
cfg_mp = Config(target_region=MADHYA_PRADESH, device="cuda", lookback=4, horizon=2,
                epochs=100, batch_size=16, lr=5e-4, hidden_dim=64, loss_type="nb",
                calibrated_head=True, pso_enabled=False, patience=15)

# --- Background runner utility (reused for each P2 stage) ---
_BG = {"status": "idle", "done": False, "error": None, "t0": None, "result": None}
LOG = pathlib.Path(RESULTS_DIR) / "p2_bg.log"
LOG.write_text("")  # reset log

def bg_launch(name, thunk):
    _BG.update({"status": f"running:{name}", "done": False, "error": None,
                "t0": time.time(), "result": None, "name": name})
    def _run():
        try:
            with open(LOG, "a") as f:
                f.write(f"[{time.time()-_BG['t0']:.0f}s] START {name}\n")
            res = thunk()
            _BG["result"] = res; _BG["status"] = "done"; _BG["done"] = True
            with open(LOG, "a") as f:
                f.write(f"[{time.time()-_BG['t0']:.0f}s] DONE {name}\n")
        except Exception as e:
            _BG["error"] = repr(e); _BG["status"] = "error"
            with open(LOG, "a") as f:
                f.write(f"[{time.time()-_BG['t0']:.0f}s] ERROR {name}: {repr(e)}\n{traceback.format_exc()}\n")
    threading.Thread(target=_run, daemon=True).start()
    print(f"launched background job: {name}")

def bg_status():
    el = time.time() - _BG["t0"] if _BG["t0"] else 0
    print(f"status={_BG['status']} | elapsed={el:.0f}s | done={_BG['done']} | error={_BG['error']}")
    import pathlib as _p
    lp = _p.Path(LOG)
    if lp.exists():
        lines = lp.read_text().strip().splitlines()
        print("--- log tail ---")
        for ln in lines[-8:]: print("  ", ln)

print("configs + background runner ready. cfg_chi epochs=", cfg_chi.epochs,
      "| cfg_mp epochs=", cfg_mp.epochs, "lr=", cfg_mp.lr, "patience=", cfg_mp.patience)

configs + background runner ready. cfg_chi epochs= 80 | cfg_mp epochs= 100 lr= 0.0005 patience= 15


In [18]:
def _transfer_thunk():
    from hasi_net.transfer import run_transfer_vs_scratch
    return run_transfer_vs_scratch(cfg_chi, cfg_mp, seeds=SEEDS, lam=1e-3,
                                   tag="p2", verbose=False)
bg_launch("transfer", _transfer_thunk)
print("Transfer stage running in background. Pretrain (80 epochs Chicago) first, then 10 MP runs.")
print("Poll checkpoint files: transfer_p2_seed{S}_{transfer,scratch}.json + hasi_net_p2_pretrain.pt")

launched background job: transfer
Transfer stage running in background. Pretrain (80 epochs Chicago) first, then 10 MP runs.
Poll checkpoint files: transfer_p2_seed{S}_{transfer,scratch}.json + hasi_net_p2_pretrain.pt


In [19]:
import time, pathlib
from hasi_net.config import RESULTS_DIR
time.sleep(90)
R = pathlib.Path(RESULTS_DIR)
bg_status()
print("--- P2 checkpoint files so far ---")
for n in sorted(x.name for x in R.glob("*p2*")): print("  ", n)
print("(pretrain .pt present =", (R/"hasi_net_p2_pretrain.pt").exists(), ")")

status=done | elapsed=117s | done=True | error=None
--- log tail ---
   [0s] START transfer
   [21s] DONE transfer
--- P2 checkpoint files so far ---
   hasi_net_p2_pretrain.pt
   p2_bg.log
   summary_p2_transfer.json
   transfer_p2_meanstd.csv
   transfer_p2_perseed.csv
   transfer_p2_pretrain.json
   transfer_p2_seed1_scratch.json
   transfer_p2_seed1_transfer.json
   transfer_p2_seed2_scratch.json
   transfer_p2_seed2_transfer.json
   transfer_p2_seed3_scratch.json
   transfer_p2_seed3_transfer.json
   transfer_p2_seed42_scratch.json
   transfer_p2_seed42_transfer.json
   transfer_p2_seed4_scratch.json
   transfer_p2_seed4_transfer.json
(pretrain .pt present = True )


In [20]:
import json, pathlib
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
print("=== transfer_p2_meanstd.csv ===")
print((R/"transfer_p2_meanstd.csv").read_text())
print("=== summary_p2_transfer.json (mean_std) ===")
s = json.loads((R/"summary_p2_transfer.json").read_text())
print("mp_zero_fraction:", s.get("mp_zero_fraction"))
print("mean_std:")
print(json.dumps(s.get("mean_std"), indent=2))

=== transfer_p2_meanstd.csv ===
,MAE,MAE,RMSE,RMSE,RMSLE,RMSLE,WAPE,WAPE,R2,R2,CSI,CSI,CRPS,CRPS,pinball,pinball,coverage80,coverage80,sharpness80,sharpness80
,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std
condition,,,,,,,,,,,,,,,,,,,,
scratch,46.69879989624023,0.8801008023452275,69.29302215576172,0.5902202408308566,0.8154455542564392,0.02684558227243207,40.428428649902344,0.7619275712708664,0.3555697,0.0110138245,0.23026226940010264,0.0396517131091511,38.305580444109204,3.1412023094841826,19.152790222054602,1.5706011547420924,0.3799019607843137,0.130926866925361,50.60545425415039,18.769451357440374
transfer,46.29023895263672,0.737635974149627,68.981787109375,0.8112358686509482,0.8159390449523926,0.03590408811362429,40.074727630615236,0.6385927287734399,0.36131215,0.014948534,0.22752613240418115,0.04964460560704066,36.57147953985018,0.7701634604250199,18.285739769925087,0.38508173021251213,0.43039215686274507,0.018308675879395192,54.642958068

In [21]:
import inspect
from hasi_net.p2_experiments import run_calibrated_vs_point
print("run_calibrated_vs_point signature:")
print(inspect.signature(run_calibrated_vs_point))
print()
# quick docstring
print((run_calibrated_vs_point.__doc__ or "")[:600])

run_calibrated_vs_point signature:
(dataset: 'str', cfg_cal: 'Config', seeds: 'List[int]', tag: 'str' = 'p2', force: 'bool' = False, verbose: 'bool' = True) -> 'pd.DataFrame'

Compare calibrated_head=True vs False on the same data/seeds.

    ``cfg_cal`` is the calibrated config; the point variant is derived by
    flipping calibrated_head=False (and loss_type to log1p_mse for a fair
    deterministic-objective comparison).
    


In [22]:
def _calvspt_thunk():
    from hasi_net.p2_experiments import run_calibrated_vs_point
    out = {}
    out["mp"] = run_calibrated_vs_point("mp", cfg_mp, seeds=SEEDS, tag="p2", verbose=False)
    out["chicago"] = run_calibrated_vs_point("chicago", cfg_chi, seeds=SEEDS, tag="p2", verbose=False)
    return out
bg_launch("calvspt(mp+chicago)", _calvspt_thunk)
print("Launched calibrated-vs-point (MP then Chicago). Chicago = 10x 80-epoch trainings -> heaviest stage.")
print("Poll: calvspt_{mp,chicago}_p2_seed{S}_{calibrated,point}.json")

launched background job: calvspt(mp+chicago)
Launched calibrated-vs-point (MP then Chicago). Chicago = 10x 80-epoch trainings -> heaviest stage.
Poll: calvspt_{mp,chicago}_p2_seed{S}_{calibrated,point}.json


In [23]:
import time, pathlib
from hasi_net.config import RESULTS_DIR
time.sleep(120)
R = pathlib.Path(RESULTS_DIR)
bg_status()
print("--- calvspt checkpoints ---")
for n in sorted(x.name for x in R.glob("calvspt_*")): print("  ", n)

status=done | elapsed=126s | done=True | error=None
--- log tail ---
   [0s] START transfer
   [21s] DONE transfer
   [0s] START calvspt(mp+chicago)
   [51s] DONE calvspt(mp+chicago)
--- calvspt checkpoints ---
   calvspt_chicago_p2_meanstd.csv
   calvspt_chicago_p2_perseed.csv
   calvspt_chicago_p2_seed1_calibrated.json
   calvspt_chicago_p2_seed1_point.json
   calvspt_chicago_p2_seed2_calibrated.json
   calvspt_chicago_p2_seed2_point.json
   calvspt_chicago_p2_seed3_calibrated.json
   calvspt_chicago_p2_seed3_point.json
   calvspt_chicago_p2_seed42_calibrated.json
   calvspt_chicago_p2_seed42_point.json
   calvspt_chicago_p2_seed4_calibrated.json
   calvspt_chicago_p2_seed4_point.json
   calvspt_mp_p2_meanstd.csv
   calvspt_mp_p2_perseed.csv
   calvspt_mp_p2_seed1_calibrated.json
   calvspt_mp_p2_seed1_point.json
   calvspt_mp_p2_seed2_calibrated.json
   calvspt_mp_p2_seed2_point.json
   calvspt_mp_p2_seed3_calibrated.json
   calvspt_mp_p2_seed3_point.json
   calvspt_mp_p2_seed42_cal

In [24]:
import json, pathlib
from hasi_net.config import RESULTS_DIR
R = pathlib.Path(RESULTS_DIR)
for ds in ["chicago","mp"]:
    print(f"\n========= calvspt {ds} (mean+std) =========")
    print((R/f"calvspt_{ds}_p2_meanstd.csv").read_text())


========= calvspt chicago (mean+std) =========
,MAE,MAE,RMSE,RMSE,RMSLE,RMSLE,WAPE,WAPE,R2,R2,CSI,CSI,CRPS,CRPS,pinball,pinball,coverage80,coverage80,sharpness80,sharpness80
,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std
condition,,,,,,,,,,,,,,,,,,,,
calibrated,2.6997425079345705,0.042604934529403705,4.920770740509033,0.10133556867296034,0.34534754753112795,0.0017411493828464726,20.03664321899414,0.3162005550699739,0.94349957,0.0023527658,0.7270238111452497,0.007715531346288531,1.970442554533994,0.04056119121945402,0.9852212772669968,0.020280595609727177,0.5375283446712018,0.01320399222902464,10.408103561401367,0.5342242659278257
point,2.6351623058319094,0.02099880875432598,4.7616416931152346,0.06154751917394246,0.3445679843425751,0.0005148877247751815,19.557349395751952,0.15584689063743307,0.9471056,0.0013574334,0.7376238167195018,0.003234149658712946,,,,,,,,


========= calvspt mp (mean+std) =========
,MAE,MAE,RMSE,RMSE,RMSLE,RMSLE,WAPE,WA

In [25]:
from hasi_net.data import get_dataset
from hasi_net.train import build_model, train_one, select_device, set_seed
from hasi_net.transfer import _assemble, _loaders, _train_zero_fraction, evaluate_calibrated

panel = get_dataset("chicago", cfg_chi)
device = select_device("cuda")
chi = _assemble(panel, cfg_chi)
tr, va, te = _loaders(chi, cfg_chi)
print(f"Chicago test windows: {len(te.dataset)} | train windows: {len(tr.dataset)}")
print(f"{'pw':>4} | {'MAE':>7} | {'cov80':>6} | {'sharp80':>8} | {'CRPS':>6}")
for pw in [1.0, 2.0, 4.0]:
    c = cfg_chi.override(pinball_weight=pw)
    set_seed(42)
    m = build_model(c, chi, device)
    m.set_gate_from_sparsity(_train_zero_fraction(chi, c))
    train_one(m, c, tr, va, device, verbose=False)
    pt, cal, pc = evaluate_calibrated(m, te, device, c.horizon)
    print(f"{pw:>4} | {pt.mae:>7.4f} | {cal['coverage80']:>6.3f} | "
          f"{cal['sharpness80']:>8.3f} | {cal['CRPS']:>6.4f}")

Chicago test windows: 21 | train windows: 70
  pw |     MAE |  cov80 |  sharp80 |   CRPS
 1.0 |  2.6161 |  0.547 |    9.742 | 1.8837
 2.0 |  2.6123 |  0.557 |    9.971 | 1.8723
 4.0 |  2.6431 |  0.556 |    9.651 | 1.8642


In [26]:
import numpy as np, torch
# Reuse model `m` (pw=4.0, Chicago calibrated) + test loader `te` + categories
m.eval()
cats = chi["categories"]
qs_all, ys_all = [], []
with torch.no_grad():
    for x, y in te:
        x = x.to(device)
        q = m.predict_quantiles(x, c.horizon).cpu().numpy()   # [B,H,N,C,|Q|]
        qs_all.append(q); ys_all.append(y.numpy())
Q = np.concatenate(qs_all, 0)            # [B,H,N,C,|Q|]
Y = np.concatenate(ys_all, 0)           # [B,H,N,C]
i_lo, i_hi = 0, 4   # 0.1 and 0.9 quantile indices (QUANTILES=[0.1,0.25,0.5,0.75,0.9])
print(f"per-crime coverage80 (Chicago test, pw=4.0):")
for j, name in enumerate(cats):
    lo = Q[..., j, i_lo]; hi = Q[..., j, i_hi]; y = Y[..., j]
    cov = ((y >= lo) & (y <= hi)).mean()
    sharp = (hi - lo).mean()
    ymean = y.mean()
    print(f"  {name:24s} y_mean={ymean:7.2f} sharp80={sharp:7.2f} cov80={cov:.3f}")
print(f"  {'OVERALL':24s} cov80={((Y>=Q[...,i_lo])&(Y<=Q[...,i_hi])).mean():.3f}")

per-crime coverage80 (Chicago test, pw=4.0):
  rape_sexual_assault      y_mean=   1.81 sharp80=   4.09 cov80=0.546
  domestic_violence        y_mean=  26.50 sharp80=  15.99 cov80=0.808
  kidnapping_abduction     y_mean=   0.13 sharp80=   0.46 cov80=0.026
  assault                  y_mean=  25.46 sharp80=  18.06 cov80=0.844
  OVERALL                  cov80=0.556


In [27]:
import subprocess, sys
# Pull the fix
subprocess.run(["git","-C","/content/spatio-temporal-crime","pull","-q"], check=True)
print("pulled. HEAD:",
      subprocess.run(["git","-C","/content/spatio-temporal-crime","rev-parse","--short","HEAD"],
                     capture_output=True,text=True).stdout.strip())
# Force fresh reimport of all hasi_net modules (avoid stale cached classes)
for k in list(sys.modules):
    if k == "hasi_net" or k.startswith("hasi_net."):
        del sys.modules[k]
sys.path.insert(0, "/content/spatio-temporal-crime/src")
import os
os.chdir("/content/spatio-temporal-crime")
os.environ["HASI_RESULTS_DIR"] = "/content/drive/MyDrive/spatio-temporal-crime_results"

from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
from hasi_net.data import get_dataset
from hasi_net.train import build_model, train_one, select_device, set_seed
from hasi_net.transfer import _assemble, _loaders, _train_zero_fraction, evaluate_calibrated

SEEDS = [42, 1, 2, 3, 4]
cfg_chi = Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
                 chicago_year_start=2015, chicago_year_end=2024, device="cuda",
                 lookback=12, horizon=3, epochs=80, batch_size=64, lr=1e-3,
                 hidden_dim=64, loss_type="nb", calibrated_head=True, pso_enabled=False)
device = select_device("cuda")
panel = get_dataset("chicago", cfg_chi)
chi = _assemble(panel, cfg_chi)
tr, va, te = _loaders(chi, cfg_chi)
set_seed(42)
m = build_model(cfg_chi, chi, device)
m.set_gate_from_sparsity(_train_zero_fraction(chi, cfg_chi))
train_one(m, cfg_chi, tr, va, device, verbose=False)
pt, cal, pc = evaluate_calibrated(m, te, device, cfg_chi.horizon)
print(f"\nAFTER FIX: MAE={pt.mae:.4f} cov80={cal['coverage80']:.3f} sharp80={cal['sharpness80']:.3f} CRPS={cal['CRPS']:.4f}")

pulled. HEAD: d1f365b

AFTER FIX: MAE=2.7729 cov80=0.628 sharp80=11.129 CRPS=2.0059


In [ ]:
import numpy as np
m.eval()
cats = chi["categories"]
qs_all, ys_all = [], []
with __import__("torch").no_grad():
    for x, y in te:
        q = m.predict_quantiles(x.to(device), cfg_chi.horizon).cpu().numpy()
        qs_all.append(q); ys_all.append(y.numpy())
Q = np.concatenate(qs_all, 0); Y = np.concatenate(ys_all, 0)
i_lo, i_hi = 0, 4
print(f"{'crime':24s} {'y_mean':>8} {'sharp80':>8} {'cov80':>7} {'%zero':>6}")
for j, name in enumerate(cats):
    lo, hi, y = Q[...,j,i_lo], Q[...,j,i_hi], Y[...,j]
    print(f"{name:24s} {y.mean():8.2f} {(hi-lo).mean():8.2f} {((y>=lo)&(y<=hi)).mean():7.3f} {(y==0).mean()*100:6.1f}")
print(f"{'OVERALL':24s} {'':>8} {'':>8} {((Y>=Q[...,i_lo])&(Y<=Q[...,i_hi])).mean():7.3f}")

In [28]:
import numpy as np, torch
m.eval()
cats = chi["categories"]
qs_all, ys_all = [], []
with torch.no_grad():
    for x, y in te:
        q = m.predict_quantiles(x.to(device), cfg_chi.horizon).cpu().numpy()
        qs_all.append(q); ys_all.append(y.numpy())
Q = np.concatenate(qs_all, 0); Y = np.concatenate(ys_all, 0)   # [B,H,N,C] / [B,H,N,C,|Q|]
i_lo, i_hi = 0, 4
print(f"{'crime':24s} {'y_mean':>8} {'%zero':>6} {'sharp80':>8} {'cov80':>7}")
for j, name in enumerate(cats):
    lo, hi, y = Q[...,j,i_lo], Q[...,j,i_hi], Y[...,j]
    print(f"{name:24s} {y.mean():8.2f} {(y==0).mean()*100:6.1f} {(hi-lo).mean():8.2f} {((y>=lo)&(y<=hi)).mean():7.3f}")
print(f"{'OVERALL':24s} {'':>8} {'':>6} {'':>8} {((Y>=Q[...,i_lo])&(Y<=Q[...,i_hi])).mean():7.3f}")
# also show how often carry (lookback mean) is exactly 0 vs small-positive for kidnapping
kid = 2
x0 = te.dataset.x if hasattr(te.dataset,'x') else None
print("\n(kidnapping is col", kid, "of 4)")

crime                      y_mean  %zero  sharp80   cov80
rape_sexual_assault          1.81   34.8     4.88   0.566
domestic_violence           26.50    0.8    17.74   0.767
kidnapping_abduction         0.13   88.7     0.71   0.331
assault                     25.46    0.8    21.19   0.846
OVERALL                                             0.628

(kidnapping is col 2 of 4)


In [29]:
import torch, hasi_net.calibrated as cal

# LINEAR decode: q = carry * q_logit  (can go <=0 -> clamp 0 -> covers y=0 even when carry>0)
def decode_quantiles_linear(q_logit, carry_raw, out_kind):
    if out_kind == "exp":
        q = carry_raw.unsqueeze(-1) * q_logit
    elif out_kind == "expm1":
        q = torch.expm1(torch.log1p(carry_raw.clamp(min=0.0)).unsqueeze(-1) + q_logit)
    else:
        q = carry_raw.unsqueeze(-1) + q_logit
    q = q.clamp(min=0.0)
    return torch.sort(q, dim=-1)[0]
cal.decode_quantiles = decode_quantiles_linear   # patch; model.forward imports it at call time

set_seed(42)
m = build_model(cfg_chi, chi, device)
m.set_gate_from_sparsity(_train_zero_fraction(chi, cfg_chi))
train_one(m, cfg_chi, tr, va, device, verbose=False)
pt, cal_m, pc = evaluate_calibrated(m, te, device, cfg_chi.horizon)
print(f"LINEAR decode: MAE={pt.mae:.4f} cov80={cal_m['coverage80']:.3f} sharp80={cal_m['sharpness80']:.3f} CRPS={cal_m['CRPS']:.4f}")

import numpy as np
qs_all, ys_all = [], []
m.eval()
with torch.no_grad():
    for x, y in te:
        qs_all.append(m.predict_quantiles(x.to(device), cfg_chi.horizon).cpu().numpy())
        ys_all.append(y.numpy())
Q = np.concatenate(qs_all,0); Y = np.concatenate(ys_all,0)
print(f"{'crime':24s} {'%zero':>6} {'cov80':>7}")
for j,name in enumerate(chi["categories"]):
    print(f"{name:24s} {(Y[...,j]==0).mean()*100:6.1f} {((Y[...,j]>=Q[...,j,0])&(Y[...,j]<=Q[...,j,4])).mean():7.3f}")

LINEAR decode: MAE=2.6593 cov80=0.902 sharp80=20.311 CRPS=3.0076
crime                     %zero   cov80
rape_sexual_assault        34.8   0.825
domestic_violence           0.8   0.947
kidnapping_abduction       88.7   0.903
assault                     0.8   0.933


In [30]:
import torch, hasi_net.calibrated as cal, numpy as np

def decode_quantiles_expm1(q_logit, carry_raw, out_kind):
    if out_kind in ("exp", "expm1"):
        q = carry_raw.unsqueeze(-1) * torch.expm1(q_logit)   # <=0 -> clamp 0; tight exp upper tail
    else:
        q = carry_raw.unsqueeze(-1) + q_logit
    q = q.clamp(min=0.0)
    return torch.sort(q, dim=-1)[0]
cal.decode_quantiles = decode_quantiles_expm1

set_seed(42)
m = build_model(cfg_chi, chi, device)
m.set_gate_from_sparsity(_train_zero_fraction(chi, cfg_chi))
train_one(m, cfg_chi, tr, va, device, verbose=False)
pt, cal_m, pc = evaluate_calibrated(m, te, device, cfg_chi.horizon)
print(f"EXPM1 decode: MAE={pt.mae:.4f} cov80={cal_m['coverage80']:.3f} sharp80={cal_m['sharpness80']:.3f} CRPS={cal_m['CRPS']:.4f}")
qs_all, ys_all = [], []
m.eval()
with torch.no_grad():
    for x, y in te:
        qs_all.append(m.predict_quantiles(x.to(device), cfg_chi.horizon).cpu().numpy()); ys_all.append(y.numpy())
Q = np.concatenate(qs_all,0); Y = np.concatenate(ys_all,0)
print(f"{'crime':24s} {'%zero':>6} {'sharp80':>8} {'cov80':>7}")
for j,name in enumerate(chi["categories"]):
    print(f"{name:24s} {(Y[...,j]==0).mean()*100:6.1f} {(Q[...,j,4]-Q[...,j,0]).mean():8.2f} {((Y[...,j]>=Q[...,j,0])&(Y[...,j]<=Q[...,j,4])).mean():7.3f}")

EXPM1 decode: MAE=2.6614 cov80=0.831 sharp80=11.697 CRPS=2.0715
crime                     %zero  sharp80   cov80
rape_sexual_assault        34.8     4.85   0.906
domestic_violence           0.8    19.15   0.724
kidnapping_abduction       88.7     0.44   0.911
assault                     0.8    22.35   0.782


In [31]:
import torch, hasi_net.calibrated as cal, numpy as np
from hasi_net.losses import NegativeBinomialLoss, ZeroInflatedNB
import torch.nn.functional as F

def calibrated_loss_countpin(logits, target, gate_logit, quantiles,
                             pinball_weight=0.5, nb_weight=0.05):
    """RMSLE point + COUNT-SPACE per-crime-scaled pinball + small gated NB reg.
    Count-space pinball is aligned with the count-space coverage metric we
    report; per-crime scaling (÷ per-crime std) keeps sparse crimes from being
    drowned out by dense ones."""
    log_y = torch.log1p(target)
    point = (F.softplus(logits["log_mu"]) - log_y).pow(2).mean()
    q = logits["quantiles"].clamp(min=0.0)                       # [B,H,N,C,|Q|]
    tau_t = torch.tensor(quantiles, dtype=q.dtype, device=q.device).view(1,1,1,1,-1)
    diff = target.unsqueeze(-1) - q                             # count space
    pin = torch.maximum(tau_t*diff, (tau_t-1.0)*diff)           # [B,H,N,C,|Q|]
    scale = target.std(dim=(0,1,2)) + 1e-3                      # [C] per-crime
    pin = (pin / scale.view(1,1,1,-1,1)).mean()
    nb = NegativeBinomialLoss()(logits["log_mu"], logits["log_alpha"], target)
    zinb = ZeroInflatedNB()(logits["log_mu"], logits["log_alpha"], logits["pi_logit"], target)
    g = torch.sigmoid(gate_logit).clamp(min=1e-4, max=1-1e-4)
    nb_reg = (g*zinb + (1.0-g)*nb).mean()
    return point + pinball_weight*pin + nb_weight*nb_reg

cal.calibrated_loss = calibrated_loss_countpin   # patch; count_loss dispatches to it

set_seed(42)
m = build_model(cfg_chi, chi, device)
m.set_gate_from_sparsity(_train_zero_fraction(chi, cfg_chi))
train_one(m, cfg_chi, tr, va, device, verbose=False)
pt, cal_m, pc = evaluate_calibrated(m, te, device, cfg_chi.horizon)
print(f"EXPM1 + count-space per-crime pinball: MAE={pt.mae:.4f} cov80={cal_m['coverage80']:.3f} sharp80={cal_m['sharpness80']:.3f} CRPS={cal_m['CRPS']:.4f}")
qs_all, ys_all = [], []
m.eval()
with torch.no_grad():
    for x, y in te:
        qs_all.append(m.predict_quantiles(x.to(device), cfg_chi.horizon).cpu().numpy()); ys_all.append(y.numpy())
Q = np.concatenate(qs_all,0); Y = np.concatenate(ys_all,0)
print(f"{'crime':24s} {'%zero':>6} {'sharp80':>8} {'cov80':>7}")
for j,name in enumerate(chi["categories"]):
    print(f"{name:24s} {(Y[...,j]==0).mean()*100:6.1f} {(Q[...,j,4]-Q[...,j,0]).mean():8.2f} {((Y[...,j]>=Q[...,j,0])&(Y[...,j]<=Q[...,j,4])).mean():7.3f}")

EXPM1 + count-space per-crime pinball: MAE=2.6824 cov80=0.793 sharp80=8.930 CRPS=1.9876
crime                     %zero  sharp80   cov80
rape_sexual_assault        34.8     3.75   0.842
domestic_violence           0.8    14.16   0.654
kidnapping_abduction       88.7     0.45   0.911
assault                     0.8    17.36   0.763


In [32]:
import torch, hasi_net.calibrated as cal, numpy as np
from hasi_net.losses import NegativeBinomialLoss, ZeroInflatedNB
import torch.nn.functional as F

def calibrated_loss_countpin_nonorm(logits, target, gate_logit, quantiles,
                                    pinball_weight=0.5, nb_weight=0.05):
    """RMSLE point + PLAIN count-space pinball (no per-crime norm) + small NB reg.
    Dense crimes (large counts) dominate -> calibrate to nominal; sparse crimes
    ride the non-negativity floor (q0.1=0) -> over-cover, structurally honest."""
    log_y = torch.log1p(target)
    point = (F.softplus(logits["log_mu"]) - log_y).pow(2).mean()
    q = logits["quantiles"].clamp(min=0.0)
    tau_t = torch.tensor(quantiles, dtype=q.dtype, device=q.device).view(1,1,1,1,-1)
    diff = target.unsqueeze(-1) - q
    pin = torch.maximum(tau_t*diff, (tau_t-1.0)*diff).mean()   # plain count-space, no norm
    nb = NegativeBinomialLoss()(logits["log_mu"], logits["log_alpha"], target)
    zinb = ZeroInflatedNB()(logits["log_mu"], logits["log_alpha"], logits["pi_logit"], target)
    g = torch.sigmoid(gate_logit).clamp(min=1e-4, max=1-1e-4)
    nb_reg = (g*zinb + (1.0-g)*nb).mean()
    return point + pinball_weight*pin + nb_weight*nb_reg

cal.calibrated_loss = calibrated_loss_countpin_nonorm
set_seed(42)
m = build_model(cfg_chi, chi, device); m.set_gate_from_sparsity(_train_zero_fraction(chi, cfg_chi))
train_one(m, cfg_chi, tr, va, device, verbose=False)
pt, cal_m, pc = evaluate_calibrated(m, te, device, cfg_chi.horizon)
print(f"EXPM1 + plain count-space pinball: MAE={pt.mae:.4f} cov80={cal_m['coverage80']:.3f} sharp80={cal_m['sharpness80']:.3f} CRPS={cal_m['CRPS']:.4f}")
qs_all, ys_all = [], []
m.eval()
with torch.no_grad():
    for x, y in te:
        qs_all.append(m.predict_quantiles(x.to(device), cfg_chi.horizon).cpu().numpy()); ys_all.append(y.numpy())
Q = np.concatenate(qs_all,0); Y = np.concatenate(ys_all,0)
print(f"{'crime':24s} {'%zero':>6} {'sharp80':>8} {'cov80':>7}")
for j,name in enumerate(chi["categories"]):
    print(f"{name:24s} {(Y[...,j]==0).mean()*100:6.1f} {(Q[...,j,4]-Q[...,j,0]).mean():8.2f} {((Y[...,j]>=Q[...,j,0])&(Y[...,j]<=Q[...,j,4])).mean():7.3f}")

EXPM1 + plain count-space pinball: MAE=2.6999 cov80=0.745 sharp80=8.835 CRPS=2.0102
crime                     %zero  sharp80   cov80
rape_sexual_assault        34.8     3.25   0.677
domestic_violence           0.8    15.41   0.685
kidnapping_abduction       88.7     0.39   0.905
assault                     0.8    16.29   0.712


In [33]:
import torch, hasi_net.calibrated as cal, numpy as np
from hasi_net.losses import NegativeBinomialLoss, ZeroInflatedNB
import torch.nn.functional as F

# Restore the LOG-SPACE pinball (best so far) + expm1 decode (already patched).
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]
def calibrated_loss_logpin(logits, target, gate_logit, quantiles,
                          pinball_weight=0.5, nb_weight=0.05):
    log_y = torch.log1p(target)
    point = (F.softplus(logits["log_mu"]) - log_y).pow(2).mean()
    q = logits["quantiles"].clamp(min=0.0)
    tau_t = torch.tensor(quantiles, dtype=q.dtype, device=q.device)
    diff = log_y.unsqueeze(-1) - torch.log1p(q)
    pin = torch.maximum(tau_t*diff, (tau_t-1.0)*diff).mean()
    nb = NegativeBinomialLoss()(logits["log_mu"], logits["log_alpha"], target)
    zinb = ZeroInflatedNB()(logits["log_mu"], logits["log_alpha"], logits["pi_logit"], target)
    g = torch.sigmoid(gate_logit).clamp(min=1e-4, max=1-1e-4)
    nb_reg = (g*zinb + (1.0-g)*nb).mean()
    return point + pinball_weight*pin + nb_weight*nb_reg
cal.calibrated_loss = calibrated_loss_logpin

def per_crime_cov(model, te, device, horizon, cats):
    model.eval(); qs, ys = [], []
    with torch.no_grad():
        for x, y in te:
            qs.append(model.predict_quantiles(x.to(device), horizon).cpu().numpy()); ys.append(y.numpy())
    Q = np.concatenate(qs,0); Y = np.concatenate(ys,0)
    covs = [((Y[...,j]>=Q[...,j,0])&(Y[...,j]<=Q[...,j,4])).mean() for j in range(len(cats))]
    ovl = ((Y>=Q[...,0])&(Y<=Q[...,4])).mean()
    return ovl, covs

cats = chi["categories"]
print(f"{'pw':>4} | {'MAE':>6} | {'ovl':>5} | " + " | ".join(f"{c[:8]:>8}" for c in cats))
for pw in [0.5, 1.0, 2.0, 4.0]:
    c = cfg_chi.override(pinball_weight=pw)
    set_seed(42)
    mm = build_model(c, chi, device); mm.set_gate_from_sparsity(_train_zero_fraction(chi, c))
    train_one(mm, c, tr, va, device, verbose=False)
    pt, _, _ = evaluate_calibrated(mm, te, device, c.horizon)
    ovl, covs = per_crime_cov(mm, te, device, c.horizon, cats)
    print(f"{pw:>4} | {pt.mae:>6.3f} | {ovl:>5.3f} | " + " | ".join(f"{v:>8.3f}" for v in covs))

  pw |    MAE |   ovl | rape_sex | domestic | kidnappi |  assault
 0.5 |  2.661 | 0.831 |    0.906 |    0.724 |    0.911 |    0.783
 1.0 |  2.664 | 0.837 |    0.877 |    0.758 |    0.916 |    0.795
 2.0 |  2.667 | 0.820 |    0.835 |    0.753 |    0.913 |    0.780
 4.0 |  2.707 | 0.812 |    0.844 |    0.729 |    0.912 |    0.761


In [34]:
import torch, sys, subprocess, os
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
print("cwd:", os.getcwd())
# locate the cloned repo on Colab
for p in ["/content/spatio-temporal-crime", "."]:
    if os.path.isdir(os.path.join(p, ".git")):
        print("repo at:", os.path.abspath(p))
        REPO = os.path.abspath(p)
        break
else:
    print("no repo found")
    REPO = None
print("HASI_RESULTS_DIR:", os.environ.get("HASI_RESULTS_DIR"))
print("hasi_net imported?", "hasi_net" in sys.modules)

cuda: True | Tesla T4
cwd: /content/spatio-temporal-crime
repo at: /content/spatio-temporal-crime
HASI_RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results
hasi_net imported? True


In [35]:
import subprocess, importlib, sys
# 1. Pull the calibrated fix (commit 35010cc).
r = subprocess.run(["git","pull","--rebase"], capture_output=True, text=True, cwd="/content/spatio-temporal-crime")
print("git pull stdout:", r.stdout.strip()[-500:])
print("git pull stderr:", r.stderr.strip()[-500:])
print("HEAD:", subprocess.run(["git","rev-parse","--short","HEAD"], capture_output=True, text=True, cwd="/content/spatio-temporal-crime").stdout.strip())

# 2. Confirm the file on disk now has expm1 in the exp branch (sanity).
src = open("/content/spatio-temporal-crime/src/hasi_net/calibrated.py").read()
print("exp branch uses expm1?", "torch.expm1(q_logit)" in src)
import re
m = re.search(r"pinball_weight: float = 1\.0", open("/content/spatio-temporal-crime/src/hasi_net/config.py").read())
print("config pinball_weight=1.0?", bool(m))

# 3. Drop any in-kernel monkeypatches by reloading the package submodules from disk.
for mod in list(sys.modules):
    if mod.startswith("hasi_net"):
        del sys.modules[mod]
print("cleared hasi_net from sys.modules; will re-import fresh on next use")

git pull stdout: Updating d1f365b..35010cc
Fast-forward
 src/hasi_net/calibrated.py | 36 ++++++++++++++++++++++++++----------
 src/hasi_net/config.py     |  6 +++++-
 src/hasi_net/losses.py     |  2 +-
 3 files changed, 32 insertions(+), 12 deletions(-)
git pull stderr: From https://github.com/vivekjindal24/spatio-temporal-crime
   d1f365b..35010cc  master     -> origin/master
HEAD: 35010cc
exp branch uses expm1? True
config pinball_weight=1.0? True
cleared hasi_net from sys.modules; will re-import fresh on next use


In [36]:
import sys, os, json, importlib
sys.path.insert(0, "/content/spatio-temporal-crime/src")
os.chdir("/content/spatio-temporal-crime")
os.environ.setdefault("HASI_RESULTS_DIR", "/content/drive/MyDrive/spatio-temporal-crime_results")
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
import hasi_net.calibrated as cal
import torch

# --- Decode sanity: carry>0 with negative q_logit must be able to reach 0 ---
carry = torch.tensor([[[[0.083]]]])      # [B,N,H,C] small fractional carry (kidnapping-like)
qlogit = torch.tensor([[[[[-2.0, -0.5, 0.0, 0.5, 2.0]]]]])  # [B,N,H,C,|Q|]
q = cal.decode_quantiles(qlogit, carry, "exp")
print("exp-branch decoded q (carry=0.083):", q.tolist())
print("  q0.1 reaches 0?", q[...,0].item() == 0.0)
print("config pinball_weight:", Config().pinball_weight)

# --- P2 configs (inherit pinball_weight=1.0 default) ---
SEEDS = [42, 1, 2, 3, 4]
cfg_chi = Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
                 chicago_year_start=2015, chicago_year_end=2024,
                 device="cuda", lookback=12, horizon=3,
                 epochs=80, batch_size=64, lr=1e-3, hidden_dim=64,
                 loss_type="nb", calibrated_head=True, pso_enabled=False)
cfg_mp = Config(target_region=MADHYA_PRADESH, device="cuda",
                lookback=4, horizon=2, epochs=100, batch_size=16,
                lr=5e-4, hidden_dim=64, loss_type="nb",
                calibrated_head=True, pso_enabled=False, patience=15)
print("cfg_chi.pinball_weight:", cfg_chi.pinball_weight, "| cfg_mp.pinball_weight:", cfg_mp.pinball_weight)
print("RESULTS_DIR:", RESULTS_DIR)
pathlib = __import__("pathlib"); pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# --- List existing P2 artifacts on Drive (stale under d1f365b exp-decode) ---
import glob
for pat in ["transfer_p2_seed*.json","calvspt_*_p2_seed*.json","robust_*_p2_*.json","dm_*_p2_seed*.json","dm_*_p1_seed*.json"]:
    fs = sorted(glob.glob(os.path.join(RESULTS_DIR, pat)))
    if fs: print(f"{pat}: {len(fs)} -> {[os.path.basename(x) for x in fs[:3]]}{' ...' if len(fs)>3 else ''}")
print("=== P1 artifacts (keep) ===")
for pat in ["multiseed_chicago_p1_chicago_seed*.json","metrics_p1_chicago_meanstd.csv","ablation_*_p1_*.json"]:
    fs = sorted(glob.glob(os.path.join(RESULTS_DIR, pat)))
    if fs: print(f"{pat}: {len(fs)}")

exp-branch decoded q (carry=0.083): [[[[[0.0, 0.0, 0.0, 0.05384386330842972, 0.5302916169166565]]]]]
  q0.1 reaches 0? True
config pinball_weight: 1.0
cfg_chi.pinball_weight: 1.0 | cfg_mp.pinball_weight: 1.0
RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results
transfer_p2_seed*.json: 10 -> ['transfer_p2_seed1_scratch.json', 'transfer_p2_seed1_transfer.json', 'transfer_p2_seed2_scratch.json'] ...
calvspt_*_p2_seed*.json: 20 -> ['calvspt_chicago_p2_seed1_calibrated.json', 'calvspt_chicago_p2_seed1_point.json', 'calvspt_chicago_p2_seed2_calibrated.json'] ...
=== P1 artifacts (keep) ===
multiseed_chicago_p1_chicago_seed*.json: 5
metrics_p1_chicago_meanstd.csv: 1
ablation_*_p1_*.json: 5


In [37]:
import glob, os, json
RESULTS_DIR = "/content/drive/MyDrive/spatio-temporal-crime_results"
# All P2-tagged artifacts (transfer pretrain cache + per-seed + calvspt + robust + dm).
p2_pats = ["transfer_p2_pretrain.json","hasi_net_p2_pretrain.pt",
           "transfer_p2_seed*.json","summary_p2_transfer.json",
           "calvspt_*_p2_seed*.json","metrics_calvspt_p2*.csv",
           "robust_*_p2_*.json","metrics_robust_p2*.csv",
           "dm_*_p2_seed*.json","metrics_dm_p2*.csv",
           "friedman_*p2*.json","bootstrap_*p2*.json"]
to_remove = []
for pat in p2_pats:
    for f in glob.glob(os.path.join(RESULTS_DIR, pat)):
        to_remove.append(f)
print(f"{len(to_remove)} stale P2 artifacts to remove:")
for f in sorted(to_remove): print("  ", os.path.basename(f))
# Sanity: do NOT touch P1 artifacts.
p1 = glob.glob(os.path.join(RESULTS_DIR,"multiseed_chicago_p1_*.json")) + glob.glob(os.path.join(RESULTS_DIR,"metrics_p1_*.csv")) + glob.glob(os.path.join(RESULTS_DIR,"ablation_*_p1_*.json"))
print(f"\nP1 artifacts preserved: {len(p1)} (untouched)")

33 stale P2 artifacts to remove:
   calvspt_chicago_p2_seed1_calibrated.json
   calvspt_chicago_p2_seed1_point.json
   calvspt_chicago_p2_seed2_calibrated.json
   calvspt_chicago_p2_seed2_point.json
   calvspt_chicago_p2_seed3_calibrated.json
   calvspt_chicago_p2_seed3_point.json
   calvspt_chicago_p2_seed42_calibrated.json
   calvspt_chicago_p2_seed42_point.json
   calvspt_chicago_p2_seed4_calibrated.json
   calvspt_chicago_p2_seed4_point.json
   calvspt_mp_p2_seed1_calibrated.json
   calvspt_mp_p2_seed1_point.json
   calvspt_mp_p2_seed2_calibrated.json
   calvspt_mp_p2_seed2_point.json
   calvspt_mp_p2_seed3_calibrated.json
   calvspt_mp_p2_seed3_point.json
   calvspt_mp_p2_seed42_calibrated.json
   calvspt_mp_p2_seed42_point.json
   calvspt_mp_p2_seed4_calibrated.json
   calvspt_mp_p2_seed4_point.json
   hasi_net_p2_pretrain.pt
   summary_p2_transfer.json
   transfer_p2_pretrain.json
   transfer_p2_seed1_scratch.json
   transfer_p2_seed1_transfer.json
   transfer_p2_seed2_scratch.j

In [38]:
import os, glob, threading, time, traceback
RESULTS_DIR = "/content/drive/MyDrive/spatio-temporal-crime_results"

# 1. Clear stale P2 artifacts.
removed = 0
for f in glob.glob(os.path.join(RESULTS_DIR, "transfer_p2_pretrain.json")) + \
         glob.glob(os.path.join(RESULTS_DIR, "hasi_net_p2_pretrain.pt")) + \
         glob.glob(os.path.join(RESULTS_DIR, "transfer_p2_seed*.json")) + \
         glob.glob(os.path.join(RESULTS_DIR, "summary_p2_transfer.json")) + \
         glob.glob(os.path.join(RESULTS_DIR, "calvspt_*_p2_seed*.json")) + \
         glob.glob(os.path.join(RESULTS_DIR, "metrics_calvspt_p2*.csv")) + \
         glob.glob(os.path.join(RESULTS_DIR, "robust_*_p2_*.json")) + \
         glob.glob(os.path.join(RESULTS_DIR, "metrics_robust_p2*.csv")) + \
         glob.glob(os.path.join(RESULTS_DIR, "dm_*_p2_seed*.json")) + \
         glob.glob(os.path.join(RESULTS_DIR, "metrics_dm_p2*.csv")):
    try: os.remove(f); removed += 1
    except OSError: pass
print(f"removed {removed} stale P2 artifacts")

# 2. Background runner (survives tool timeouts; writes heartbeat to LOG file).
LOG = "/content/drive/MyDrive/spatio-temporal-crime_results/_p2_run.log"
_BG = {"status": "idle", "done": False, "error": None, "step": "", "t0": 0.0}
def _log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG, "a") as fh: fh.write(line + "\n")

def bg_launch(name, thunk):
    _BG.update(status="running", done=False, error=None, step=name, t0=time.time())
    open(LOG, "w").write(f"=== {name} started {time.strftime('%H:%M:%S')} ===\n")
    def _run():
        try:
            thunk()
            _BG.update(status="done", done=True, step=name)
            _log(f"=== {name} DONE ===")
        except Exception as e:
            _BG.update(status="error", done=True, error=repr(e), step=name)
            _log(f"=== {name} ERROR: {e}\n{traceback.format_exc()}")
    t = threading.Thread(target=_run, daemon=True); t.start()
    return t

def bg_status():
    el = time.time() - _BG["t0"] if _BG["t0"] else 0.0
    return {k: _BG[k] for k in _BG}, f"elapsed {el/60:.1f} min"
print("runner ready; LOG=", LOG)

removed 33 stale P2 artifacts
runner ready; LOG= /content/drive/MyDrive/spatio-temporal-crime_results/_p2_run.log


In [39]:
from hasi_net import p2_experiments as p2, transfer as tr
print("p2_experiments:", [n for n in dir(p2) if n.startswith("run_") or "friedman" in n or "bootstrap" in n])
print("transfer:", [n for n in dir(tr) if n.startswith("run_")])
import inspect
for fn in [p2.run_calibrated_vs_point, p2.run_missing_data_robustness, p2.run_dm_comparison]:
    print(fn.__name__, "sig:", str(inspect.signature(fn)))

p2_experiments: ['run_calibrated_vs_point', 'run_dm_comparison', 'run_missing_data_robustness']
transfer: ['run_transfer_vs_scratch']
run_calibrated_vs_point sig: (dataset: 'str', cfg_cal: 'Config', seeds: 'List[int]', tag: 'str' = 'p2', force: 'bool' = False, verbose: 'bool' = True) -> 'pd.DataFrame'
run_missing_data_robustness sig: (dataset: 'str', cfg: 'Config', seeds: 'List[int]', fractions: 'List[float]', tag: 'str' = 'p2', force: 'bool' = False, verbose: 'bool' = True) -> 'pd.DataFrame'
run_dm_comparison sig: (dataset: 'str', cfg: 'Config', seeds: 'List[int]', pairs: 'List[tuple]', tag: 'str' = 'p2', h_step: 'int' = 1, force: 'bool' = False, verbose: 'bool' = True) -> 'pd.DataFrame'


In [ ]:
def _p2_pipeline():
    _log("P2 pipeline starting (expm1 decode + log-space pinball + pinball_weight=1.0)")
    results = {}
    # 1. Transfer Chicago->MP (rebuilds pretrain cache under pw=1.0).
    try:
        _log("[1/4] transfer Chicago->MP (L2-SP) ...")
        from hasi_net.transfer import run_transfer_vs_scratch
        tr_df = run_transfer_vs_scratch(cfg_chi, cfg_mp, seeds=SEEDS, lam=1e-3,
                                        tag="p2", verbose=True)
        results["transfer"] = tr_df
        _log("[1/4] transfer done:\n" + tr_df.round(4).to_string())
    except Exception as e:
        _log(f"[1/4] transfer FAILED: {e}\n{traceback.format_exc()}")

    # 2. Calibrated vs point head (mp + chicago) -- the calibration evidence.
    try:
        _log("[2/4] calibrated-vs-point MP ...")
        from hasi_net.p2_experiments import run_calibrated_vs_point
        df_mp = run_calibrated_vs_point("mp", cfg_mp, seeds=SEEDS, tag="p2", verbose=True)
        results["calvspt_mp"] = df_mp
        _log("[2/4] calvspt MP done:\n" + df_mp.round(4).to_string())
    except Exception as e:
        _log(f"[2/4] calvspt MP FAILED: {e}\n{traceback.format_exc()}")
    try:
        _log("[2b/4] calibrated-vs-point Chicago ...")
        from hasi_net.p2_experiments import run_calibrated_vs_point
        df_chi = run_calibrated_vs_point("chicago", cfg_chi, seeds=SEEDS, tag="p2", verbose=True)
        results["calvspt_chi"] = df_chi
        _log("[2b/4] calvspt Chicago done:\n" + df_chi.round(4).to_string())
    except Exception as e:
        _log(f"[2b/4] calvspt Chicago FAILED: {e}\n{traceback.format_exc()}")

    # 3. Missing-data robustness (MP).
    try:
        _log("[3/4] missing-data robustness MP ...")
        from hasi_net.p2_experiments import run_missing_data_robustness
        df_rob = run_missing_data_robustness("mp", cfg_mp, seeds=SEEDS,
                                             fractions=[0.0, 0.1, 0.25, 0.5],
                                             tag="p2", verbose=True)
        results["robust"] = df_rob
        _log("[3/4] robustness done:\n" + df_rob.round(4).to_string())
    except Exception as e:
        _log(f"[3/4] robustness FAILED: {e}\n{traceback.format_exc()}")

    # 4. Diebold-Mariano model comparison (Chicago point head, h=3).
    try:
        _log("[4/4] DM comparison Chicago (point head) ...")
        from hasi_net.p2_experiments import run_dm_comparison
        df_dm = run_dm_comparison("chicago",
                  cfg_chi.override(calibrated_head=False, loss_type="log1p_mse"),
                  seeds=SEEDS,
                  pairs=[("HASI-Net","GraphWaveNet"),("HASI-Net","HA"),
                         ("GraphWaveNet","HA")],
                  tag="p1", h_step=3, verbose=True)
        results["dm"] = df_dm
        _log("[4/4] DM done:\n" + df_dm.round(4).to_string())
    except Exception as e:
        _log(f"[4/4] DM FAILED: {e}\n{traceback.format_exc()}")

    _log("P2 pipeline complete. Results keys: " + str(list(results.keys())))
    # stash for later retrieval
    import json
    with open(os.path.join(RESULTS_DIR, "_p2_pipeline_summary.json"), "w") as fh:
        json.dump({k: (v.to_dict() if hasattr(v,"to_dict") else str(v))
                   for k, v in results.items()}, fh, indent=2, default=str)

bg_launch("P2_full", _p2_pipeline)
print("launched; poll with bg_status() and tail LOG")

[10:13:30] P2 pipeline starting (expm1 decode + log-space pinball + pinball_weight=1.0)launched; poll with bg_status() and tail LOG



[10:13:30] [1/4] transfer Chicago->MP (L2-SP) ...
Pretraining on Chicago ...
  epoch   0 | train 0.7749 | val MAE 2.8246 | RMSE 5.2742
  epoch   5 | train 0.2735 | val MAE 2.8208 | RMSE 5.3252
  epoch  10 | train 0.2127 | val MAE 2.4874 | RMSE 4.4791
  epoch  15 | train 0.1809 | val MAE 2.5084 | RMSE 4.5376
  epoch  20 | train 0.1743 | val MAE 2.4760 | RMSE 4.4541
  epoch  25 | train 0.1721 | val MAE 2.4848 | RMSE 4.4777
  epoch  30 | train 0.1717 | val MAE 2.5397 | RMSE 4.6340
  early stop at epoch 31
  ft epoch   0 | train 1.5835 | val MAE 22.9507 | RMSE 37.7327
  ft epoch   1 | train 1.4584 | val MAE 22.5774 | RMSE 37.6024
  ft epoch   2 | train 1.3496 | val MAE 22.4812 | RMSE 37.7754
  ft epoch   3 | train 1.2394 | val MAE 22.7389 | RMSE 38.2142
  ft epoch   4 | train 1.1304 | val MAE 23.2122 | RMSE 38.8593
  ft epoch   5 | train 1.0294 | val MAE 23.7780 | RMSE 39.6265
  ft epoch   6 | train 0.9440 | val MAE 24.4460 | RMSE 40.4441
  ft epoch   7 | train 0.8531 | val MAE 25.1076 | R

In [ ]:
import time
# Brief poll: confirm the background thread is alive and training started.
st, el = bg_status()
print("status:", st, "|", el)
print("--- LOG tail ---")
try:
    lines = open(LOG).read().splitlines()
    for ln in lines[-12:]: print(ln)
except Exception as e:
    print("log read err:", e)

status: {'status': 'running', 'done': False, 'error': None, 'step': 'P2_full', 't0': 1783246410.4238794} | elapsed 0.2 min
--- LOG tail ---
=== P2_full started 10:13:30 ===
[10:13:30] P2 pipeline starting (expm1 decode + log-space pinball + pinball_weight=1.0)
[10:13:30] [1/4] transfer Chicago->MP (L2-SP) ...
  ft epoch  14 | train 0.5512 | val MAE 26.0109 | RMSE 42.3539


  ft epoch  15 | train 0.5172 | val MAE 25.9133 | RMSE 42.2627
  ft epoch  16 | train 0.5083 | val MAE 25.7927 | RMSE 42.1459
  ft early stop at epoch 17 (best val MAE 22.4812)
[seed 42/transfer] MAE=43.8882 CRPS=98.14622861544291 cov80=0.022058823529411766
  epoch   0 | train 1.5758 | val MAE 22.8141 | RMSE 38.5053
  epoch   5 | train 0.9097 | val MAE 24.5009 | RMSE 40.5683
  epoch  10 | train 0.5702 | val MAE 24.9090 | RMSE 41.1409
  epoch  15 | train 0.4196 | val MAE 24.7594 | RMSE 40.9270
  early stop at epoch 15
[seed 42/scratch] MAE=45.2388 CRPS=105.45471484219327 cov80=0.0
  ft epoch   0 | train 1.4607 | val MAE 25.5662 | RMSE 41.9223
  ft epoch   1 | train 1.3396 | val MAE 24.8514 | RMSE 40.8187
  ft epoch   2 | train 1.2316 | val MAE 24.4028 | RMSE 40.0856
  ft epoch   3 | train 1.1214 | val MAE 24.1250 | RMSE 39.6407
  ft epoch   4 | train 1.0170 | val MAE 24.0605 | RMSE 39.4738
  ft epoch   5 | train 0.9190 | val MAE 24.1374 | RMSE 39.5521
  ft epoch   6 | train 0.8260 | val

In [42]:
import time, re
# Persistent "seen" marker across watcher calls.
if "_WATCH_SEEN" not in globals():
    _WATCH_SEEN = set()

def _milestones_in(log_lines):
    """Return indices of lines that mark a step boundary."""
    out = {}
    for i, ln in enumerate(log_lines):
        if re.search(r"\[\d+(/\d+b?)?/\d+\] .* (done|FAILED)|pipeline complete|=== .* (DONE|ERROR)", ln):
            out[i] = ln
    return out

def watch(budget_s=140, poll=20):
    """Poll LOG until a NEW milestone appears or budget elapses.
    Returns dict: status, elapsed_min, new_milestones[], tail[], done."""
    t0 = time.time(); new = []
    while time.time() - t0 < budget_s:
        st, el = bg_status()
        try:
            lines = open(LOG).read().splitlines()
        except Exception:
            lines = []
        ms = _milestones_in(lines)
        for i, ln in ms.items():
            if i not in _WATCH_SEEN:
                _WATCH_SEEN.add(i); new.append(ln)
        if new or st["done"]:
            return {"status": st, "elapsed_min": el,
                    "new_milestones": new, "tail": lines[-4:], "done": st["done"]}
        time.sleep(poll)
    st, el = bg_status()
    try: tail = open(LOG).read().splitlines()[-4:]
    except Exception: tail = []
    return {"status": st, "elapsed_min": el, "new_milestones": new,
            "tail": tail, "done": st["done"]}

r = watch()
print("done?", r["done"], "| elapsed:", r["elapsed_min"], "| status:", r["status"]["status"], r["status"].get("error"))
if r["new_milestones"]:
    print("=== NEW MILESTONES ===")
    for m in r["new_milestones"]: print("  *", m)
print("=== live tail ===")
for ln in r["tail"]: print("  ", ln)

done? True | elapsed: elapsed 3.5 min | status: done None
=== NEW MILESTONES ===
  * [10:13:49] [1/4] transfer done:
  * [10:13:55] [2/4] calvspt MP done:
  * [10:14:54] [3/4] robustness done:
  * [10:15:27] [4/4] DM done:
  * [10:15:27] P2 pipeline complete. Results keys: ['transfer', 'calvspt_mp', 'calvspt_chi', 'robust', 'dm']
  * [10:15:27] === P2_full DONE ===
=== live tail ===
   13  chicago     4      HASI-Net            HA  5.4041  0.0000  2.6356  2.5913
   14  chicago     4  GraphWaveNet            HA -0.6960  0.4864  2.5820  2.5913
   [10:15:27] P2 pipeline complete. Results keys: ['transfer', 'calvspt_mp', 'calvspt_chi', 'robust', 'dm']
   [10:15:27] === P2_full DONE ===


In [43]:
import os, json, glob
print("=== FULL LOG ===")
print(open(LOG).read())
print("\n=== RESULT FILES NOW ON DRIVE ===")
for pat in ["transfer_p2_seed*.json","calvspt_*_p2_seed*.json","robust_*_p2_*.json",
            "dm_*_p1_seed*.json","summary_p2_transfer.json","hasi_net_p2_pretrain.pt",
            "transfer_p2_pretrain.json","_p2_pipeline_summary.json"]:
    fs = sorted(glob.glob(os.path.join(RESULTS_DIR, pat)))
    if fs: print(f"  {pat}: {len(fs)} -> {[os.path.basename(x) for x in fs][:4]}")

=== FULL LOG ===
=== P2_full started 10:13:30 ===
[10:13:30] P2 pipeline starting (expm1 decode + log-space pinball + pinball_weight=1.0)
[10:13:30] [1/4] transfer Chicago->MP (L2-SP) ...
[10:13:49] [1/4] transfer done:
   seed condition      MAE     RMSE   RMSLE     WAPE      R2     CSI      CRPS  pinball  coverage80  sharpness80  transferred_params                                                                    per_crime_MAE
0    42  transfer  43.8882  67.9328  0.8743  37.9952  0.3807  0.3171   98.1462  49.0731      0.0221      31.0688                  38    [22.215980529785156, 41.10944366455078, 66.25279998779297, 45.97470474243164]
1    42   scratch  45.2388  68.6364  0.8579  39.1645  0.3678  0.2439  105.4547  52.7274      0.0000      16.9440                   0     [22.124618530273438, 40.8570442199707, 65.20905303955078, 52.76469039916992]
2     1  transfer  48.2894  70.8960  0.8418  41.8055  0.3254  0.2195   83.4558  41.7279      0.0588      52.2203                  38    [3

In [44]:
import json, glob, os, numpy as np
try:
    from scipy import stats as sps
    SCIPY=True
except Exception as e:
    SCIPY=False; print("no scipy:", e)

RD=RESULTS_DIR
def load(pat):
    out=[]
    for f in sorted(glob.glob(os.path.join(RD,pat))):
        try: out.append(json.load(open(f)))
        except: pass
    return out

# ---- Transfer aggregate (transfer vs scratch) ----
tr=load("transfer_p2_seed*_transfer.json"); sc=load("transfer_p2_seed*_scratch.json")
def agg(rows, key):
    v=np.array([r[key] for r in rows if key in r and r[key] is not None], float)
    return (v.mean(), v.std(), len(v)) if v.size else (float('nan'),)*3
print("=== TRANSFER (MP) transfer vs scratch, mean±std over 5 seeds ===")
for k in ["MAE","RMSE","RMSLE","WAPE","R2","CSI","CRPS","pinball","coverage80","sharpness80"]:
    mt=agg(tr,k); ms=agg(sc,k)
    print(f"  {k:11s} transfer {mt[0]:.3f}±{mt[1]:.3f}  scratch {ms[0]:.3f}±{ms[1]:.3f}")

# ---- Calvspt Chicago aggregate ----
print("\n=== CALVSPT CHICAGO calibrated vs point, mean±std over 5 seeds ===")
for cond,pat in [("calibrated","calvspt_chicago_p2_seed*_calibrated.json"),("point","calvspt_chicago_p2_seed*_point.json")]:
    rows=load(pat)
    print(f" [{cond}] n={len(rows)}")
    for k in ["MAE","RMSLE","WAPE","R2","CRPS","pinball","coverage80","sharpness80"]:
        m=agg(rows,k); print(f"    {k:11s} {m[0]:.4f}±{m[1]:.4f}")

# ---- Calvspt MP aggregate (the data-scarcity story) ----
print("\n=== CALVSPT MP calibrated, mean±std over 5 seeds (scarcity regime) ===")
mpc=load("calvspt_mp_p2_seed*_calibrated.json")
for k in ["MAE","CRPS","pinball","coverage80","sharpness80"]:
    m=agg(mpc,k); print(f"  {k:11s} {m[0]:.4f}±{m[1]:.4f}")

# ---- DM aggregate + Friedman ----
print("\n=== DM (Chicago, point head, h=3) ===")
dm=load("dm_*_p1_seed*.json")
# group by modelA/modelB
from collections import defaultdict
g=defaultdict(list)
for r in dm: g[(r["modelA"],r["modelB"])].append((r["DM"],r["p"],r["meanA"],r["meanB"]))
for (a,b),lst in g.items():
    dmv=np.array([x[0] for x in lst]); pv=np.array([x[1] for x in lst])
    mA=np.array([x[2] for x in lst]); mB=np.array([x[3] for x in lst])
    print(f"  {a:13s} vs {b:13s}: DM {dmv.mean():+.3f}±{dmv.std():.3f} | p {pv.mean():.4f}±{pv.std():.4f} | MAE {mA.mean():.3f} vs {mB.mean():.3f}")

# ---- Per-crime MAE Chicago calibrated (the 4 unified crimes) ----
print("\n=== Chicago calibrated per-crime MAE (mean±std, 5 seeds) ===")
UNI=["rape_sexual_assault","domestic_violence","kidnapping_abduction","assault"]
pcm=np.array([r["per_crime_MAE"] for r in load("calvspt_chicago_p2_seed*_calibrated.json")])
for i,c in enumerate(UNI):
    print(f"  {c:24s} {pcm[:,i].mean():.4f}±{pcm[:,i].std():.4f}")

=== TRANSFER (MP) transfer vs scratch, mean±std over 5 seeds ===
  MAE         transfer 46.518±1.436  scratch 47.039±1.011
  RMSE        transfer 69.447±0.971  scratch 69.965±1.080
  RMSLE       transfer 0.844±0.026  scratch 0.844±0.026
  WAPE        transfer 40.272±1.243  scratch 40.723±0.875
  R2          transfer 0.353±0.018  scratch 0.343±0.020
  CSI         transfer 0.243±0.052  scratch 0.230±0.033
  CRPS        transfer 81.986±26.447  scratch 88.118±29.105
  pinball     transfer 40.993±13.223  scratch 44.059±14.553
  coverage80  transfer 0.128±0.210  scratch 0.158±0.306
  sharpness80 transfer 41.710±18.205  scratch 43.701±44.665

=== CALVSPT CHICAGO calibrated vs point, mean±std over 5 seeds ===
 [calibrated] n=5
    MAE         2.6900±0.0184
    RMSLE       0.3443±0.0004
    WAPE        19.9645±0.1368
    R2          0.9440±0.0012
    CRPS        2.0607±0.0356
    pinball     1.0304±0.0178
    coverage80  0.8407±0.0081
    sharpness80 11.6682±0.8917
 [point] n=5
    MAE         

KeyError: 'modelA'

In [ ]:
import json, glob, os
from collections import defaultdict
import numpy as np
RD=RESULTS_DIR
dm=sorted(glob.glob(os.path.join(RD,"dm_*_p1_seed*.json")))
print("n dm files:", len(dm))
print("sample keys:", list(json.load(open(dm[0])).keys()))
print("sample:", json.load(open(dm[0])))

In [45]:
import os, glob, json, torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("HASI_RESULTS_DIR:", os.environ.get("HASI_RESULTS_DIR"))
RD=os.environ.get("HASI_RESULTS_DIR")
# Verify P2 artifacts survived the disconnect.
for pat in ["transfer_p2_seed*_transfer.json","calvspt_chicago_p2_seed*_calibrated.json",
            "calvspt_mp_p2_seed*_calibrated.json","robust_mp_p2_*.json",
            "dm_*_p1_seed*.json","hasi_net_p2_pretrain.pt"]:
    print(f"  {pat}: {len(glob.glob(os.path.join(RD,pat)))}")
dm=sorted(glob.glob(os.path.join(RD,"dm_*_p1_seed*.json")))
s=json.load(open(dm[0]))
print("DM sample keys:", list(s.keys()))
print("DM sample:", {k:s[k] for k in s if not isinstance(s[k],(list,dict))})

cuda: True Tesla T4
HASI_RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results
  transfer_p2_seed*_transfer.json: 5
  calvspt_chicago_p2_seed*_calibrated.json: 5
  calvspt_mp_p2_seed*_calibrated.json: 5
  robust_mp_p2_*.json: 20
  dm_*_p1_seed*.json: 15
  hasi_net_p2_pretrain.pt: 1
DM sample keys: ['err']
DM sample: {}


In [46]:
import json, glob, os
RD=os.environ.get("HASI_RESULTS_DIR")
dm=sorted(glob.glob(os.path.join(RD,"dm_*_p1_seed*.json")))
for f in dm[:4]:
    print(os.path.basename(f), "->", json.load(open(f)))
print("...")
print(os.path.basename(dm[-1]), "->", json.load(open(dm[-1])))

dm_chicago_p1_seed1_GraphWaveNet.json -> {'err': [[2.1535158157348633, 2.943063735961914, 0.0, 3.2695350646972656, 0.8612068891525269, 2.492860794067383, 0.2886834740638733, 2.392772674560547, 0.7350664138793945, 7.533561706542969, 0.0, 3.4268569946289062, 0.8608303070068359, 3.042191505432129, 0.058153945952653885, 2.554851531982422, 0.34477293491363525, 0.19223499298095703, 0.0, 0.18015527725219727, 1.913619041442871, 3.3165245056152344, 0.08973725140094757, 7.569343566894531, 0.7504496574401855, 1.905411720275879, 0.0, 1.1093273162841797, 1.7757978439331055, 6.347282409667969, 0.2716722786426544, 15.009559631347656, 0.14449307322502136, 0.9904839992523193, 0.0, 1.649339199066162, 0.36620473861694336, 3.5760746002197266, 0.8703530728816986, 0.6952276229858398, 0.5301485061645508, 0.3214550018310547, 0.0, 3.5582399368286133, 0.09259290248155594, 1.4167060852050781, 0.0, 2.416743278503418, 0.6313842535018921, 1.544766902923584, 0.0, 0.9314231872558594, 1.1856443881988525, 2.63645935058

In [47]:
import os, glob, json, pandas as pd, numpy as np
RD=os.environ.get("HASI_RESULTS_DIR")
print("=== DM per-seed (saved CSV) ===")
dm_csv=os.path.join(RD,"dm_chicago_p1_perseed.csv")
if os.path.exists(dm_csv):
    dm=pd.read_csv(dm_csv)
    print(dm.to_string(index=False))
    print("\n=== DM mean±std ===")
    agg=dm.groupby(["modelA","modelB"])[["DM","p"]].agg(["mean","std"]).round(4)
    print(agg)
else:
    print("no dm csv")

print("\n=== P1 multiseed per-seed files (for Friedman) ===")
ms=sorted(glob.glob(os.path.join(RD,"multiseed_chicago_p1_chicago_seed*.json")))
print("n files:", len(ms), "->", [os.path.basename(x) for x in ms])
if ms:
    s=json.load(open(ms[0]))
    print("keys:", list(s.keys()))
    # show structure (truncated)
    import pprint
    def trim(o):
        if isinstance(o,dict): return {k:trim(v) for k,v in list(o.items())[:12]}
        if isinstance(o,list): return f"list[{len(o)}]"
        return o
    print("sample:", trim(s))

=== DM per-seed (saved CSV) ===
dataset  seed       modelA       modelB        DM            p    meanA    meanB
chicago    42     HASI-Net GraphWaveNet  4.356493 1.328408e-05 2.600833 2.556137
chicago    42     HASI-Net           HA  1.176915 2.392437e-01 2.600833 2.591253
chicago    42 GraphWaveNet           HA -3.730533 1.916194e-04 2.556137 2.591253
chicago     1     HASI-Net GraphWaveNet  5.841026 5.270917e-09 2.657303 2.593033
chicago     1     HASI-Net           HA  6.189739 6.147306e-10 2.657303 2.591253
chicago     1 GraphWaveNet           HA  0.182827 8.549360e-01 2.593033 2.591253
chicago     2     HASI-Net GraphWaveNet  4.926532 8.439089e-07 2.637460 2.589714
chicago     2     HASI-Net           HA  4.630885 3.664681e-06 2.637460 2.591253
chicago     2 GraphWaveNet           HA -0.179404 8.576221e-01 2.589714 2.591253
chicago     3     HASI-Net GraphWaveNet  7.540841 4.874840e-14 2.644633 2.563576
chicago     3     HASI-Net           HA  6.194891 5.949928e-10 2.644633 2.591

In [48]:
import os, glob, json, numpy as np, pandas as pd
RD=os.environ.get("HASI_RESULTS_DIR")
from hasi_net.stats import friedman_nemenyi, bootstrap_ci, diebold_mariano

MODELS=["HA","LSTM","STGCN","GraphWaveNet","DCRNN","MTGNN","InformerOnly","HASI-Net"]
SEEDS=[42,1,2,3,4]

# ---- 1. Friedman + Nemenyi across 8 models on per-seed MAE (5 blocks) ----
rows=[]
for f in sorted(glob.glob(os.path.join(RD,"multiseed_chicago_p1_chicago_seed*.json"))):
    s=json.load(open(f)); seed=s["seed"]
    rows.append([s["metrics"][m]["MAE"] for m in MODELS])
perf=np.array(rows)  # [5 seeds, 8 models]
fr=friedman_nemenyi(perf)
print("=== Friedman + Nemenyi (8 models, blocks=5 seeds, metric=MAE, h=3) ===")
print("  chi2:", round(fr["friedman_chi2"],4), "| p:", fr["friedman_p"])
print("  mean rank (1=best):", {m:round(r,3) for m,r in zip(MODELS,fr["mean_ranks"])})
print("  Nemenyi CD (alpha=0.1):", round(fr["cd"],4))
# rank matrix per seed
rmat=np.array([np.argsort(np.argsort(perf[i]))+1 for i in range(len(perf))])
rk=pd.DataFrame(rmat, columns=MODELS, index=[f"seed{s}" for s in SEEDS])
print("\n  per-seed ranks:\n", rk.to_string())

# ---- 2. Bootstrap CIs on headline metrics ----
def bs(vals): 
    m,lo,hi=bootstrap_ci(vals,0.95,10000,seed=0); return m,lo,hi
# Chicago calibrated coverage80 (headline)
cov=[json.load(open(f))["coverage80"] for f in glob.glob(os.path.join(RD,"calvspt_chicago_p2_seed*_calibrated.json"))]
cov_m,cov_lo,cov_hi=bs(cov)
# Chicago calibrated MAE vs point MAE
cal_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"calvspt_chicago_p2_seed*_calibrated.json"))]
pt_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"calvspt_chicago_p2_seed*_point.json"))]
# transfer vs scratch MAE
tr_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"transfer_p2_seed*_transfer.json"))]
sc_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"transfer_p2_seed*_scratch.json"))]
print("\n=== Bootstrap 95% CIs (5 seeds) ===")
print(f"  Chicago coverage80: {cov_m:.4f} [{cov_lo:.4f}, {cov_hi:.4f}]")
cm,cl,cu=bs(cal_mae); pm,pl,pu=bs(pt_mae)
print(f"  Chicago MAE calibrated: {cm:.4f} [{cl:.4f}, {cu:.4f}] | point: {pm:.4f} [{pl:.4f}, {pu:.4f}]")
tm,tl,tu=bs(tr_mae); sm,sl,su=bs(sc_mae)
print(f"  MP transfer MAE: {tm:.4f} [{tl:.4f}, {tu:.4f}] | scratch: {sm:.4f} [{sl:.4f}, {su:.4f}]")

# ---- 3. Save consolidated P2 stats ----
out={"friedman":{"chi2":fr["friedman_chi2"],"p":fr["friedman_p"],
     "mean_ranks":{m:float(r) for m,r in zip(MODELS,fr["mean_ranks"])},"cd_alpha0.1":fr["cd"],
     "per_seed_ranks":rk.to_dict()},
    "bootstrap_95":{"chi_coverage80":{"mean":cov_m,"lo":cov_lo,"hi":cov_hi},
      "chi_mae_calibrated":{"mean":cm,"lo":cl,"hi":cu},"chi_mae_point":{"mean":pm,"lo":pl,"hi":pu},
      "mp_mae_transfer":{"mean":tm,"lo":tl,"hi":tu},"mp_mae_scratch":{"mean":sm,"lo":sl,"hi":su}},
    "dm_mean_std":pd.read_csv(os.path.join(RD,"dm_chicago_p1_perseed.csv"))
        .groupby(["modelA","modelB"])[["DM","p"]].agg(["mean","std"]).round(4).to_dict()}
json.dump(out, open(os.path.join(RD,"p2_stats_summary.json"),"w"), indent=2, default=str)
print("\nsaved p2_stats_summary.json")

=== Friedman + Nemenyi (8 models, blocks=5 seeds, metric=MAE, h=3) ===


KeyError: 'friedman_chi2'

In [49]:
import os, glob, json, numpy as np
RD=os.environ.get("HASI_RESULTS_DIR")
from hasi_net.stats import friedman_nemenyi
MODELS=["HA","LSTM","STGCN","GraphWaveNet","DCRNN","MTGNN","InformerOnly","HASI-Net"]
rows=[]
for f in sorted(glob.glob(os.path.join(RD,"multiseed_chicago_p1_chicago_seed*.json"))):
    s=json.load(open(f)); rows.append([s["metrics"][m]["MAE"] for m in MODELS])
perf=np.array(rows)
fr=friedman_nemenyi(perf)
print("return keys:", list(fr.keys()))
print({k:(v if not isinstance(v,(list,np.ndarray)) else list(v)[:8]) for k,v in fr.items()})

return keys: ['chi2', 'p', 'k', 'n_blocks', 'mean_ranks', 'cd', 'alpha']
{'chi2': 22.866666666666667, 'p': 0.0017981621925030713, 'k': 8, 'n_blocks': 5, 'mean_ranks': [2.2, 6.6, 6.8, 1.8, 5.8, 2.8, 4.8, 5.2], 'cd': 6.090419837696953, 'alpha': 0.1}


In [50]:
import os, glob, json, numpy as np, pandas as pd
RD=os.environ.get("HASI_RESULTS_DIR")
from hasi_net.stats import friedman_nemenyi, bootstrap_ci
MODELS=["HA","LSTM","STGCN","GraphWaveNet","DCRNN","MTGNN","InformerOnly","HASI-Net"]
SEEDS=[42,1,2,3,4]
rows=[]
for f in sorted(glob.glob(os.path.join(RD,"multiseed_chicago_p1_chicago_seed*.json"))):
    s=json.load(open(f)); rows.append([s["metrics"][m]["MAE"] for m in MODELS])
perf=np.array(rows)
fr=friedman_nemenyi(perf)
ranks={m:round(r,2) for m,r in sorted(zip(MODELS,fr["mean_ranks"]), key=lambda x:x[1])}
print("=== Friedman + Nemenyi (8 models, 5 seed-blocks, MAE, h=3) ===")
print(f"  chi2={fr['chi2']:.3f} p={fr['p']:.4f} (k=8, n=5) -> {'SIGNIFICANT' if fr['p']<0.05 else 'ns'}")
print("  mean ranks (1=best):", ranks)
print(f"  Nemenyi CD@0.1={fr['cd']:.3f}  (|rank diff|>CD => sig. different)")
# which pairs cross the CD?
mr=np.array(fr["mean_ranks"])
sig_pairs=[]
for i,a in enumerate(MODELS):
    for j,b in enumerate(MODELS):
        if i<j and abs(mr[i]-mr[j])>fr["cd"]: sig_pairs.append((a,b,round(abs(mr[i]-mr[j]),2)))
print("  significant rank gaps:", sig_pairs if sig_pairs else "none cross CD (5 blocks => conservative CD)")

def bs(vals): m,lo,hi=bootstrap_ci(vals,0.95,10000,seed=0); return m,lo,hi
cov=[json.load(open(f))["coverage80"] for f in glob.glob(os.path.join(RD,"calvspt_chicago_p2_seed*_calibrated.json"))]
cal_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"calvspt_chicago_p2_seed*_calibrated.json"))]
pt_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"calvspt_chicago_p2_seed*_point.json"))]
tr_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"transfer_p2_seed*_transfer.json"))]
sc_mae=[json.load(open(f))["MAE"] for f in glob.glob(os.path.join(RD,"transfer_p2_seed*_scratch.json"))]
cov_m,cov_lo,cov_hi=bs(cov); cm,cl,cu=bs(cal_mae); pm,pl,pu=bs(pt_mae)
tm,tl,tu=bs(tr_mae); sm,sl,su=bs(sc_mae)
print("\n=== Bootstrap 95% CIs (5 seeds) ===")
print(f"  Chicago coverage80 : {cov_m:.4f} [{cov_lo:.4f}, {cov_hi:.4f}]")
print(f"  Chicago MAE cal   : {cm:.4f} [{cl:.4f}, {cu:.4f}]")
print(f"  Chicago MAE point  : {pm:.4f} [{pl:.4f}, {pu:.4f}]")
print(f"  MP transfer MAE    : {tm:.4f} [{tl:.4f}, {tu:.4f}]")
print(f"  MP scratch MAE     : {sm:.4f} [{sl:.4f}, {su:.4f}]")

out={"friedman":{"chi2":fr["chi2"],"p":fr["p"],"k":fr["k"],"n_blocks":fr["n_blocks"],
     "mean_ranks":{m:float(r) for m,r in zip(MODELS,fr["mean_ranks"])},"cd_alpha0.1":fr["cd"],
     "sig_rank_gaps":sig_pairs},
    "bootstrap_95":{"chi_coverage80":{"mean":cov_m,"lo":cov_lo,"hi":cov_hi},
      "chi_mae_calibrated":{"mean":cm,"lo":cl,"hi":cu},"chi_mae_point":{"mean":pm,"lo":pl,"hi":pu},
      "mp_mae_transfer":{"mean":tm,"lo":tl,"hi":tu},"mp_mae_scratch":{"mean":sm,"lo":sl,"hi":su}}}
json.dump(out, open(os.path.join(RD,"p2_stats_summary.json"),"w"), indent=2, default=str)
print("\nsaved p2_stats_summary.json")

=== Friedman + Nemenyi (8 models, 5 seed-blocks, MAE, h=3) ===
  chi2=22.867 p=0.0018 (k=8, n=5) -> SIGNIFICANT
  mean ranks (1=best): {'GraphWaveNet': 1.8, 'HA': 2.2, 'MTGNN': 2.8, 'InformerOnly': 4.8, 'HASI-Net': 5.2, 'DCRNN': 5.8, 'LSTM': 6.6, 'STGCN': 6.8}
  Nemenyi CD@0.1=6.090  (|rank diff|>CD => sig. different)
  significant rank gaps: none cross CD (5 blocks => conservative CD)

=== Bootstrap 95% CIs (5 seeds) ===
  Chicago coverage80 : 0.8407 [0.8345, 0.8485]
  Chicago MAE cal   : 2.6900 [2.6748, 2.7067]
  Chicago MAE point  : 2.6352 [2.6165, 2.6490]
  MP transfer MAE    : 46.5184 [45.0655, 47.6899]
  MP scratch MAE     : 47.0388 [46.0507, 47.8025]

saved p2_stats_summary.json


In [ ]:
import os, threading, time, traceback, sys
sys.path.insert(0, "/content/spatio-temporal-crime/src")
os.chdir("/content/spatio-temporal-crime")
RD=os.environ.get("HASI_RESULTS_DIR","/content/drive/MyDrive/spatio-temporal-crime_results")
from hasi_net.config import Config, MADHYA_PRADESH
from hasi_net.multiseed import run_multiseed

SEEDS=[42,1,2,3,4]
LOG=os.path.join(RD,"_p1_horizons.log")
_BG={"status":"idle","done":False,"error":None,"step":"","t0":0.0}
def _log(m):
    line=f"[{time.strftime('%H:%M:%S')}] {m}"; print(line,flush=True)
    open(LOG,"a").write(line+"\n")
def bg_launch(name,thunk):
    _BG.update(status="running",done=False,error=None,step=name,t0=time.time())
    open(LOG,"w").write(f"=== {name} started {time.strftime('%H:%M:%S')} ===\n")
    def _run():
        try: thunk(); _BG.update(status="done",done=True,step=name); _log(f"=== {name} DONE ===")
        except Exception as e:
            _BG.update(status="error",done=True,error=repr(e),step=name)
            _log(f"=== {name} ERROR: {e}\n{traceback.format_exc()}")
    threading.Thread(target=_run,daemon=True).start()
def bg_status():
    el=time.time()-_BG["t0"] if _BG["t0"] else 0.0
    return {k:_BG[k] for k in _BG}, f"elapsed {el/60:.1f} min"

def _p1_horizons(horizons=[6,12]):
    for h in horizons:
        cfg=Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
                   chicago_year_start=2015, chicago_year_end=2024, device="cuda",
                   lookback=12, horizon=h, epochs=80, batch_size=64, lr=1e-3,
                   hidden_dim=64, n_graph_layers=2, n_attn_heads=4,
                   loss_type="log1p_mse", pso_enabled=True)
        tag=f"p1_chicago_h{h}"
        _log(f"===== P1 horizon={h} (tag={tag}) starting =====")
        mstd,perseed=run_multiseed("chicago", cfg, seeds=SEEDS, tag=tag,
                                    pso=True, force=False, verbose=True)
        _log(f"===== horizon={h} DONE =====\n"+mstd.round(4).to_string())

bg_launch("P1_h6_h12", _p1_horizons)
print("launched P1 h=6/h=12 (8 models x 5 seeds + PSO each); LOG=", LOG)

[10:23:04] ===== P1 horizon=6 (tag=p1_chicago_h6) starting =====
launched P1 h=6/h=12 (8 models x 5 seeds + PSO each); LOG= /content/drive/MyDrive/spatio-temporal-crime_results/_p1_horizons.log


Device: cuda | dataset=chicago tag=p1_chicago_h6 seeds=[42, 1, 2, 3, 4]
Panel: T=120 N=77 C=4


In [52]:
import time, re
if "_W1" not in globals(): _W1=set()
KEY=re.compile(r"horizon=\d+ (starting|DONE)|PSO selected|PSO: loaded|=== .* (DONE|ERROR)|ERROR:|cached ->")
def watch1(budget=140, poll=20):
    t0=time.time(); new=[]
    while time.time()-t0<budget:
        st,el=bg_status()
        try: lines=open(LOG).read().splitlines()
        except: lines=[]
        for i,ln in enumerate(lines):
            if i not in _W1 and KEY.search(ln):
                _W1.add(i); new.append(ln)
        if new or st["done"]: return {"st":st,"el":el,"new":new,"tail":lines[-3:],"done":st["done"]}
        time.sleep(poll)
    st,el=bg_status()
    try: tail=open(LOG).read().splitlines()[-3:]
    except: tail=[]
    return {"st":st,"el":el,"new":new,"tail":tail,"done":st["done"]}
r=watch1()
print("done?",r["done"],"| elapsed:",r["el"],"| status:",r["st"]["status"],r["st"].get("error"))
if r["new"]:
    print("=== NEW ==="); [print("  *",x) for x in r["new"]]
print("=== live tail ==="); [print("  ",x) for x in r["tail"]]

  PSO iter 0: gbest MAE=2.5248  mean=2.5729  w=0.900
  PSO iter 1: gbest MAE=2.5243  mean=2.5940  w=0.800
  PSO iter 2: gbest MAE=2.5050  mean=2.5444  w=0.700
done? False | elapsed: elapsed 2.6 min | status: running None
=== live tail ===
   === P1_h6_h12 started 10:23:04 ===
   [10:23:04] ===== P1 horizon=6 (tag=p1_chicago_h6) starting =====


[None, None]

In [53]:
import time, re
# widen KEY to also flag the final PSO result line
KEY=re.compile(r"horizon=\d+ (starting|DONE)|PSO selected|PSO: loaded|=== .* (DONE|ERROR)|ERROR:|cached ->|Device:|Panel:")
def watch1(budget=150, poll=25):
    t0=time.time(); new=[]
    while time.time()-t0<budget:
        st,el=bg_status()
        try: lines=open(LOG).read().splitlines()
        except: lines=[]
        for i,ln in enumerate(lines):
            if i not in _W1 and KEY.search(ln):
                _W1.add(i); new.append(ln)
        if new or st["done"]: return {"st":st,"el":el,"new":new,"tail":lines[-3:],"done":st["done"]}
        time.sleep(poll)
    st,el=bg_status()
    try: tail=open(LOG).read().splitlines()[-3:]
    except: tail=[]
    return {"st":st,"el":el,"new":new,"tail":tail,"done":st["done"]}
r=watch1()
print("done?",r["done"],"| elapsed:",r["el"],"| status:",r["st"]["status"],r["st"].get("error"))
if r["new"]:
    print("=== NEW ==="); [print("  *",x) for x in r["new"]]
print("=== live tail ==="); [print("  ",x) for x in r["tail"]]

  PSO iter 3: gbest MAE=2.5027  mean=2.5262  w=0.600
  PSO iter 4: gbest MAE=2.5000  mean=2.5244  w=0.500
  PSO iter 5: gbest MAE=2.5000  mean=2.5232  w=0.400
PSO selected: {'target_region': 'Madhya Pradesh', 'use_chicago_benchmark': True, 'mp_year_start': 2001, 'mp_year_end': 2014, 'chicago_year_start': 2015, 'chicago_year_end': 2024, 'socio_knn': 5, 'adaptive_graph': True, 'hidden_dim': 30, 'n_graph_layers': 3, 'n_attn_heads': 5, 'informer_factor': 5, 'tcn_channels': 32, 'tcn_kernel_size': 3, 'dropout': 0.05000000074505806, 'fusion_alpha_init': 0.34, 'lookback': 12, 'horizon': 6, 'loss_type': 'log1p_mse', 'focal_gamma': 1.5, 'calibrated_head': False, 'pinball_weight': 1.0, 'epochs': 80, 'batch_size': 64, 'lr': 0.004999999888241291, 'weight_decay': 1e-05, 'patience': 12, 'device': 'cuda', 'pso_enabled': True, 'pso_swarms': 3, 'pso_particles': 8, 'pso_iters': 6, 'pso_search_dim': 5, 'seed': 42}
  [seed 42] HA ...
    HA: MAE=2.6311 RMSE=4.7201 R2=0.9477 CSI=0.7307
  [seed 42] LSTM ...


[None, None, None]

In [54]:
import time, re
KEY=re.compile(r"horizon=\d+ (starting|DONE)|PSO selected|=== .* (DONE|ERROR)|ERROR:|mean \+/- std")
def watch1(budget=160, poll=25):
    t0=time.time(); new=[]
    while time.time()-t0<budget:
        st,el=bg_status()
        try: lines=open(LOG).read().splitlines()
        except: lines=[]
        for i,ln in enumerate(lines):
            if i not in _W1 and KEY.search(ln): _W1.add(i); new.append(ln)
        if (new and any("DONE" in x or "ERROR" in x for x in new)) or st["done"]:
            return {"st":st,"el":el,"new":new,"tail":lines[-4:],"done":st["done"]}
        time.sleep(poll)
    st,el=bg_status()
    try: tail=open(LOG).read().splitlines()[-4:]
    except: tail=[]
    return {"st":st,"el":el,"new":new,"tail":tail,"done":st["done"]}
r=watch1()
print("done?",r["done"],"| elapsed:",r["el"],"| status:",r["st"]["status"],r["st"].get("error"))
if r["new"]: print("=== NEW ==="); [print("  *",x) for x in r["new"]]
print("=== live tail ==="); [print("  ",x) for x in r["tail"]]

done? False | elapsed: elapsed 5.5 min | status: running None
=== NEW ===
  * [10:28:00] ===== horizon=6 DONE =====
=== live tail ===
   MTGNN           2.7149     4.9882      0.3470    20.2291   0.9415    0.7246   0.0389    0.0984     0.0009    0.2901  0.0023   0.0053      5.0
   InformerOnly    2.7461     5.0854      0.3483    20.4613   0.9392    0.7257   0.0497    0.1473     0.0016    0.3703  0.0035   0.0084      5.0
   HASI-Net        2.6874     4.8955      0.3470    20.0236   0.9437    0.7318   0.0360    0.0886     0.0010    0.2686  0.0020   0.0062      5.0
   [10:28:00] ===== P1 horizon=12 (tag=p1_chicago_h12) starting =====


[None, None, None, None]

In [ ]:
import time, re
KEY=re.compile(r"horizon=12 DONE|=== P1_h6_h12 (DONE|ERROR)|ERROR:|horizon=12.*starting")
def watch1(budget=170, poll=25):
    t0=time.time(); new=[]
    while time.time()-t0<budget:
        st,el=bg_status()
        try: lines=open(LOG).read().splitlines()
        except: lines=[]
        for i,ln in enumerate(lines):
            if i not in _W1 and KEY.search(ln): _W1.add(i); new.append(ln)
        if any("DONE" in x or "ERROR" in x for x in new) or st["done"]:
            return {"st":st,"el":el,"new":new,"tail":lines[-6:],"done":st["done"]}
        time.sleep(poll)
    st,el=bg_status()
    try: tail=open(LOG).read().splitlines()[-6:]
    except: tail=[]
    return {"st":st,"el":el,"new":new,"tail":tail,"done":st["done"]}
r=watch1()
print("done?",r["done"],"| elapsed:",r["el"],"| status:",r["st"]["status"],r["st"].get("error"))
if r["new"]: print("=== NEW ==="); [print("  *",x) for x in r["new"]]
print("=== live tail (last 6) ==="); [print("  ",x) for x in r["tail"]]

  PSO iter 0: gbest MAE=2.6752  mean=2.7303  w=0.900
  PSO iter 1: gbest MAE=2.6616  mean=2.7914  w=0.800
  PSO iter 2: gbest MAE=2.6609  mean=2.7112  w=0.700
  PSO iter 3: gbest MAE=2.6609  mean=2.6796  w=0.600
  PSO iter 4: gbest MAE=2.6609  mean=2.6795  w=0.500
  PSO iter 5: gbest MAE=2.6606  mean=2.6941  w=0.400
PSO selected: {'target_region': 'Madhya Pradesh', 'use_chicago_benchmark': True, 'mp_year_start': 2001, 'mp_year_end': 2014, 'chicago_year_start': 2015, 'chicago_year_end': 2024, 'socio_knn': 5, 'adaptive_graph': True, 'hidden_dim': 32, 'n_graph_layers': 3, 'n_attn_heads': 8, 'informer_factor': 5, 'tcn_channels': 32, 'tcn_kernel_size': 3, 'dropout': 0.07377080619335175, 'fusion_alpha_init': 0.34, 'lookback': 12, 'horizon': 12, 'loss_type': 'log1p_mse', 'focal_gamma': 1.5, 'calibrated_head': False, 'pinball_weight': 1.0, 'epochs': 80, 'batch_size': 64, 'lr': 0.004999999888241291, 'weight_decay': 1e-05, 'patience': 12, 'device': 'cuda', 'pso_enabled': True, 'pso_swarms': 3, '

[None, None, None, None, None, None]

    GraphWaveNet: MAE=2.8596 RMSE=5.3350 R2=0.9303 CSI=0.7124
  [seed 2] DCRNN ...
    DCRNN: MAE=2.6960 RMSE=4.9030 R2=0.9411 CSI=0.7240
  [seed 2] MTGNN ...
    MTGNN: MAE=2.7070 RMSE=4.9579 R2=0.9398 CSI=0.7194
  [seed 2] InformerOnly ...
    InformerOnly: MAE=2.7803 RMSE=5.1046 R2=0.9362 CSI=0.7166
  [seed 2] HASI-Net ...
    HASI-Net: MAE=2.7663 RMSE=5.0867 R2=0.9366 CSI=0.7215
  [seed 3] HA ...
    HA: MAE=2.6622 RMSE=4.7990 R2=0.9436 CSI=0.7314
  [seed 3] LSTM ...
    LSTM: MAE=2.7365 RMSE=5.0417 R2=0.9377 CSI=0.7200
  [seed 3] STGCN ...
    STGCN: MAE=2.8267 RMSE=5.2383 R2=0.9328 CSI=0.7192
  [seed 3] GraphWaveNet ...
    GraphWaveNet: MAE=2.7740 RMSE=5.1261 R2=0.9356 CSI=0.7133
  [seed 3] DCRNN ...
    DCRNN: MAE=2.7129 RMSE=4.9666 R2=0.9396 CSI=0.7240
  [seed 3] MTGNN ...
    MTGNN: MAE=2.7592 RMSE=5.0920 R2=0.9365 CSI=0.7215
  [seed 3] InformerOnly ...
    InformerOnly: MAE=3.0399 RMSE=5.8293 R2=0.9168 CSI=0.7088
  [seed 3] HASI-Net ...


In [56]:
import time, re
KEY=re.compile(r"horizon=12 DONE|=== P1_h6_h12 (DONE|ERROR)|ERROR:")
def watch1(budget=180, poll=20):
    t0=time.time(); new=[]
    while time.time()-t0<budget:
        st,el=bg_status()
        try: lines=open(LOG).read().splitlines()
        except: lines=[]
        for i,ln in enumerate(lines):
            if i not in _W1 and KEY.search(ln): _W1.add(i); new.append(ln)
        if new or st["done"]:
            return {"st":st,"el":el,"new":new,"tail":lines[-12:],"done":st["done"]}
        time.sleep(poll)
    st,el=bg_status()
    try: tail=open(LOG).read().splitlines()[-12:]
    except: tail=[]
    return {"st":st,"el":el,"new":new,"tail":tail,"done":st["done"]}
r=watch1()
print("done?",r["done"],"| elapsed:",r["el"],"| status:",r["st"]["status"],r["st"].get("error"))
if r["new"]: print("=== NEW ==="); [print("  *",x) for x in r["new"]]
print("=== h=12 mean±std table (last 12 lines) ==="); [print("  ",x) for x in r["tail"]]

    HASI-Net: MAE=2.7535 RMSE=5.0553 R2=0.9374 CSI=0.7274
  [seed 4] HA ...
    HA: MAE=2.6622 RMSE=4.7990 R2=0.9436 CSI=0.7314
  [seed 4] LSTM ...
    LSTM: MAE=2.7356 RMSE=5.0378 R2=0.9378 CSI=0.7170
  [seed 4] STGCN ...
    STGCN: MAE=2.7637 RMSE=5.1059 R2=0.9361 CSI=0.7187
  [seed 4] GraphWaveNet ...
    GraphWaveNet: MAE=2.7625 RMSE=5.1011 R2=0.9363 CSI=0.7189
  [seed 4] DCRNN ...
    DCRNN: MAE=2.7635 RMSE=5.1087 R2=0.9361 CSI=0.7144
  [seed 4] MTGNN ...
    MTGNN: MAE=2.7388 RMSE=5.0382 R2=0.9378 CSI=0.7215
  [seed 4] InformerOnly ...
    InformerOnly: MAE=2.8076 RMSE=5.2142 R2=0.9334 CSI=0.7163
  [seed 4] HASI-Net ...
    HASI-Net: MAE=2.7779 RMSE=5.1050 R2=0.9362 CSI=0.7187

=== mean +/- std ===
              MAE_mean  RMSE_mean  RMSLE_mean  WAPE_mean  R2_mean  CSI_mean  \
HA              2.6622     4.7990      0.3554    20.2149   0.9436    0.7314   
LSTM            2.7635     5.1015      0.3531    20.9841   0.9362    0.7157   
STGCN           2.7553     5.0695      0.3533    

[None, None, None, None, None, None, None, None, None, None, None, None]

In [57]:
import os, glob, json, numpy as np, pandas as pd
RD=os.environ.get("HASI_RESULTS_DIR")
from hasi_net.stats import friedman_nemenyi
MODELS=["HA","LSTM","STGCN","GraphWaveNet","DCRNN","MTGNN","InformerOnly","HASI-Net"]

# verify saved files
for h in [6,12]:
    fs=glob.glob(os.path.join(RD,f"multiseed_chicago_p1_chicago_h{h}_seed*.json"))
    csv=os.path.join(RD,f"metrics_p1_chicago_h{h}_meanstd.csv")
    print(f"h={h}: {len(fs)} seed jsons, meanstd csv exists={os.path.exists(csv)}")

# Friedman per horizon (h=3 from existing p1_chicago, h=6/h=12 from new tags)
def perf_matrix(tag):
    rows=[]
    for f in sorted(glob.glob(os.path.join(RD,f"multiseed_chicago_{tag}_seed*.json"))):
        s=json.load(open(f)); rows.append([s["metrics"][m]["MAE"] for m in MODELS])
    return np.array(rows)

print("\n=== Friedman per horizon (8 models, 5 seed-blocks, MAE) ===")
summary={}
for h,tag in [(3,"p1_chicago"),(6,"p1_chicago_h6"),(12,"p1_chicago_h12")]:
    P=perf_matrix(tag)
    fr=friedman_nemenyi(P)
    ranks={m:round(r,2) for m,r in sorted(zip(MODELS,fr["mean_ranks"]),key=lambda x:x[1])}
    hasi_rank=[m for m,_ in sorted(zip(MODELS,fr["mean_ranks"]),key=lambda x:x[1])].index("HASI-Net")+1
    print(f"  h={h:2d}: chi2={fr['chi2']:.2f} p={fr['p']:.4f} | ranks {ranks} | HASI-Net rank {hasi_rank}/8, MAE {P[:,7].mean():.4f}±{P[:,7].std():.4f}")
    summary[f"h{h}"]={"chi2":fr["chi2"],"p":fr["p"],"mean_ranks":ranks,
                     "hasi_rank":hasi_rank,"hasi_mae_mean":float(P[:,7].mean()),
                     "hasi_mae_std":float(P[:,7].std()),
                     "ha_mae_mean":float(P[:,0].mean()),
                     "best_learned":min(MODELS[1:],key=lambda m:P[:,MODELS.index(m)].mean())}

# consolidated multi-horizon MAE table
print("\n=== Multi-horizon MAE (mean±std, 5 seeds) ===")
tbl={}
for h,tag in [(3,"p1_chicago"),(6,"p1_chicago_h6"),(12,"p1_chicago_h12")]:
    P=perf_matrix(tag)
    tbl[h]={m:f"{P[:,i].mean():.4f}±{P[:,i].std():.4f}" for i,m in enumerate(MODELS)}
mt=pd.DataFrame(tbl).T
print(mt.to_string())

json.dump({"friedman_per_horizon":summary,"mae_table":tbl},
          open(os.path.join(RD,"p1_multihorizon_stats.json"),"w"),indent=2,default=str)
print("\nsaved p1_multihorizon_stats.json")

h=6: 5 seed jsons, meanstd csv exists=True
h=12: 5 seed jsons, meanstd csv exists=True

=== Friedman per horizon (8 models, 5 seed-blocks, MAE) ===
  h= 3: chi2=22.87 p=0.0018 | ranks {'GraphWaveNet': 1.8, 'HA': 2.2, 'MTGNN': 2.8, 'InformerOnly': 4.8, 'HASI-Net': 5.2, 'DCRNN': 5.8, 'LSTM': 6.6, 'STGCN': 6.8} | HASI-Net rank 5/8, MAE 2.6617±0.0244
  h= 6: chi2=15.73 p=0.0277 | ranks {'HA': 1.2, 'HASI-Net': 3.4, 'STGCN': 4.2, 'GraphWaveNet': 4.6, 'DCRNN': 5.0, 'LSTM': 5.4, 'MTGNN': 5.6, 'InformerOnly': 6.6} | HASI-Net rank 2/8, MAE 2.6874±0.0360
  h=12: chi2=24.00 p=0.0011 | ranks {'HA': 1.0, 'MTGNN': 3.0, 'DCRNN': 3.2, 'LSTM': 5.0, 'STGCN': 5.0, 'HASI-Net': 5.4, 'GraphWaveNet': 5.8, 'InformerOnly': 7.6} | HASI-Net rank 6/8, MAE 2.7659±0.0117

=== Multi-horizon MAE (mean±std, 5 seeds) ===
               HA           LSTM          STGCN   GraphWaveNet          DCRNN          MTGNN   InformerOnly       HASI-Net
3   2.5898±0.0000  2.6830±0.0108  2.6905±0.0156  2.5848±0.0223  2.6794±0.0236  

In [58]:
import os, glob, json, pandas as pd, numpy as np
RD=os.environ.get("HASI_RESULTS_DIR")
# Ablation (h=3, already on Drive) + channel weights.
abl=sorted(glob.glob(os.path.join(RD,"ablation_chicago_p1_chicago_seed*.json")))
print("ablation seed files:", len(abl))
if abl:
    s=json.load(open(abl[0]))
    print("ablation sample keys:", list(s.keys())[:6])
    # collect per-variant MAE across seeds
    from collections import defaultdict
    v=defaultdict(list)
    for f in abl:
        d=json.load(open(f))
        for k,val in d.items():
            if isinstance(val,dict) and "MAE" in val: v[k].append(val["MAE"])
    print("\n=== Ablation (h=3, MAE mean±std over 5 seeds) ===")
    for k in sorted(v): print(f"  {k:28s} {np.mean(v[k]):.4f}±{np.std(v[k]):.4f}")
# channel weights from one multiseed seed
ms=json.load(open(sorted(glob.glob(os.path.join(RD,"multiseed_chicago_p1_chicago_seed*.json")))[0]))
print("\nHASI-Net learned channel weights (seed42):", ms.get("channel_weights"))
# PSO-selected configs per horizon
for h,tag in [(3,"p1_chicago"),(6,"p1_chicago_h6"),(12,"p1_chicago_h12")]:
    p=os.path.join(RD,f"multiseed_chicago_{tag}_pso.json")
    if os.path.exists(p):
        c=json.load(open(p))["config"]
        print(f"PSO h={h}: hidden={c['hidden_dim']} layers={c['n_graph_layers']} heads={c['n_attn_heads']} dropout={round(c['dropout'],3)} lr={c['lr']}")

ablation seed files: 5
ablation sample keys: ['Full', 'no-adaptive-graph', 'no-socio', 'no-spatial', 'ZINB-loss']

=== Ablation (h=3, MAE mean±std over 5 seeds) ===
  Full                         2.6332±0.0190
  ZINB-loss                    2.7223±0.0428
  no-adaptive-graph            2.6548±0.0222
  no-socio                     2.6734±0.0276
  no-spatial                   2.6863±0.0132

HASI-Net learned channel weights (seed42): [0.3310461938381195, 0.3410719931125641, 0.327881783246994]
PSO h=3: hidden=35 layers=3 heads=7 dropout=0.307 lr=0.0016965981340035796
PSO h=6: hidden=30 layers=3 heads=5 dropout=0.05 lr=0.004999999888241291
PSO h=12: hidden=32 layers=3 heads=8 dropout=0.074 lr=0.004999999888241291


In [59]:
import torch, os, sys, subprocess
print("python:", sys.version.split()[0])
print("cuda available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
# find repo
for p in ["/content/spatio-temporal-crime", "/content/hasi-net", "/mnt"]:
    pass
print("cwd:", os.getcwd())
print("--- git ---")
r = subprocess.run(["bash","-lc","cd /content 2>/dev/null; ls -d */ 2>/dev/null | head; find / -maxdepth 4 -name 'spatio-temporal-crime' -type d 2>/dev/null | head"], capture_output=True, text=True)
print(r.stdout)
print(r.stderr[:500])

python: 3.12.13
cuda available: True | device: Tesla T4
cwd: /content/spatio-temporal-crime
--- git ---
drive/
sample_data/
spatio-temporal-crime/
/content/spatio-temporal-crime
/content/drive/MyDrive/spatio-temporal-crime




In [60]:
import subprocess, os
def sh(cmd):
    r = subprocess.run(["bash","-lc",cmd], capture_output=True, text=True)
    return r.stdout + ("\n[stderr] "+r.stderr[-400:] if r.stderr.strip() else "")
print("=== git pull ===")
print(sh("cd /content/spatio-temporal-crime && git pull -q origin master 2>&1 | tail -5 && git log --oneline -1"))
print("=== HASI_RESULTS_DIR ===", os.environ.get("HASI_RESULTS_DIR","<unset>"))
print("=== geopandas ===")
print(sh("python -c 'import geopandas; print(geopandas.__version__)' 2>&1 | tail -3"))
print("=== repo on path? ===")
print(sh("python -c 'import sys; sys.path.insert(0,\"/content/spatio-temporal-crime/src\"); import hasi_net; print(\"hasi_net OK\")' 2>&1 | tail -3"))

=== git pull ===
85a220b feat(data): add Austin dataset for Paper-2 2nd-city calibration + Chicago->Austin transfer

=== HASI_RESULTS_DIR === /content/drive/MyDrive/spatio-temporal-crime_results
=== geopandas ===
1.1.3

=== repo on path? ===
hasi_net OK



In [61]:
import subprocess, os, time
os.chdir("/content/spatio-temporal-crime")
# background the verify run, log to file; survive IDE polling gaps
log = "/content/verify_austin.log"
with open(log, "w") as f: f.write("")  # reset
p = subprocess.Popen(["bash","-lc", f"cd /content/spatio-temporal-crime && python scripts/verify_austin.py > {log} 2>&1; echo EXIT=$? >> {log}"])
print("launched verify pid", p.pid, "->", log)
time.sleep(3)
print("--- early log ---")
print(open(log).read())

launched verify pid 26505 -> /content/verify_austin.log
--- early log ---
Building Austin panel ...
Traceback (most recent call last):
  File "/content/spatio-temporal-crime/scripts/verify_austin.py", line 99, in <module>
    main()
  File "/content/spatio-temporal-crime/scripts/verify_austin.py", line 39, in main
    panel = get_dataset("austin", cfg)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/spatio-temporal-crime/src/hasi_net/data.py", line 612, in get_dataset
    return build_austin_panel(cfg)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/spatio-temporal-crime/src/hasi_net/data.py", line 541, in build_austin_panel
    r.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 400 Client Error: Bad Request for url: https://data.austintexas.gov/resource/fdj4-gpfu.json?$select=council_district%252Ccrime_type%252Cocc_date%252

In [62]:
import sys; sys.path.insert(0, "/content/spatio-temporal-crime/src")
from urllib.parse import quote
import urllib.request, json

def _enc(s): return quote(s, safe="%'()")
def _socrata_url(base, select, where, limit, offset):
    return (f"{base}?$select={_enc(select)}&$where={_enc(where)}"
            f"&$limit={limit}&$offset={offset}")

keys=("SEXUAL","SODOMY","RAPE","KIDNAPPING","ASSAULT","ASLT")
or_like=" OR ".join(f"upper(crime_type) like '%{k}%'" for k in keys)
where=f"family_violence='Y' OR {or_like}"
select="council_district,crime_type,occ_date,family_violence"
url=_socrata_url("https://data.austintexas.gov/resource/fdj4-gpfu.json", select, where, 5, 0)
print("URL:", url[:160], "...")

req=urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as resp:
    data=json.loads(resp.read().decode())
print("rows returned:", len(data))
print("sample:", data[0] if data else None)

URL: https://data.austintexas.gov/resource/fdj4-gpfu.json?$select=council_district%2Ccrime_type%2Cocc_date%2Cfamily_violence&$where=family_violence%3D'Y'%20OR%20uppe ...


HTTPError: HTTP Error 400: Bad Request

In [63]:
from urllib.parse import quote
import urllib.request, json, urllib.error

def _enc(s): return quote(s, safe="%'()")
keys=("SEXUAL","SODOMY","RAPE","KIDNAPPING","ASSAULT","ASLT")
or_like=" OR ".join(f"upper(crime_type) like '%{k}%'" for k in keys)
where=f"family_violence='Y' OR {or_like}"
select="council_district,crime_type,occ_date,family_violence"
url=(f"https://data.austintexas.gov/resource/fdj4-gpfu.json?$select={_enc(select)}"
     f"&$where={_enc(where)}&$limit=5&$offset=0")
print("FULL URL:\n", url, "\n")
req=urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
try:
    with urllib.request.urlopen(req, timeout=60) as resp:
        print("OK", len(json.loads(resp.read())))
except urllib.error.HTTPError as e:
    print("HTTP", e.code)
    print("BODY:", e.read().decode()[:800])

FULL URL:
 https://data.austintexas.gov/resource/fdj4-gpfu.json?$select=council_district%2Ccrime_type%2Cocc_date%2Cfamily_violence&$where=family_violence%3D'Y'%20OR%20upper(crime_type)%20like%20'%SEXUAL%'%20OR%20upper(crime_type)%20like%20'%SODOMY%'%20OR%20upper(crime_type)%20like%20'%RAPE%'%20OR%20upper(crime_type)%20like%20'%KIDNAPPING%'%20OR%20upper(crime_type)%20like%20'%ASSAULT%'%20OR%20upper(crime_type)%20like%20'%ASLT%'&$limit=5&$offset=0 

HTTP 400
BODY: <html>
<head><title>400 Bad Request</title></head>
<body>
<center><h1>400 Bad Request</h1></center>
</body>
</html>



In [ ]:
import urllib.request, json, urllib.error
base="https://data.austintexas.gov/resource/fdj4-gpfu.json"
tests={
 "raw quotes": f"{base}?$where=family_violence='Y'&$limit=3",
 "enc quotes %27": f"{base}?$where=family_violence%3D%27Y%27&$limit=3",
 "no where (all)": f"{base}?$limit=2",
}
for name,u in tests.items():
    req=urllib.request.Request(u, headers={"User-Agent":"Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            d=json.loads(resp.read()); print(f"[{name}] OK rows={len(d)}")
    except urllib.error.HTTPError as e:
        body=e.read().decode()[:120].replace("\n"," ")
        print(f"[{name}] HTTP {e.code} :: {body}")

In [64]:
from urllib.parse import quote
import urllib.request, json, urllib.error

def _enc(s): return quote(s, safe="%")   # ONLY keep % wildcard literal
keys=("SEXUAL","SODOMY","RAPE","KIDNAPPING","ASSAULT","ASLT")
or_like=" OR ".join(f"upper(crime_type) like '%{k}%'" for k in keys)
where=f"family_violence='Y' OR {or_like}"
select="council_district,crime_type,occ_date,family_violence"
url=(f"https://data.austintexas.gov/resource/fdj4-gpfu.json?$select={_enc(select)}"
     f"&$where={_enc(where)}&$limit=5&$offset=0")
print("URL:", url[:200], "...\n")
req=urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
try:
    with urllib.request.urlopen(req, timeout=60) as resp:
        d=json.loads(resp.read()); print("OK rows=", len(d)); print("sample:", d[0])
except urllib.error.HTTPError as e:
    print("HTTP", e.code, "::", e.read().decode()[:200].replace("\n"," "))

URL: https://data.austintexas.gov/resource/fdj4-gpfu.json?$select=council_district%2Ccrime_type%2Cocc_date%2Cfamily_violence&$where=family_violence%3D%27Y%27%20OR%20upper%28crime_type%29%20like%20%27%SEXUA ...

 </html>><h1>400 Bad Request</h1></center>d>


In [65]:
import urllib.request, json, urllib.error
base="https://data.austintexas.gov/resource/fdj4-gpfu.json"
tests=[
 ("1 bare limit",        f"{base}?$limit=2"),
 ("2 short select",      f"{base}?$select=council_district,crime_type&$limit=2"),
 ("3 where raw quote",   f"{base}?$where=family_violence='Y'&$limit=2"),
 ("4 where enc quote",   f"{base}?$where=family_violence%3D%27Y%27&$limit=2"),
 ("5 groupby (worked b4)",f"{base}?$select=district,count(*)&$group=district&$where=family_violence='Y'&$limit=3"),
 ("6 select enc commas", f"{base}?$select=council_district%2Ccrime_type&$limit=2"),
]
for name,u in tests:
    req=urllib.request.Request(u, headers={"User-Agent":"Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            d=json.loads(resp.read()); print(f"[{name}] OK rows={len(d)} keys={list(d[0]) if d else []}")
    except urllib.error.HTTPError as e:
        b=e.read().decode()[:80].replace("\n"," ")
        print(f"[{name}] HTTP {e.code} :: {b}")

[1 bare limit] OK rows=2 keys=['incident_report_number', 'crime_type', 'ucr_code', 'family_violence', 'occ_date_time', 'occ_date', 'occ_time', 'rep_date_time', 'rep_date', 'rep_time', 'location_type', 'council_district', 'sector', 'district', 'clearance_status', 'census_block_group']
[2 short select] OK rows=2 keys=['council_district', 'crime_type']
[3 where raw quote] OK rows=2 keys=['incident_report_number', 'crime_type', 'ucr_code', 'family_violence', 'occ_date_time', 'occ_date', 'occ_time', 'rep_date_time', 'rep_date', 'rep_time', 'location_type', 'council_district', 'sector', 'district', 'clearance_status', 'clearance_date', 'census_block_group']
[4 where enc quote] OK rows=2 keys=['incident_report_number', 'crime_type', 'ucr_code', 'family_violence', 'occ_date_time', 'occ_date', 'occ_time', 'rep_date_time', 'rep_date', 'rep_time', 'location_type', 'council_district', 'sector', 'district', 'clearance_status', 'clearance_date', 'census_block_group']
[5 groupby (worked b4)] OK rows=

In [66]:
from urllib.parse import quote
import urllib.request, json, urllib.error

def _enc(s): return quote(s, safe="")   # encode % -> %25 (valid escape on the wire)
keys=("SEXUAL","SODOMY","RAPE","KIDNAPPING","ASSAULT","ASLT")
or_like=" OR ".join(f"upper(crime_type) like '%{k}%'" for k in keys)
where=f"family_violence='Y' OR {or_like}"
select="council_district,crime_type,occ_date,family_violence"
url=(f"https://data.austintexas.gov/resource/fdj4-gpfu.json?$select={_enc(select)}"
     f"&$where={_enc(where)}&$limit=5&$offset=0")
print("URL:", url[:220], "...\n")
req=urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
try:
    with urllib.request.urlopen(req, timeout=60) as resp:
        d=json.loads(resp.read()); print("OK rows=", len(d)); print("sample:", d[0])
except urllib.error.HTTPError as e:
    print("HTTP", e.code, "::", e.read().decode()[:200].replace("\n"," "))

URL: https://data.austintexas.gov/resource/fdj4-gpfu.json?$select=council_district%2Ccrime_type%2Cocc_date%2Cfamily_violence&$where=family_violence%3D%27Y%27%20OR%20upper%28crime_type%29%20like%20%27%25SEXUAL%25%27%20OR%20upp ...

OK rows= 5
sample: {'council_district': '9', 'crime_type': 'ASSAULT WITH INJURY', 'occ_date': '2003-01-01T00:00:00.000', 'family_violence': 'N'}


In [67]:
import requests, json
keys=("SEXUAL","SODOMY","RAPE","KIDNAPPING","ASSAULT","ASLT")
or_like=" OR ".join(f"upper(crime_type) like '%{k}%'" for k in keys)
where=f"family_violence='Y' OR {or_like}"
params={"$select":"council_district,crime_type,occ_date,family_violence",
        "$where":where, "$limit":5, "$offset":0}
r=requests.get("https://data.austintexas.gov/resource/fdj4-gpfu.json",
              params=params, timeout=60, headers={"User-Agent":"Mozilla/5.0"})
print("status", r.status_code, "| url:", r.url[:200], "...")
if r.status_code==200:
    d=r.json(); print("rows", len(d), "| sample", d[0])
else:
    print("body", r.text[:200])

status 200 | url: https://data.austintexas.gov/resource/fdj4-gpfu.json?%24select=council_district%2Ccrime_type%2Cocc_date%2Cfamily_violence&%24where=family_violence%3D%27Y%27+OR+upper%28crime_type%29+like+%27%25SEXUAL% ...
rows 5 | sample {'council_district': '9', 'crime_type': 'ASSAULT WITH INJURY', 'occ_date': '2003-01-01T00:00:00.000', 'family_violence': 'N'}


In [68]:
import subprocess, os, time
os.chdir("/content/spatio-temporal-crime")
subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git pull -q origin master 2>&1 | tail -2 && git log --oneline -1"], check=False)
log="/content/verify_austin.log"
open(log,"w").close()
p=subprocess.Popen(["bash","-lc",f"cd /content/spatio-temporal-crime && python scripts/verify_austin.py > {log} 2>&1; echo EXIT=$? >> {log}"])
print("launched pid", p.pid)
time.sleep(2)
print(open(log).read() or "(no output yet)")

launched pid 27160
(no output yet)


In [69]:
import time
for _ in range(8):  # poll up to ~80s
    time.sleep(10)
    txt=open("/content/verify_austin.log").read()
    if "EXIT=" in txt or "Traceback" in txt:
        break
print(txt or "(still no output)")

Building Austin panel ...
  counts shape [T,N,C] = [120, 10, 4]
  districts (10): ['CD1', 'CD2', 'CD3', 'CD4', 'CD5', 'CD6', 'CD7', 'CD8', 'CD9', 'CD10']
  categories (4): ['rape_sexual_assault', 'domestic_violence', 'kidnapping_abduction', 'assault']
  months: 2015-01 .. 2024-12 (120 months)
  meta region: Austin
  meta dv_signal: explicit family_violence flag (not a proxy)
  shape OK

Per-crime totals (sum over area x month) and zero-fraction:
  rape_sexual_assault      total=      8211  zero_frac= 0.03
  domestic_violence        total=     69714  zero_frac= 0.00
  kidnapping_abduction     total=       196  zero_frac= 0.86
  assault                  total=     66942  zero_frac= 0.00

  domestic_violence total = 69714
  assault total = 66942 (distinct from DV: yes)

Building A_geo (rook contiguity from council-district GeoJSON) ...
  A_geo shape [10,10]  rook edges = 0  min degree 0  max degree 0
  A_socio shape [10,10]  socio edges/row ~ 0
Traceback (most recent call last):
  File "/

In [70]:
import sys; sys.path.insert(0,"/content/spatio-temporal-crime/src")
import geopandas as gpd, pandas as pd
from hasi_net.graph import _load_austin_districts, AUSTIN_COUNCIL_GEOJSON
from hasi_net.config import DATA_DIR
from hasi_net.data import download

dest=DATA_DIR/"austin_council_districts.geojson"
print("exists:", dest.exists(), "size:", dest.stat().st_size if dest.exists() else 0)
gdf=_load_austin_districts()
print("gdf index (district_number):", list(gdf.index))
print("crs:", gdf.crs)
print("geom types:", gdf.geometry.geom_type.value_counts().to_dict())
print("bounds:", gdf.total_bounds)
# try sjoin touches
sub=gdf
joined=gpd.sjoin(sub, gdf, how="left", predicate="touches")
right_col=next((c for c in joined.columns if c.endswith("_right") and c.rsplit("_right",1)[0]==str(gdf.index.name)), None)
print("right_col:", right_col, "| index name:", gdf.index.name)
print("non-null matches:", joined[right_col].notna().sum() if right_col else "N/A")
# also try intersects (more lenient)
ji=gpd.sjoin(sub, gdf, how="left", predicate="intersects")
print("intersects matches:", ji.index_right.notna().sum() if 'index_right' in ji.columns else (ji[right_col].notna().sum() if right_col else 'N/A'))

ImportError: cannot import name '_load_austin_districts' from 'hasi_net.graph' (/content/spatio-temporal-crime/src/hasi_net/graph.py)

In [71]:
import subprocess
print(subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && grep -n '_load_austin_districts\\|_is_austin\\|AUSTIN_COUNCIL' src/hasi_net/graph.py | head -20"],capture_output=True,text=True).stdout)
print("--- git log for graph.py ---")
print(subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git log --oneline -3 -- src/hasi_net/graph.py && git status -s src/hasi_net/graph.py"],capture_output=True,text=True).stdout)
print("--- pycache ---")
print(subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && ls -la src/hasi_net/__pycache__/graph*.pyc 2>/dev/null; find src -name '*.pyc' -path '*graph*' -newer src/hasi_net/graph.py 2>/dev/null | head"],capture_output=True,text=True).stdout or "(none)")

43:AUSTIN_COUNCIL_GEOJSON = (
77:def _is_austin(districts: List[str]) -> bool:
81:def _load_austin_districts():
86:    download(AUSTIN_COUNCIL_GEOJSON, dest, timeout=120)
221:        elif _is_austin(districts):
222:            gdf = _load_austin_districts()

--- git log for graph.py ---
85a220b feat(data): add Austin dataset for Paper-2 2nd-city calibration + Chicago->Austin transfer
a4494cf HASI-Net crime-forecasting project: 4-crime panels, 8-model benchmark, calibrated P2 head + transfer + stats

--- pycache ---
-rw-r--r-- 1 root root 17380 Jul  5 11:05 src/hasi_net/__pycache__/graph.cpython-312.pyc
src/hasi_net/__pycache__/graph.cpython-312.pyc



In [72]:
import sys; sys.path.insert(0,"/content/spatio-temporal-crime/src")
# purge any stale hasi_net.* from the persistent kernel cache
for m in list(sys.modules):
    if m=="hasi_net" or m.startswith("hasi_net."):
        del sys.modules[m]
import geopandas as gpd, pandas as pd
from hasi_net.graph import _load_austin_districts
from hasi_net.config import DATA_DIR

gdf=_load_austin_districts()
print("index (district_number):", list(gdf.index))
print("crs:", gdf.crs, "| n:", len(gdf))
print("geom types:", gdf.geometry.geom_type.value_counts().to_dict())
print("bounds:", [round(x,4) for x in gdf.total_bounds])
# make valid (fix self-intersections) then test predicates
gdf2=gdf.copy(); gdf2["geometry"]=gdf2.geometry.buffer(0)
for pred in ("touches","intersects"):
    j=gpd.sjoin(gdf2, gdf2, how="left", predicate=pred)
    rc=next((c for c in j.columns if c.endswith("_right") and c.rsplit("_right",1)[0]==str(gdf2.index.name)),None)
    rc=rc or ("index_right" if "index_right" in j.columns else None)
    nn=j[rc].notna().sum() if rc else "N/A"
    print(f"{pred}: right_col={rc} non-null matches={nn}")

index (district_number): [np.int64(7), np.int64(10), np.int64(5), np.int64(4), np.int64(3), np.int64(1), np.int64(9), np.int64(2), np.int64(8), np.int64(6)]
crs: EPSG:4326 | n: 10
geom types: {'MultiPolygon': 10}
bounds: [np.float64(-97.9384), np.float64(30.0756), np.float64(-97.5276), np.float64(30.5167)]
touches: right_col=__n_right non-null matches=32
intersects: right_col=__n_right non-null matches=50


In [73]:
import subprocess, os
os.chdir("/content/spatio-temporal-crime")
subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git pull -q origin master 2>&1 | tail -2 && git log --oneline -1"], check=False)
r=subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && python scripts/verify_austin.py 2>&1 | tail -40"],capture_output=True,text=True)
print(r.stdout)
if r.returncode: print("[stderr tail]", r.stderr[-400:])

Building Austin panel ...
  counts shape [T,N,C] = [120, 10, 4]
  districts (10): ['CD1', 'CD2', 'CD3', 'CD4', 'CD5', 'CD6', 'CD7', 'CD8', 'CD9', 'CD10']
  categories (4): ['rape_sexual_assault', 'domestic_violence', 'kidnapping_abduction', 'assault']
  months: 2015-01 .. 2024-12 (120 months)
  meta region: Austin
  meta dv_signal: explicit family_violence flag (not a proxy)
  shape OK

Per-crime totals (sum over area x month) and zero-fraction:
  rape_sexual_assault      total=      8211  zero_frac= 0.03
  domestic_violence        total=     69714  zero_frac= 0.00
  kidnapping_abduction     total=       196  zero_frac= 0.86
  assault                  total=     66942  zero_frac= 0.00

  domestic_violence total = 69714
  assault total = 66942 (distinct from DV: yes)

Building A_geo (rook contiguity from council-district GeoJSON) ...
  A_geo shape [10,10]  rook edges = 17  min degree 1  max degree 6
  A_socio shape [10,10]  socio edges = 35 (kNN on identical ones-features)
  A_geo OK (r

In [74]:
import subprocess, os, time
os.chdir("/content/spatio-temporal-crime")
subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git pull -q origin master 2>&1 | tail -2 && git log --oneline -1"], check=False)
# check the Chicago pretrain cache exists (1b reuses it) + prior P2 artifacts
rd=os.environ.get("HASI_RESULTS_DIR")
print("RESULTS_DIR:", rd)
print(subprocess.run(["bash","-lc",f"ls -la '{rd}'/hasi_net_p2_pretrain.pt '{rd}'/transfer_p2_pretrain.json 2>&1 | head; echo '--- austin/p2 artifacts present? ---'; ls '{rd}'/ | grep -i 'austin\\|p2chi\\|p2austin' | head; echo '(none yet expected)'"],capture_output=True,text=True).stdout)

RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results
-rw------- 1 root root 584781 Jul  5 10:13 /content/drive/MyDrive/spatio-temporal-crime_results/hasi_net_p2_pretrain.pt
-rw------- 1 root root     26 Jul  5 10:13 /content/drive/MyDrive/spatio-temporal-crime_results/transfer_p2_pretrain.json
--- austin/p2 artifacts present? ---
(none yet expected)



In [75]:
import subprocess, os, time
os.chdir("/content/spatio-temporal-crime")
log="/content/austin_p2.log"
open(log,"w").close()
cmd=(f"cd /content/spatio-temporal-crime && stdbuf -oL python scripts/run_p2_austin.py > {log} 2>&1; "
     f"echo P2AUSTIN_EXIT=$? >> {log}")
p=subprocess.Popen(["bash","-lc",cmd])
print("launched Austin P2 pid", p.pid, "->", log)
time.sleep(8)
print("--- early log ---")
print(open(log).read() or "(starting...)")

launched Austin P2 pid 28220 -> /content/austin_p2.log
--- early log ---
=== 1b. Cross-region transfer (Chicago -> Austin) ===



In [76]:
import time
log="/content/austin_p2.log"
last=""
for _ in range(15):  # up to ~150s
    time.sleep(10)
    txt=open(log).read()
    if "P2AUSTIN_EXIT" in txt or "Traceback" in txt:
        break
print(txt)

=== 1b. Cross-region transfer (Chicago -> Austin) ===
Pretrain (chicago): cached [p2]
  ft epoch   0 | train 0.9497 | val MAE 6.8429 | RMSE 13.0466
  ft epoch   1 | train 0.6309 | val MAE 5.7092 | RMSE 10.2949
  ft epoch   2 | train 0.4069 | val MAE 5.0792 | RMSE 8.7793
  ft epoch   3 | train 0.2677 | val MAE 5.0323 | RMSE 8.5937
  ft epoch   4 | train 0.2096 | val MAE 5.2042 | RMSE 8.9527
  ft epoch   5 | train 0.1996 | val MAE 5.0480 | RMSE 8.6379
  ft epoch   6 | train 0.1905 | val MAE 4.8901 | RMSE 8.3866
  ft epoch   7 | train 0.1683 | val MAE 4.7259 | RMSE 7.9662
  ft epoch   8 | train 0.1437 | val MAE 4.6855 | RMSE 7.8484
  ft epoch   9 | train 0.1258 | val MAE 4.7660 | RMSE 7.9612
  ft epoch  10 | train 0.1147 | val MAE 4.8260 | RMSE 8.0642
  ft epoch  11 | train 0.1053 | val MAE 4.8107 | RMSE 8.0719
  ft epoch  12 | train 0.0953 | val MAE 4.7721 | RMSE 7.9873
  ft epoch  13 | train 0.0862 | val MAE 4.6386 | RMSE 7.7208
  ft epoch  14 | train 0.0803 | val MAE 4.5144 | RMSE 7.54

In [77]:
import subprocess, os, json
rd=os.environ["HASI_RESULTS_DIR"]
print("=== Austin/P2 files on Drive ===")
print(subprocess.run(["bash","-lc",f"ls '{rd}'/ | grep -iE 'austin|p2chi|p2austin' "],capture_output=True,text=True).stdout)
print("=== transfer_p2chi_aus_meanstd.csv ===")
print(subprocess.run(["bash","-lc",f"cat '{rd}/transfer_p2chi_aus_meanstd.csv'"],capture_output=True,text=True).stdout)
print("=== summary_p2chi_aus_transfer.json ===")
print(subprocess.run(["bash","-lc",f"cat '{rd}/summary_p2chi_aus_transfer.json'"],capture_output=True,text=True).stdout[:1500])

=== Austin/P2 files on Drive ===
calvspt_austin_p2austin_meanstd.csv
calvspt_austin_p2austin_perseed.csv
calvspt_austin_p2austin_seed1_calibrated.json
calvspt_austin_p2austin_seed1_point.json
calvspt_austin_p2austin_seed2_calibrated.json
calvspt_austin_p2austin_seed2_point.json
calvspt_austin_p2austin_seed3_calibrated.json
calvspt_austin_p2austin_seed3_point.json
calvspt_austin_p2austin_seed42_calibrated.json
calvspt_austin_p2austin_seed42_point.json
calvspt_austin_p2austin_seed4_calibrated.json
calvspt_austin_p2austin_seed4_point.json
summary_p2chi_aus_transfer.json
transfer_p2chi_aus_meanstd.csv
transfer_p2chi_aus_perseed.csv
transfer_p2chi_aus_seed1_scratch.json
transfer_p2chi_aus_seed1_transfer.json
transfer_p2chi_aus_seed2_scratch.json
transfer_p2chi_aus_seed2_transfer.json
transfer_p2chi_aus_seed3_scratch.json
transfer_p2chi_aus_seed3_transfer.json
transfer_p2chi_aus_seed42_scratch.json
transfer_p2chi_aus_seed42_transfer.json
transfer_p2chi_aus_seed4_scratch.json
transfer_p2chi_a

In [78]:
import subprocess, os
rd=os.environ["HASI_RESULTS_DIR"]
print("=== calvspt_austin_p2austin_meanstd.csv ===")
print(subprocess.run(["bash","-lc",f"cat '{rd}/calvspt_austin_p2austin_meanstd.csv'"],capture_output=True,text=True).stdout)
print("=== calvspt_austin_p2austin_perseed.csv ===")
print(subprocess.run(["bash","-lc",f"cat '{rd}/calvspt_austin_p2austin_perseed.csv'"],capture_output=True,text=True).stdout)
print("=== transfer_p2chi_aus_perseed.csv (key cols) ===")
print(subprocess.run(["bash","-lc",f"cat '{rd}/transfer_p2chi_aus_perseed.csv'"],capture_output=True,text=True).stdout)

=== calvspt_austin_p2austin_meanstd.csv ===
,MAE,MAE,RMSE,RMSE,RMSLE,RMSLE,WAPE,WAPE,R2,R2,CSI,CSI,CRPS,CRPS,pinball,pinball,coverage80,coverage80,sharpness80,sharpness80
,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std,mean,std
condition,,,,,,,,,,,,,,,,,,,,
calibrated,4.6013514518737795,0.08571055554156229,8.010604667663575,0.1093789357257259,0.3052535951137543,0.0018336873249634523,16.22869930267334,0.3022959506863277,0.944986,0.0014978338,0.6034846340418827,0.027240692715772646,6.233212412597571,1.5219334224947514,3.1166062062987856,0.7609667112473757,0.7902380952380952,0.09181025310364797,36.53441734313965,17.313524392949365
point,4.526560688018799,0.09671068903279771,7.8559956550598145,0.28016462704573325,0.30573413968086244,0.002687313543031352,15.964916801452636,0.34109294452221256,0.9470431,0.0037874428,0.6197338466947265,0.008964085288338797,,,,,,,,

=== calvspt_austin_p2austin_perseed.csv ===
dataset,seed,condition,MAE,RMSE,RMSLE,WAPE,R2,CSI,C

In [79]:
import subprocess, sys, os
# Pull latest (has the prob-baselines commit 086968a)
r = subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git pull --ff-only origin master 2>&1 | tail -5"], capture_output=True, text=True)
print(r.stdout); print("STDERR:", r.stderr[-500:] if r.stderr else "")
# Purge any stale hasi_net modules cached from a prior session
for m in list(sys.modules):
    if m.startswith("hasi_net"):
        del sys.modules[m]
print("pulled + purged")

 src/hasi_net/calibrated.py      |  24 +++
 src/hasi_net/prob_baselines.py  | 343 ++++++++++++++++++++++++++++++++++++++++
 3 files changed, 437 insertions(+)
 create mode 100644 scripts/run_p2_probbaselines.py
 create mode 100644 src/hasi_net/prob_baselines.py

STDERR: 
pulled + purged


In [80]:
import sys, numpy as np, torch
sys.path.insert(0, "/content/spatio-temporal-crime/src")
from hasi_net.prob_baselines import _conformal_halfwidths, _gaussian_quantiles, _Z_80, ALPHA
from hasi_net.calibrated import calibration_metrics, quantile_regression_loss, QUANTILES

rng = np.random.default_rng(0)
n_win, H, N, C = 60, 3, 8, 4
lam = np.array([1.0, 5.0, 0.5, 10.0])
cal_true = rng.poisson(lam, size=(n_win, H, N, C)).astype(np.float64)
cal_pred = np.clip(cal_true + rng.normal(0, 1.0, cal_true.shape), 0, None)
te_true = rng.poisson(lam, size=(n_win, H, N, C)).astype(np.float64)
te_pred = np.clip(te_true + rng.normal(0, 1.0, te_true.shape), 0, None)

qhat = _conformal_halfwidths(cal_pred, cal_true, C, ALPHA)
print("conformal qhat per crime:", np.round(qhat, 3))
q_c = _gaussian_quantiles(te_pred, qhat, C).reshape(-1, C, len(QUANTILES))
true_c = te_true.reshape(-1, C)
cm = calibration_metrics(torch.from_numpy(q_c), torch.from_numpy(true_c))
print(f"conformal coverage80={cm.coverage80:.4f} (target ~0.80) sharpness80={cm.sharpness80:.4f} CRPS={cm.crps:.4f}")
lo, hi = q_c[..., QUANTILES.index(0.1)], q_c[..., QUANTILES.index(0.9)]
print(f"direct interval coverage={((true_c>=lo)&(true_c<=hi)).mean():.4f} (must match coverage80)")

# QR loss: ~0 when q==y, >0 otherwise
y = torch.from_numpy(rng.poisson(lam, size=(5, H, N, C)).astype(np.float32))
q_perfect = y.unsqueeze(-1).repeat(1,1,1,1,len(QUANTILES))
print(f"QR loss q==y (~0): {float(quantile_regression_loss({'quantiles':q_perfect}, y, QUANTILES)):.6e}")
q_bad = (y+2.0).unsqueeze(-1).repeat(1,1,1,1,len(QUANTILES))
print(f"QR loss q=y+2 (>0): {float(quantile_regression_loss({'quantiles':q_bad}, y, QUANTILES)):.4f}")
print("SMOKE_TEST_OK")

conformal qhat per crime: [1.    1.254 1.    1.28 ]
conformal coverage80=0.7774 (target ~0.80) sharpness80=2.0377 CRPS=0.4787
direct interval coverage=0.7774 (must match coverage80)
QR loss q==y (~0): 0.000000e+00
QR loss q=y+2 (>0): 0.2794
SMOKE_TEST_OK


In [81]:
import subprocess, os
log = "/content/probbaselines.log"
# Launch the full prob-baselines driver (Chicago + Austin, 5 seeds) on the GPU,
# detached. The repo is already pulled on /content/spatio-temporal-crime.
cmd = (
    "cd /content/spatio-temporal-crime && "
    f"export HASI_RESULTS_DIR='{os.environ['HASI_RESULTS_DIR']}' && "
    "python scripts/run_p2_probbaselines.py --datasets chicago austin --seeds 42 1 2 3 4 "
    f"> {log} 2>&1; echo PROBBASE_EXIT=$? >> {log}"
)
subprocess.Popen(["bash","-lc",cmd])
print("launched ->", log)

launched -> /content/probbaselines.log


In [82]:
import subprocess, time
time.sleep(90)
r = subprocess.run(["bash","-lc","tail -25 /content/probbaselines.log; echo ---; grep -c PROBBASE_EXIT /content/probbaselines.log 2>/dev/null"], capture_output=True, text=True)
print(r.stdout)

[austin 3/quantreg] MAE=5.7247 cov80=0.8865 sharp=34.4479
[austin 4/conformal] MAE=4.4257 cov80=0.7976 sharp=14.4486
  qr epoch   0 | train 0.8684 | val median-MAE 25.8453
  qr epoch   5 | train 0.1235 | val median-MAE 29.9264
  qr epoch  10 | train 0.0920 | val median-MAE 6.9395
  qr epoch  15 | train 0.0825 | val median-MAE 8.5964
  qr epoch  20 | train 0.0794 | val median-MAE 5.8051
  qr epoch  25 | train 0.0783 | val median-MAE 5.7358
  qr epoch  30 | train 0.0775 | val median-MAE 6.4563
  qr early stop at epoch 35 (best median-val-MAE 5.4949)
[austin 4/quantreg] MAE=5.1675 cov80=0.7861 sharp=18.5704

=== 4-way prob comparison (austin, mean) ===
               MAE             RMSE  ... coverage80 sharpness80         
              mean     std     mean  ...        std        mean      std
condition                            ...                                
calibrated  4.6014  0.0857   8.0106  ...     0.0918     36.5344  17.3135
conformal   4.5266  0.0967   7.8560  ...     0.006

In [83]:
import subprocess, os, json
rd=os.environ["HASI_RESULTS_DIR"]
print("=== files ===")
print(subprocess.run(["bash","-lc",f"ls '{rd}'/ | grep -iE 'probbase|summary_p2.*4way' "],capture_output=True,text=True).stdout)
for ds,tag in [("chicago","p2"),("austin","p2austin")]:
    f=rd+f"/probbase_{ds}_{tag}_4way_meanstd.csv"
    print(f"\n=== {ds} 4-way meanstd ===")
    print(subprocess.run(["bash","-lc",f"cat '{f}' 2>/dev/null || echo MISSING"],capture_output=True,text=True).stdout)
    print(f"=== {ds} 4way summary JSON ===")
    print(subprocess.run(["bash","-lc",f"cat '{rd}/summary_{tag}_{ds}_4way.json' 2>/dev/null | head -80"],capture_output=True,text=True).stdout)

=== files ===
probbase_austin_p2austin_4way_meanstd.csv
probbase_austin_p2austin_meanstd.csv
probbase_austin_p2austin_perseed.csv
probbase_austin_p2austin_seed1_conformal.json
probbase_austin_p2austin_seed1_quantreg.json
probbase_austin_p2austin_seed2_conformal.json
probbase_austin_p2austin_seed2_quantreg.json
probbase_austin_p2austin_seed3_conformal.json
probbase_austin_p2austin_seed3_quantreg.json
probbase_austin_p2austin_seed42_conformal.json
probbase_austin_p2austin_seed42_quantreg.json
probbase_austin_p2austin_seed4_conformal.json
probbase_austin_p2austin_seed4_quantreg.json
probbase_chicago_p2_4way_meanstd.csv
probbase_chicago_p2_meanstd.csv
probbase_chicago_p2_perseed.csv
probbase_chicago_p2_seed1_conformal.json
probbase_chicago_p2_seed1_quantreg.json
probbase_chicago_p2_seed2_conformal.json
probbase_chicago_p2_seed2_quantreg.json
probbase_chicago_p2_seed3_conformal.json
probbase_chicago_p2_seed3_quantreg.json
probbase_chicago_p2_seed42_conformal.json
probbase_chicago_p2_seed42_

In [84]:
import sys, numpy as np, torch, json, os
for m in list(sys.modules):
    if m.startswith("hasi_net"): del sys.modules[m]
sys.path.insert(0,"/content/spatio-temporal-crime/src")
from hasi_net.config import Config, MADHYA_PRADESH
from hasi_net.data import get_dataset
from hasi_net.train import build_model, train_one, set_seed, select_device
from hasi_net.transfer import _assemble, _loaders, _train_zero_fraction
from hasi_net.prob_baselines import _conformal_halfwidths, _gaussian_quantiles, evaluate_conformal
from hasi_net.calibrated import calibration_metrics, QUANTILES

rd=os.environ["HASI_RESULTS_DIR"]; dev=select_device("cuda")
cfg=Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
           chicago_year_start=2015, chicago_year_end=2024, device="cuda",
           lookback=12, horizon=3, epochs=80, batch_size=64, lr=1e-3,
           hidden_dim=64, loss_type="nb", calibrated_head=True, pso_enabled=False)
cfg_pt=cfg.override(calibrated_head=False, loss_type="log1p_mse")
panel=_assemble(get_dataset("chicago",cfg),cfg); C=panel["counts"].shape[2]
cats=panel["categories"]; zf=_train_zero_fraction(panel,cfg)
set_seed(42); tr,va,te=_loaders(panel,cfg_pt)
mpt=build_model(cfg_pt,panel,dev); train_one(mpt,cfg_pt,tr,va,dev,verbose=False)

# Conformal qhat on val; coverage on val (in-sample, ~0.80 by construction) vs test (drift test)
from torch.utils.data import DataLoader
from hasi_net.train import WindowDataset, make_splits
def preds(model,loader):
    model.eval(); P,T=[],[]
    with torch.no_grad():
        for x,y in loader:
            P.append(model.predict_mean(x.to(dev),3).cpu().numpy()); T.append(y.numpy())
    return np.clip(np.concatenate(P,0),0,None), np.concatenate(T,0)
vp,vt=preds(mpt,va); ep,et=preds(mpt,te)
qhat=_conformal_halfwidths(vp,vt,C)
def cov(pred,true,qhat):
    q=_gaussian_quantiles(pred,qhat,C).reshape(-1,C,len(QUANTILES))
    lo,hi=q[...,QUANTILES.index(0.1)],q[...,QUANTILES.index(0.9)]
    return ((true>=lo)&(true<=hi)).mean()
print("=== Chicago seed42 conformal coverage: val (exchangeable) vs test (drifted) ===")
print(f"  overall: val={cov(vp,vt,qhat):.4f}  test={cov(ep,et,qhat):.4f}  (target 0.80)")
for j,c in enumerate(cats):
    qh=_conformal_halfwidths(vp[:,:,:,j:j+1],vt[:,:,:,j:j+1],1)
    print(f"  {c:22s} val={cov(vp[:,:,:,j:j+1],vt[:,:,:,j:j+1],qh):.4f}  test={cov(ep[:,:,:,j:j+1],et[:,:,:,j:j+1],qh):.4f}")

# Calibrated head per-crime test coverage
set_seed(42); tr,va,te=_loaders(panel,cfg)
mc=build_model(cfg,panel,dev); mc.set_gate_from_sparsity(zf); train_one(mc,cfg,tr,va,dev,verbose=False)
mc.eval(); Q,Y=[],[]
with torch.no_grad():
    for x,y in te:
        Q.append(mc.predict_quantiles(x.to(dev),3).cpu().numpy()); Y.append(y.numpy())
Q=np.concatenate(Q,0); Y=np.concatenate(Y,0)
lo,hi=Q[...,QUANTILES.index(0.1)],Q[...,QUANTILES.index(0.9)]
print("\n=== Chicago seed42 calibrated head per-crime test coverage80 ===")
print(f"  overall: {((Y>=lo)&(Y<=hi)).mean():.4f}")
for j,c in enumerate(cats):
    print(f"  {c:22s} {((Y[...,j]>=lo[...,j])&(Y[...,j]<=hi[...,j])).mean():.4f}")
print("DIAG_OK")

=== Chicago seed42 conformal coverage: val (exchangeable) vs test (drifted) ===


ValueError: operands could not be broadcast together with shapes (15,3,77,4) (3465,4) 

In [85]:
def cov(pred,true,qhat):
    q=_gaussian_quantiles(pred,qhat,C).reshape(-1,C,len(QUANTILES))
    t=true.reshape(-1,C)
    lo,hi=q[...,QUANTILES.index(0.1)],q[...,QUANTILES.index(0.9)]
    return float(((t>=lo)&(t<=hi)).mean())
def cov1(pred,true,j):
    qh=_conformal_halfwidths(pred[...,j:j+1],true[...,j:j+1],1)
    q=_gaussian_quantiles(pred[...,j:j+1],qh,1).reshape(-1,1,len(QUANTILES))
    t=true[...,j:j+1].reshape(-1,1)
    lo,hi=q[...,0,QUANTILES.index(0.1)],q[...,0,QUANTILES.index(0.9)]
    return float(((t>=lo)&(t<=hi)).mean())
print("=== Chicago seed42 conformal coverage: val (exchangeable) vs test (drifted) ===")
print(f"  overall: val={cov(vp,vt,qhat):.4f}  test={cov(ep,et,qhat):.4f}  (target 0.80)")
for j,c in enumerate(cats):
    print(f"  {c:22s} val={cov1(vp,vt,j):.4f}  test={cov1(ep,et,j):.4f}")
print("\n=== Chicago seed42 calibrated head per-crime test coverage80 ===")
print(f"  overall: {float(((Y>=lo)&(Y<=hi)).mean()):.4f}")
lo,hi=Q[...,QUANTILES.index(0.1)],Q[...,QUANTILES.index(0.9)]
for j,c in enumerate(cats):
    print(f"  {c:22s} {float(((Y[...,j]>=lo[...,j])&(Y[...,j]<=hi[...,j])).mean()):.4f}")
print("DIAG_OK")

=== Chicago seed42 conformal coverage: val (exchangeable) vs test (drifted) ===
  overall: val=0.8003  test=0.7909  (target 0.80)
  rape_sexual_assault    val=0.5884  test=0.5664
  domestic_violence      val=0.2683  test=0.2590
  kidnapping_abduction   val=0.7924  test=0.7877
  assault                val=0.2917  test=0.2711

=== Chicago seed42 calibrated head per-crime test coverage80 ===


ValueError: operands could not be broadcast together with shapes (21,3,77,4) (1440,4) 

In [86]:
import numpy as np
def conf_percrime_test(pred, true):
    """Per-crime conformal test coverage80 + sharpness80 (each crime calibrated
    on its own val residuals). pred/true: [n,H,N,C]."""
    out=[]
    for j in range(C):
        qh=_conformal_halfwidths(vp[...,j],vt[...,j],1)  # calibrate on val for crime j
        q=_gaussian_quantiles(ep[...,j:j+1],qh,1).reshape(-1,5)   # (cells,5)
        t=et[...,j].reshape(-1)                                   # (cells,)
        lo,hi=q[...,QUANTILES.index(0.1)],q[...,QUANTILES.index(0.9)]
        cov=float(((t>=lo)&(t<=hi)).mean()); shp=float((hi-lo).mean())
        out.append((cov,shp,float(qh[0])))
    return out
print("=== Chicago seed42 conformal per-crime TEST coverage80 / sharpness80 / qhat ===")
for j,c in enumerate(cats):
    cov,shp,qh=conf_percrime_test(ep,et)[j]
    print(f"  {c:22s} cov80={cov:.4f}  sharp={shp:7.3f}  qhat={qh:7.3f}")
print("\n=== Chicago seed42 calibrated head per-crime TEST coverage80 / sharpness80 ===")
lo,hi=Q[...,QUANTILES.index(0.1)],Q[...,QUANTILES.index(0.9)]
for j,c in enumerate(cats):
    cov=float(((Y[...,j]>=lo[...,j])&(Y[...,j]<=hi[...,j])).mean())
    shp=float((hi[...,j]-lo[...,j]).mean())
    print(f"  {c:22s} cov80={cov:.4f}  sharp={shp:7.3f}")
print(f"\n  zero-fraction (train): {[round(z,3) for z in zf]}")
print("  categories order:", cats)
print("DIAG_OK")

=== Chicago seed42 conformal per-crime TEST coverage80 / sharpness80 / qhat ===
  rape_sexual_assault    cov80=0.8258  sharp=  2.624  qhat=  1.641
  domestic_violence      cov80=0.7914  sharp= 13.029  qhat=  6.696
  kidnapping_abduction   cov80=0.7572  sharp=  0.312  qhat=  0.222
  assault                cov80=0.7893  sharp= 13.705  qhat=  7.038

=== Chicago seed42 calibrated head per-crime TEST coverage80 / sharpness80 ===
  rape_sexual_assault    cov80=0.6772  sharp=  3.255
  domestic_violence      cov80=0.6846  sharp= 15.414
  kidnapping_abduction   cov80=0.9050  sharp=  0.387
  assault                cov80=0.7118  sharp= 16.285

  zero-fraction (train): [0.347, 0.005, 0.853, 0.006]
  categories order: ['rape_sexual_assault', 'domestic_violence', 'kidnapping_abduction', 'assault']
DIAG_OK


In [87]:
import subprocess, os, json
rd=os.environ["HASI_RESULTS_DIR"]
for s in [42,1,2,3,4]:
    d=json.loads(open(f"{rd}/calvspt_chicago_p2_seed{s}_calibrated.json").read())
    print(f"saved chicago seed{s} calibrated: cov80={d['coverage80']:.4f} sharp={d['sharpness80']:.3f} "
          f"CRPS={d['CRPS']:.4f} pinball={d['pinball']:.4f}  per_crime_MAE={[round(x,3) for x in d['per_crime_MAE']]}")
print("\n--- conformal saved per-seed (chicago) ---")
for s in [42,1,2,3,4]:
    d=json.loads(open(f"{rd}/probbase_chicago_p2_seed{s}_conformal.json").read())
    print(f"saved chicago seed{s} conformal: cov80={d['coverage80']:.4f} sharp={d['sharpness80']:.3f} "
          f"CRPS={d['CRPS']:.4f}  qhat={[round(x,3) for x in d['conformal_qhat']]}")
print("\n--- quantreg saved per-seed (chicago) ---")
for s in [42,1,2,3,4]:
    d=json.loads(open(f"{rd}/probbase_chicago_p2_seed{s}_quantreg.json").read())
    print(f"saved chicago seed{s} quantreg: cov80={d['coverage80']:.4f} sharp={d['sharpness80']:.3f} "
          f"CRPS={d['CRPS']:.4f} MAE={d['MAE']:.4f}")

saved chicago seed42 calibrated: cov80=0.8365 sharp=11.597 CRPS=2.0640 pinball=1.0320  per_crime_MAE=[1.02, 4.714, 0.159, 4.762]
saved chicago seed1 calibrated: cov80=0.8316 sharp=10.938 CRPS=2.0303 pinball=1.0152  per_crime_MAE=[1.018, 4.71, 0.165, 4.843]
saved chicago seed2 calibrated: cov80=0.8364 sharp=10.495 CRPS=2.0113 pinball=1.0056  per_crime_MAE=[1.021, 4.723, 0.165, 4.816]
saved chicago seed3 calibrated: cov80=0.8545 sharp=12.890 CRPS=2.1018 pinball=1.0509  per_crime_MAE=[1.02, 4.686, 0.161, 4.955]
saved chicago seed4 calibrated: cov80=0.8445 sharp=12.421 CRPS=2.0964 pinball=1.0482  per_crime_MAE=[1.017, 4.768, 0.159, 4.919]

--- conformal saved per-seed (chicago) ---
saved chicago seed42 conformal: cov80=0.7909 sharp=7.417 CRPS=1.9182  qhat=[1.641, 6.696, 0.222, 7.038]
saved chicago seed1 conformal: cov80=0.7831 sharp=7.256 CRPS=1.9694  qhat=[1.644, 6.525, 0.195, 6.91]
saved chicago seed2 conformal: cov80=0.7837 sharp=7.229 CRPS=1.9583  qhat=[1.629, 6.491, 0.201, 6.849]
save

In [88]:
import subprocess, sys, os
for m in list(sys.modules):
    if m.startswith("hasi_net"): del sys.modules[m]
r=subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git pull --ff-only origin master 2>&1 | tail -3"],capture_output=True,text=True)
print(r.stdout)
log="/content/condcal.log"
cmd=("cd /content/spatio-temporal-crime && "
     f"export HASI_RESULTS_DIR='{os.environ['HASI_RESULTS_DIR']}' && "
     "python scripts/run_p2_condcal.py --datasets chicago austin --seeds 42 1 2 3 4 "
     f"> {log} 2>&1; echo CONDCAL_EXIT=$? >> {log}")
subprocess.Popen(["bash","-lc",cmd])
print("launched ->",log)

 src/hasi_net/prob_baselines.py | 161 +++++++++++++++++++++++++++++++++++++++++
 2 files changed, 214 insertions(+)
 create mode 100644 scripts/run_p2_condcal.py

launched -> /content/condcal.log


In [89]:
import subprocess, time
time.sleep(200)
r=subprocess.run(["bash","-lc","tail -30 /content/condcal.log; echo ---; grep -c CONDCAL_EXIT /content/condcal.log 2>/dev/null"],capture_output=True,text=True)
print(r.stdout)

=== chicago: conditional calibration (calibrated vs conformal) ===
Traceback (most recent call last):
  File "/content/spatio-temporal-crime/scripts/run_p2_condcal.py", line 53, in <module>
    main()
  File "/content/spatio-temporal-crime/scripts/run_p2_condcal.py", line 47, in main
    run_conditional_calibration(ds, _cfg(), args.seeds, tag="p2cc",
  File "/content/spatio-temporal-crime/src/hasi_net/prob_baselines.py", line 400, in run_conditional_calibration
    zf = _train_zero_fraction(panel, cfg)
         ^^^^^^^^^^^^^^^^^^^^
NameError: name '_train_zero_fraction' is not defined
CONDCAL_EXIT=1
---
1



In [90]:
import subprocess, sys, os
for m in list(sys.modules):
    if m.startswith("hasi_net"): del sys.modules[m]
r=subprocess.run(["bash","-lc","cd /content/spatio-temporal-crime && git pull --ff-only origin master 2>&1 | tail -2"],capture_output=True,text=True)
print(r.stdout)
log="/content/condcal.log"
cmd=("cd /content/spatio-temporal-crime && "
     f"export HASI_RESULTS_DIR='{os.environ['HASI_RESULTS_DIR']}' && "
     "python scripts/run_p2_condcal.py --datasets chicago austin --seeds 42 1 2 3 4 "
     f"> {log} 2>&1; echo CONDCAL_EXIT=$? >> {log}")
subprocess.Popen(["bash","-lc",cmd])
print("relaunched ->",log)

 src/hasi_net/prob_baselines.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)

relaunched -> /content/condcal.log


In [91]:
import subprocess, time
time.sleep(180)
r=subprocess.run(["bash","-lc","tail -15 /content/condcal.log; echo ---; grep -c CONDCAL_EXIT /content/condcal.log 2>/dev/null"],capture_output=True,text=True)
print(r.stdout)

[austin 1/calibrated] marginal=0.8500 gap=0.0810 low/med/high=0.849/0.840/0.861 top=0.881 pernode_std=0.0244
[austin 1/conformal] marginal=0.7948 gap=0.2325 low/med/high=0.828/0.833/0.722 top=0.567 pernode_std=0.1675
[austin 2/calibrated] marginal=0.9119 gap=0.1444 low/med/high=0.883/0.912/0.943 top=0.944 pernode_std=0.0280
[austin 2/conformal] marginal=0.7829 gap=0.2405 low/med/high=0.832/0.806/0.707 top=0.560 pernode_std=0.1752
[austin 3/calibrated] marginal=0.7560 gap=0.1157 low/med/high=0.815/0.684/0.761 top=0.798 pernode_std=0.0569
[austin 3/conformal] marginal=0.7929 gap=0.2762 low/med/high=0.829/0.830/0.718 top=0.524 pernode_std=0.1676
[austin 4/calibrated] marginal=0.6758 gap=0.2563 low/med/high=0.808/0.593/0.611 top=0.544 pernode_std=0.0352
[austin 4/conformal] marginal=0.7976 gap=0.1889 low/med/high=0.827/0.822/0.742 top=0.611 pernode_std=0.1663

=== conditional calibration (austin, mean over seeds) ===
  calibrated marginal=0.7902 cond_gap=0.1381 low/med/high=0.834/0.747/0.7

In [92]:
import subprocess, os, json
rd=os.environ["HASI_RESULTS_DIR"]
for ds in ["chicago","austin"]:
    print(f"\n=== {ds} conditional-calibration summary ===")
    print(subprocess.run(["bash","-lc",f"cat '{rd}/summary_p2cc_{ds}_condcal.json'"],capture_output=True,text=True).stdout)
    print(f"--- {ds} per-seed (key cols) ---")
    print(subprocess.run(["bash","-lc",f"cat '{rd}/condcal_{ds}_p2cc_perseed.csv' 2>/dev/null | cut -d, -f2,3,5-12 | column -t -s, | head -12"],capture_output=True,text=True).stdout)


=== chicago conditional-calibration summary ===
{
  "tag": "p2cc",
  "dataset": "chicago",
  "seeds": [
    42,
    1,
    2,
    3,
    4
  ],
  "conditions": {
    "calibrated": {
      "coverage80_mean": 0.840702947845805,
      "conditional_gap_mean": 0.11546024326710562,
      "low_mean": 0.9034694661374983,
      "med_mean": 0.7256341275941584,
      "high_mean": 0.8576929096885271,
      "top_decile_mean": 0.9153386454183267,
      "per_node_coverage_std_mean": 0.07187768828462307,
      "sharpness80_mean": 11.668160247802735
    },
    "conformal": {
      "coverage80_mean": 0.7864667078952793,
      "conditional_gap_mean": 0.3363545816733068,
      "low_mean": 0.8346434515426961,
      "med_mean": 0.8448501152959264,
      "high_mean": 0.6800125215213649,
      "top_decile_mean": 0.46364541832669326,
      "per_node_coverage_std_mean": 0.1871731236796727,
      "sharpness80_mean": 7.330486411521852
    }
  }
}
--- chicago per-seed (key cols) ---
seed  condition   buckets     

In [93]:
import os, subprocess, torch
print("cwd:", os.getcwd())
print("cuda available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("HASI_RESULTS_DIR:", os.environ.get("HASI_RESULTS_DIR"))
# locate repo on colab
for cand in ["/content/spatio-temporal-crime", "/content/drive/MyDrive/spatio-temporal-crime"]:
    if os.path.isdir(cand):
        print("repo found:", cand)
print("git head:")
try:
    print(subprocess.run(["git","log","--oneline","-3"],capture_output=True,text=True,cwd=os.getcwd()).stdout)
except Exception as e:
    print("git err", e)

cwd: /content/spatio-temporal-crime
cuda available: True | device: Tesla T4
HASI_RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results
repo found: /content/spatio-temporal-crime
repo found: /content/drive/MyDrive/spatio-temporal-crime
git head:
3b251a9 fix(p2): import _train_zero_fraction in prob_baselines
0d74957 feat(p2): conditional calibration (calibrated vs conformal, per-bucket/node)
086968a feat(p2): probabilistic baselines (conformal + quantile regression)



In [94]:
import subprocess, sys
r = subprocess.run(["git","pull","origin","master"],capture_output=True,text=True,cwd="/content/spatio-temporal-crime")
print("PULL:", r.stdout.strip(), r.stderr.strip())
print("HEAD:", subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout.strip())
# import + helper smoke test
sys.path.insert(0, "/content/spatio-temporal-crime/src")
import numpy as np
from hasi_net.hotspot import (event_contingency, hit_at_k, per_node_thresholds,
                              global_threshold, train_time_end, hotspot_metrics)
rng=np.random.default_rng(0)
counts=rng.poisson(3.0,size=(24,6,2)).astype(float)
counts[20,3,0]=40
tend=train_time_end(list(range(0,14)),12,3)
nq=per_node_thresholds(counts,tend); gq=global_threshold(counts,tend)
true=np.stack([counts[14+t:14+t+3] for t in range(3)])  # [3,3,6,2]
pred=true.copy(); pred[1,0,3,0]=2
ec=event_contingency(pred,true,nq)
print("per-node:",{k:round(v,3) if isinstance(v,float) else v for k,v in ec.items()})
print("perfect CSI:",round(event_contingency(true,true,nq)["CSI"],3),
      "| perfect Hit@5:",round(hit_at_k(true,true,5),3))
print("zero-pred POD:",round(event_contingency(np.zeros_like(true),true,nq)["POD"],3))
print("hotspot_metrics keys:", sorted(hotspot_metrics(pred,true,counts,list(range(0,14)),12,3).keys()))

PULL: Updating 3b251a9..e813b57
Fast-forward
 scripts/run_p1_hotspot.py |  90 ++++++++++++
 src/hasi_net/hotspot.py   | 359 ++++++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 449 insertions(+)
 create mode 100644 scripts/run_p1_hotspot.py
 create mode 100644 src/hasi_net/hotspot.py From https://github.com/vivekjindal24/spatio-temporal-crime
 * branch            master     -> FETCH_HEAD
   3b251a9..e813b57  master     -> origin/master
HEAD: e813b57 feat(p1): hotspot reframe -- event verification + top-k Hit@k
per-node: {'POD': 1.0, 'FAR': 0.0, 'CSI': 1.0, 'bias': 1.0, 'event_rate': 0.167, 'n_events': 18.0, 'hits': 18.0, 'false_alarms': 0.0, 'misses': 0.0, 'correct_negatives': 90.0}
perfect CSI: 1.0 | perfect Hit@5: 1.0
zero-pred POD: 0.0
hotspot_metrics keys: ['Hit@10', 'Hit@5', 'global_CSI', 'global_FAR', 'global_POD', 'global_bias', 'global_correct_negatives', 'global_event_rate', 'global_false_alarms', 'global_hits', 'global_misses', 'global_n_events', 'node_CSI', 'node

In [95]:
import subprocess, os, time
repo="/content/spatio-temporal-crime"
log_path=os.path.join(os.environ["HASI_RESULTS_DIR"], "run_p1_hotspot.log")
cmd=["python","scripts/run_p1_hotspot.py","--seeds","42","1","2","3","4"]
# run detached, logs to file; writes go to HASI_RESULTS_DIR (Drive) via config.RESULTS_DIR
proc=subprocess.Popen(cmd, cwd=repo, stdout=open(log_path,"w"), stderr=subprocess.STDOUT)
with open(os.path.join(os.environ["HASI_RESULTS_DIR"],"run_p1_hotspot.pid"),"w") as f:
    f.write(str(proc.pid))
print("launched pid",proc.pid,"-> log:",log_path)
time.sleep(5)
print("--- first lines ---")
try:
    print(open(log_path).read())
except Exception as e:
    print("log read err", e)

launched pid 36311 -> log: /content/drive/MyDrive/spatio-temporal-crime_results/run_p1_hotspot.log
--- first lines ---
[hotspot] no cached PSO config at /content/drive/MyDrive/spatio-temporal-crime_results/summary_p1_prelim_multiseed.json; using default cfg

===== chicago: hotspot reframe (tag=p1hot, h=3) =====
Device: cuda | dataset=chicago tag=p1hot seeds=[42, 1, 2, 3, 4]



In [96]:
import os, time
log=os.path.join(os.environ["HASI_RESULTS_DIR"],"run_p1_hotspot.log")
pid_path=os.path.join(os.environ["HASI_RESULTS_DIR"],"run_p1_hotspot.pid")
pid=int(open(pid_path).read())
# poll up to ~10 min, printing new lines; exit early if process done
pos=0
deadline=time.time()+600
done=False
while time.time()<deadline:
    time.sleep(20)
    try:
        s=os.stat(log).st_size
    except FileNotFoundError:
        continue
    if s>pos:
        with open(log) as f:
            f.seek(pos); chunk=f.read(); pos=s
            # print only progress / completion lines (not full stdout spam)
            for ln in chunk.splitlines():
                if any(t in ln for t in ("seed","=====","cached","complete","Error","Traceback","MAE","Hit@","node_CSI","Device","Panel")):
                    print(ln)
    # check process alive
    r=os.waitpid(os.WAIT_ANY, os.WNOHANG) if False else None
    # portable liveness check
    import subprocess
    al=subprocess.run(["ps","-p",str(pid)],capture_output=True).returncode==0
    if not al:
        print("PROCESS EXITED")
        done=True
        break
# final tail
with open(log) as f:
    f.seek(max(0,os.stat(log).st_size-1500)); print("=== TAIL ==="); print(f.read())
print("done=",done)

[hotspot] no cached PSO config at /content/drive/MyDrive/spatio-temporal-crime_results/summary_p1_prelim_multiseed.json; using default cfg
===== chicago: hotspot reframe (tag=p1hot, h=3) =====
Device: cuda | dataset=chicago tag=p1hot seeds=[42, 1, 2, 3, 4]
Panel: T=120 N=77 C=4 train_windows=70
  [seed 42] HA ...
    HA: node_CSI=0.393 node_POD=0.417 node_FAR=0.130 Hit@5=0.879
  [seed 42] LSTM ...
    LSTM: node_CSI=0.374 node_POD=0.388 node_FAR=0.086 Hit@5=0.886
  [seed 42] STGCN ...
    STGCN: node_CSI=0.370 node_POD=0.381 node_FAR=0.076 Hit@5=0.879
  [seed 42] GraphWaveNet ...
    GraphWaveNet: node_CSI=0.397 node_POD=0.423 node_FAR=0.132 Hit@5=0.886
  [seed 42] DCRNN ...
    DCRNN: node_CSI=0.380 node_POD=0.397 node_FAR=0.100 Hit@5=0.879
  [seed 42] MTGNN ...
    MTGNN: node_CSI=0.400 node_POD=0.424 node_FAR=0.127 Hit@5=0.879
  [seed 42] InformerOnly ...
    InformerOnly: node_CSI=0.386 node_POD=0.409 node_FAR=0.124 Hit@5=0.781
  [seed 42] HASI-Net ...
    HASI-Net: node_CSI=0.390 

In [97]:
import os, pandas as pd, json, glob
R=os.environ["HASI_RESULTS_DIR"]
print("=== files written ===")
for f in sorted(glob.glob(os.path.join(R,"hotspot_*p1hot*"))): print(" ", os.path.basename(f))
for f in sorted(glob.glob(os.path.join(R,"preds_*p1hot*.npz")))[:3]: print(" ", os.path.basename(f), "(npz)")
print("npz count:", len(glob.glob(os.path.join(R,"preds_*p1hot*.npz"))))
ms=pd.read_csv(os.path.join(R,"hotspot_p1hot_chicago_meanstd.csv"),index_col=0)
cols=[c for c in ms.columns if c.endswith("_mean") and any(k in c for k in
      ("MAE","node_POD","node_FAR","node_CSI","node_bias","node_event_rate",
       "global_POD","global_FAR","global_CSI","Hit@"))]
print("\n=== mean (5 seeds) ===")
print(ms[cols].round(4).to_string())
print("\nn_seeds:", dict(ms["n_seeds"]))

=== files written ===
  hotspot_chicago_p1hot_seed1.json
  hotspot_chicago_p1hot_seed2.json
  hotspot_chicago_p1hot_seed3.json
  hotspot_chicago_p1hot_seed4.json
  hotspot_chicago_p1hot_seed42.json
  hotspot_p1hot_chicago_meanstd.csv
  hotspot_p1hot_chicago_perseed.csv
  preds_chicago_p1hot_seed1_DCRNN.npz (npz)
  preds_chicago_p1hot_seed1_GraphWaveNet.npz (npz)
  preds_chicago_p1hot_seed1_HA.npz (npz)
npz count: 40

=== mean (5 seeds) ===
              MAE_mean  node_POD_mean  node_FAR_mean  node_CSI_mean  node_bias_mean  node_event_rate_mean  global_POD_mean  global_FAR_mean  global_CSI_mean  Hit@5_mean  Hit@10_mean
HA              2.5913         0.4170         0.1297         0.3926          0.4792                 0.291           0.8349           0.1229           0.7474      0.8794       0.8905
LSTM            2.6797         0.3889         0.0885         0.3748          0.4267                 0.291           0.8045           0.1002           0.7383      0.8863       0.8892
STGCN     

In [98]:
import os, pandas as pd, json
R=os.environ["HASI_RESULTS_DIR"]
ms=pd.read_csv(os.path.join(R,"hotspot_p1hot_chicago_meanstd.csv"),index_col=0)
key=["MAE","node_POD","node_FAR","node_CSI","global_CSI","global_POD","Hit@5","Hit@10"]
print("=== mean +/- std (key metrics) ===")
rows=[]
for m in ms.index:
    parts=[]
    for k in key:
        mean=ms.loc[m,f"{k}_mean"]; std=ms.loc[m,f"{k}_std"]
        parts.append(f"{k}={mean:.3f}±{std:.3f}")
    rows.append(f"{m:14s} "+" ".join(parts))
print("\n".join(rows))
# per-seed spread for the headline per-node CSI + Hit@5 to judge significance
ps=pd.read_csv(os.path.join(R,"hotspot_p1hot_chicago_perseed.csv"))
print("\n=== per-seed node_CSI (HA is deterministic=constant) ===")
print(ps.pivot(index="model",columns="seed",values="node_CSI").round(3).to_string())
print("\n=== per-seed Hit@5 ===")
print(ps.pivot(index="model",columns="seed",values="Hit@5").round(3).to_string())
print("\n=== per-seed MAE ===")
print(ps.pivot(index="model",columns="seed",values="MAE").round(3).to_string())

=== mean +/- std (key metrics) ===
HA             MAE=2.591±0.000 node_POD=0.417±0.000 node_FAR=0.130±0.000 node_CSI=0.393±0.000 global_CSI=0.747±0.000 global_POD=0.835±0.000 Hit@5=0.879±0.000 Hit@10=0.890±0.000
LSTM           MAE=2.680±0.010 node_POD=0.389±0.002 node_FAR=0.089±0.003 node_CSI=0.375±0.002 global_CSI=0.738±0.001 global_POD=0.805±0.003 Hit@5=0.886±0.005 Hit@10=0.889±0.003
STGCN          MAE=2.693±0.044 node_POD=0.386±0.007 node_FAR=0.082±0.011 node_CSI=0.373±0.005 global_CSI=0.728±0.009 global_POD=0.792±0.018 Hit@5=0.879±0.000 Hit@10=0.888±0.004
GraphWaveNet   MAE=2.552±0.016 node_POD=0.426±0.004 node_FAR=0.132±0.008 node_CSI=0.400±0.002 global_CSI=0.756±0.003 global_POD=0.828±0.008 Hit@5=0.887±0.003 Hit@10=0.868±0.009
DCRNN          MAE=2.680±0.014 node_POD=0.391±0.005 node_FAR=0.092±0.005 node_CSI=0.376±0.004 global_CSI=0.730±0.006 global_POD=0.799±0.011 Hit@5=0.879±0.000 Hit@10=0.883±0.009
MTGNN          MAE=2.572±0.023 node_POD=0.413±0.007 node_FAR=0.109±0.010 node_CS

In [99]:
import os, pandas as pd, numpy as np
from scipy import stats
R=os.environ["HASI_RESULTS_DIR"]
ps=pd.read_csv(os.path.join(R,"hotspot_p1hot_chicago_perseed.csv"))
def paired(a,b):
    # paired t on 5 seeds; also sign count
    d=a-b
    t,p=stats.ttest_rel(a,b)
    wins=int((d>0).sum()); ties=int((d==0).sum()); n=len(d)
    return t,p,wins,ties,n,d.mean(),d.std()
print("model vs HA (paired t, 5 seeds). mean±std of (model-HA). wins=times model>HA")
for metric in ["MAE","node_CSI","global_CSI","Hit@5"]:
    print(f"\n--- {metric} ---")
    ha=ps[ps.model=="HA"].set_index("seed")[metric]
    for m in ["GraphWaveNet","MTGNN","HASI-Net","InformerOnly"]:
        x=ps[ps.model==m].set_index("seed")[metric].reindex(ha.index)
        t,p,w,ties,n,dm,ds=paired(x.values,ha.values)
        # for MAE lower=better; for CSI/Hit higher=better
        better = "model<HA (model better)" if metric=="MAE" else "model>HA (model better)"
        print(f"  {m:14s} d={dm:+.4f}±{ds:.4f}  t={t:+.2f} p={p:.3f}  wins(model>HA)={w}/{n}  [{better if (w>n/2 if metric!='MAE' else w<n/2) else 'HA better or tied'}]")
# Friedman across the 7 deep models + HA on per-node CSI
print("\n--- Friedman chi2 across 8 models (per-node CSI, 5 seeds) ---")
mat=ps.pivot(index="seed",columns="model",values="node_CSI")
chi2,p=stats.friedmanchisquare(*[mat[c].values for c in mat.columns])
print(f"  chi2={chi2:.2f} p={p:.4f}  (small effect sizes; n=5 seeds => low power)")
print("\n=== SUMMARY verdict ===")
print("GraphWaveNet: marginal, consistent edge over HA on MAE (5/5) & node_CSI (5/5); effect ~1.5% MAE, +0.007 CSI.")
print("HASI-Net: mid-pack; WORSE than HA on MAE (4/5), node_CSI (5/5), global_CSI (4/5).")
print("Nobody beats persistence by a practically meaningful margin on any hotspot/event metric.")

model vs HA (paired t, 5 seeds). mean±std of (model-HA). wins=times model>HA

--- MAE ---
  GraphWaveNet   d=-0.0394±0.0158  t=-4.99 p=0.008  wins(model>HA)=0/5  [model<HA (model better)]
  MTGNN          d=-0.0189±0.0226  t=-1.67 p=0.170  wins(model>HA)=1/5  [model<HA (model better)]
  HASI-Net       d=+0.0600±0.0473  t=+2.54 p=0.064  wins(model>HA)=5/5  [HA better or tied]
  InformerOnly   d=+0.1148±0.0373  t=+6.15 p=0.004  wins(model>HA)=5/5  [HA better or tied]

--- node_CSI ---
  GraphWaveNet   d=+0.0071±0.0023  t=+6.21 p=0.003  wins(model>HA)=5/5  [model>HA (model better)]
  MTGNN          d=+0.0008±0.0043  t=+0.35 p=0.741  wins(model>HA)=3/5  [model>HA (model better)]
  HASI-Net       d=-0.0093±0.0071  t=-2.60 p=0.060  wins(model>HA)=0/5  [HA better or tied]
  InformerOnly   d=-0.0121±0.0068  t=-3.54 p=0.024  wins(model>HA)=0/5  [HA better or tied]

--- global_CSI ---
  GraphWaveNet   d=+0.0082±0.0031  t=+5.36 p=0.006  wins(model>HA)=5/5  [model>HA (model better)]
  MTGNN       

In [100]:
import os, pandas as pd, glob, json
R=os.environ["HASI_RESULTS_DIR"]
print("=== available meanstd csvs ===")
for f in sorted(glob.glob(os.path.join(R,"*meanstd.csv"))):
    print("  ",os.path.basename(f))
print("\n=== stats jsons ===")
for f in sorted(glob.glob(os.path.join(R,"*stats*.json"))):
    print("  ",os.path.basename(f))

=== available meanstd csvs ===
   ablation_p1_chicago_meanstd.csv
   calvspt_austin_p2austin_meanstd.csv
   calvspt_chicago_p2_meanstd.csv
   calvspt_mp_p2_meanstd.csv
   dm_chicago_p1_meanstd.csv
   hotspot_p1hot_chicago_meanstd.csv
   metrics_p1_chicago_h12_meanstd.csv
   metrics_p1_chicago_h6_meanstd.csv
   metrics_p1_chicago_meanstd.csv
   probbase_austin_p2austin_4way_meanstd.csv
   probbase_austin_p2austin_meanstd.csv
   probbase_chicago_p2_4way_meanstd.csv
   probbase_chicago_p2_meanstd.csv
   robust_mp_p2_meanstd.csv
   transfer_p2_meanstd.csv
   transfer_p2chi_aus_meanstd.csv

=== stats jsons ===
   p1_multihorizon_stats.json
   p2_stats_summary.json


In [101]:
import os, pandas as pd, json
R=os.environ["HASI_RESULTS_DIR"]
order=["HA","LSTM","STGCN","GraphWaveNet","DCRNN","MTGNN","InformerOnly","HASI-Net"]
def show(tag, metrics=("MAE","RMSE","RMSLE","WAPE","R2","CSI")):
    df=pd.read_csv(os.path.join(R,f"metrics_{tag}_meanstd.csv"),index_col=0)
    df=df.reindex([m for m in order if m in df.index])
    print(f"\n===== {tag} (mean±std, 5 seeds) =====")
    for m in df.index:
        cells=[f"{df.loc[m,f'{k}_mean']:.3f}±{df.loc[m,f'{k}_std']:.3f}" for k in metrics]
        print(f"{m:14s} "+"  ".join(cells))
    print("  cols:", "  ".join(metrics))
show("p1_chicago")
show("p1_chicago_h6")
show("p1_chicago_h12")
print("\n===== ablation_p1_chicago (h=3, 40ep) =====")
ab=pd.read_csv(os.path.join(R,"ablation_p1_chicago_meanstd.csv"),index_col=0)
for v in ab.index:
    print(f"{v:18s} MAE={ab.loc[v,'MAE_mean']:.3f}±{ab.loc[v,'MAE_std']:.3f}  CSI={ab.loc[v,'CSI_mean']:.4f}  R2={ab.loc[v,'R2_mean']:.4f}")
print("\n===== p1_multihorizon_stats.json (key parts) =====")
s=json.load(open(os.path.join(R,"p1_multihorizon_stats.json")))
print(json.dumps(s,indent=1)[:1800])


===== p1_chicago (mean±std, 5 seeds) =====
HA             2.590±0.000  4.620±0.000  0.349±0.000  19.232±0.000  0.950±0.000  0.742±0.000
LSTM           2.683±0.011  4.910±0.029  0.346±0.000  19.924±0.080  0.944±0.001  0.721±0.001
STGCN          2.691±0.016  4.924±0.042  0.346±0.000  19.980±0.116  0.943±0.001  0.721±0.004
GraphWaveNet   2.585±0.022  4.668±0.059  0.344±0.000  19.195±0.165  0.949±0.001  0.750±0.005
DCRNN          2.679±0.024  4.897±0.063  0.346±0.001  19.897±0.175  0.944±0.001  0.722±0.006
MTGNN          2.599±0.049  4.702±0.124  0.344±0.001  19.300±0.366  0.948±0.003  0.740±0.013
InformerOnly   2.661±0.021  4.842±0.070  0.346±0.001  19.762±0.158  0.945±0.002  0.736±0.008
HASI-Net       2.662±0.024  4.830±0.061  0.346±0.001  19.766±0.181  0.945±0.001  0.731±0.003
  cols: MAE  RMSE  RMSLE  WAPE  R2  CSI

===== p1_chicago_h6 (mean±std, 5 seeds) =====
HA             2.631±0.000  4.720±0.000  0.350±0.000  19.604±0.000  0.948±0.000  0.731±0.000
LSTM           2.722±0.016  5.00

In [102]:
import os, pandas as pd, json
R=os.environ["HASI_RESULTS_DIR"]
def show(f, idx=0, cols=None):
    p=os.path.join(R,f)
    if not os.path.exists(p): print(f"MISSING {f}"); return
    df=pd.read_csv(p,index_col=idx)
    print(f"\n===== {f} =====")
    print(df.round(4).to_string())
show("probbase_chicago_p2_4way_meanstd.csv")
show("probbase_austin_p2austin_4way_meanstd.csv")
show("transfer_p2_meanstd.csv")
show("transfer_p2chi_aus_meanstd.csv")
show("robust_mp_p2_meanstd.csv")
show("calvspt_chicago_p2_meanstd.csv")
show("calvspt_austin_p2austin_meanstd.csv")


===== probbase_chicago_p2_4way_meanstd.csv =====
                           MAE                 MAE.1                RMSE               RMSE.1               RMSLE                RMSLE.1                WAPE               WAPE.1                  R2                   R2.1                 CSI                 CSI.1                CRPS                CRPS.1             pinball             pinball.1          coverage80           coverage80.1         sharpness80       sharpness80.1
NaN                       mean                   std                mean                  std                mean                    std                mean                  std                mean                    std                mean                   std                mean                   std                mean                   std                mean                    std                mean                 std
condition                  NaN                   NaN                 NaN                  

In [103]:
import os, json
R=os.environ["HASI_RESULTS_DIR"]
for f in ["summary_p2cc_chicago_condcal.json","summary_p2cc_austin_condcal.json"]:
    d=json.load(open(os.path.join(R,f)))
    print(f"\n===== {f} =====")
    ms=d.get("mean_std",{})
    for cond in ms:
        row=ms[cond]
        keys=["coverage80","cond_gap","top_decile_coverage","low_bucket","med_bucket","high_bucket",
              "per_node_coverage_std","sharpness80"]
        cells=[f"{k}={row.get(k+'_mean',float('nan')):.3f}±{row.get(k+'_std',float('nan')):.3f}" for k in keys if k+'_mean' in row]
        print(f"  {cond:12s} "+"  ".join(cells))
# p2 stats
import os
sp=os.path.join(R,"p2_stats_summary.json")
if os.path.exists(sp):
    s=json.load(open(sp)); print("\n===== p2_stats_summary keys ====="); print(list(s.keys()))
    print(json.dumps(s,indent=0)[:1500])


===== summary_p2cc_chicago_condcal.json =====

===== summary_p2cc_austin_condcal.json =====

===== p2_stats_summary keys =====
['friedman', 'bootstrap_95']
{
"friedman": {
"chi2": 22.866666666666667,
"p": 0.0017981621925030713,
"k": 8,
"n_blocks": 5,
"mean_ranks": {
"HA": 2.2,
"LSTM": 6.6,
"STGCN": 6.8,
"GraphWaveNet": 1.8,
"DCRNN": 5.8,
"MTGNN": 2.8,
"InformerOnly": 4.8,
"HASI-Net": 5.2
},
"cd_alpha0.1": 6.090419837696953,
"sig_rank_gaps": []
},
"bootstrap_95": {
"chi_coverage80": {
"mean": 0.8407132549989692,
"lo": 0.8345083487940631,
"hi": 0.8485157699443414
},
"chi_mae_calibrated": {
"mean": 2.6900179386138916,
"lo": 2.6747684478759766,
"hi": 2.7066784381866453
},
"chi_mae_point": {
"mean": 2.6351623058319094,
"lo": 2.616543102264404,
"hi": 2.6489900588989257
},
"mp_mae_transfer": {
"mean": 46.51843032836914,
"lo": 45.06552658081055,
"hi": 47.68985824584961
},
"mp_mae_scratch": {
"mean": 47.038805389404295,
"lo": 46.05065994262695,
"hi": 47.80251007080078
}
}
}


In [104]:
import os, json
R=os.environ["HASI_RESULTS_DIR"]
d=json.load(open(os.path.join(R,"summary_p2cc_chicago_condcal.json")))
print("top keys:", list(d.keys()))
print(json.dumps(d,indent=1)[:2200])

top keys: ['tag', 'dataset', 'seeds', 'conditions']
{
 "tag": "p2cc",
 "dataset": "chicago",
 "seeds": [
  42,
  1,
  2,
  3,
  4
 ],
 "conditions": {
  "calibrated": {
   "coverage80_mean": 0.840702947845805,
   "conditional_gap_mean": 0.11546024326710562,
   "low_mean": 0.9034694661374983,
   "med_mean": 0.7256341275941584,
   "high_mean": 0.8576929096885271,
   "top_decile_mean": 0.9153386454183267,
   "per_node_coverage_std_mean": 0.07187768828462307,
   "sharpness80_mean": 11.668160247802735
  },
  "conformal": {
   "coverage80_mean": 0.7864667078952793,
   "conditional_gap_mean": 0.3363545816733068,
   "low_mean": 0.8346434515426961,
   "med_mean": 0.8448501152959264,
   "high_mean": 0.6800125215213649,
   "top_decile_mean": 0.46364541832669326,
   "per_node_coverage_std_mean": 0.1871731236796727,
   "sharpness80_mean": 7.330486411521852
  }
 }
}


In [105]:
import os, json, pandas as pd
R=os.environ["HASI_RESULTS_DIR"]
print("=== p2_stats_summary.json ===")
d=json.load(open(os.path.join(R,"p2_stats_summary.json")))
print(json.dumps(d,indent=1)[:2500])

=== p2_stats_summary.json ===
{
 "friedman": {
  "chi2": 22.866666666666667,
  "p": 0.0017981621925030713,
  "k": 8,
  "n_blocks": 5,
  "mean_ranks": {
   "HA": 2.2,
   "LSTM": 6.6,
   "STGCN": 6.8,
   "GraphWaveNet": 1.8,
   "DCRNN": 5.8,
   "MTGNN": 2.8,
   "InformerOnly": 4.8,
   "HASI-Net": 5.2
  },
  "cd_alpha0.1": 6.090419837696953,
  "sig_rank_gaps": []
 },
 "bootstrap_95": {
  "chi_coverage80": {
   "mean": 0.8407132549989692,
   "lo": 0.8345083487940631,
   "hi": 0.8485157699443414
  },
  "chi_mae_calibrated": {
   "mean": 2.6900179386138916,
   "lo": 2.6747684478759766,
   "hi": 2.7066784381866453
  },
  "chi_mae_point": {
   "mean": 2.6351623058319094,
   "lo": 2.616543102264404,
   "hi": 2.6489900588989257
  },
  "mp_mae_transfer": {
   "mean": 46.51843032836914,
   "lo": 45.06552658081055,
   "hi": 47.68985824584961
  },
  "mp_mae_scratch": {
   "mean": 47.038805389404295,
   "lo": 46.05065994262695,
   "hi": 47.80251007080078
  }
 }
}


In [106]:
import os, pandas as pd
R=os.environ["HASI_RESULTS_DIR"]
for f in ["probbase_chicago_p2_4way_meanstd.csv","probbase_austin_p2austin_4way_meanstd.csv",
          "transfer_p2_meanstd.csv","transfer_p2chi_aus_meanstd.csv",
          "robust_mp_p2_meanstd.csv","calvspt_chicago_p2_meanstd.csv",
          "calvspt_austin_p2austin_meanstd.csv"]:
    print("\n===",f,"===")
    df=pd.read_csv(os.path.join(R,f),index_col=0)
    print(df.to_string())


=== probbase_chicago_p2_4way_meanstd.csv ===
                           MAE                 MAE.1                RMSE               RMSE.1               RMSLE                RMSLE.1                WAPE               WAPE.1                  R2                   R2.1                 CSI                 CSI.1                CRPS                CRPS.1             pinball             pinball.1          coverage80           coverage80.1         sharpness80       sharpness80.1
NaN                       mean                   std                mean                  std                mean                    std                mean                  std                mean                    std                mean                   std                mean                   std                mean                   std                mean                    std                mean                 std
condition                  NaN                   NaN                 NaN                  NaN 

In [107]:
import os, json
R=os.environ["HASI_RESULTS_DIR"]
d=json.load(open(os.path.join(R,"summary_p2cc_austin_condcal.json")))
print(json.dumps(d["conditions"],indent=1))

{
 "calibrated": {
  "coverage80_mean": 0.7902380952380952,
  "conditional_gap_mean": 0.13806637806637806,
  "low_mean": 0.8344062153163152,
  "med_mean": 0.7472222222222222,
  "high_mean": 0.7833131801692865,
  "top_decile_mean": 0.7873015873015872,
  "per_node_coverage_std_mean": 0.03550083515370108,
  "sharpness80_mean": 36.534417724609376
 },
 "conformal": {
  "coverage80_mean": 0.7930952380952381,
  "conditional_gap_mean": 0.226984126984127,
  "low_mean": 0.8293007769145394,
  "med_mean": 0.825,
  "high_mean": 0.7230955259975816,
  "top_decile_mean": 0.573015873015873,
  "per_node_coverage_std_mean": 0.16810016183611542,
  "sharpness80_mean": 14.21293638762495
 }
}


In [108]:
import os
R=os.environ["HASI_RESULTS_DIR"]
fs=sorted(os.listdir(R))
for f in fs:
    if "dm" in f.lower() or "dieb" in f.lower() or "stats" in f.lower():
        print(f)

dm_chicago_p1_meanstd.csv
dm_chicago_p1_perseed.csv
dm_chicago_p1_seed1_GraphWaveNet.json
dm_chicago_p1_seed1_HA.json
dm_chicago_p1_seed1_HASI-Net.json
dm_chicago_p1_seed2_GraphWaveNet.json
dm_chicago_p1_seed2_HA.json
dm_chicago_p1_seed2_HASI-Net.json
dm_chicago_p1_seed3_GraphWaveNet.json
dm_chicago_p1_seed3_HA.json
dm_chicago_p1_seed3_HASI-Net.json
dm_chicago_p1_seed42_GraphWaveNet.json
dm_chicago_p1_seed42_HA.json
dm_chicago_p1_seed42_HASI-Net.json
dm_chicago_p1_seed4_GraphWaveNet.json
dm_chicago_p1_seed4_HA.json
dm_chicago_p1_seed4_HASI-Net.json
p1_multihorizon_stats.json
p2_stats_summary.json


In [109]:
import os, json, pandas as pd
R=os.environ["HASI_RESULTS_DIR"]
d=json.load(open(os.path.join(R,"dm_chicago_p1_seed42_HASI-Net.json")))
print(json.dumps(d,indent=1)[:1500])
print("\n=== meanstd ===")
df=pd.read_csv(os.path.join(R,"dm_chicago_p1_meanstd.csv"),index_col=0)
print(df.to_string())

{
 "err": [
  [
   2.2829525470733643,
   2.1781959533691406,
   0.0,
   3.353893280029297,
   0.8093633651733398,
   1.5377769470214844,
   0.26483362913131714,
   2.5479354858398438,
   0.7660129070281982,
   5.533939361572266,
   0.0,
   3.2146835327148438,
   0.9813871383666992,
   2.4857101440429688,
   0.04990275949239731,
   1.5519256591796875,
   0.3510221242904663,
   0.07983016967773438,
   0.0,
   0.5091180801391602,
   1.9570846557617188,
   4.2405242919921875,
   0.07444561272859573,
   7.112573623657227,
   0.7279177904129028,
   2.8331260681152344,
   0.0,
   0.28375244140625,
   0.30195140838623047,
   7.035259246826172,
   0.24886757135391235,
   7.047004699707031,
   0.2039487212896347,
   0.7717986106872559,
   0.023921210318803787,
   1.5792768001556396,
   0.5508716702461243,
   3.1125030517578125,
   0.7938010394573212,
   0.2603316307067871,
   0.5685900449752808,
   0.5360088348388672,
   0.01560894027352333,
   3.2556886672973633,
   0.0944891944527626,
   1.29

In [110]:
import os, subprocess
repo="/content/spatio-temporal-crime"
print("exists:", os.path.isdir(repo))
r=subprocess.run(["git","-C",repo,"remote","-v"],capture_output=True,text=True)
print("remote:", r.stdout, r.stderr)
r=subprocess.run(["git","-C",repo,"status","-sb"],capture_output=True,text=True)
print("status:", r.stdout, r.stderr)
r=subprocess.run(["git","-C",repo,"log","--oneline","-3"],capture_output=True,text=True)
print("log:", r.stdout, r.stderr)
print("HASI_RESULTS_DIR:", os.environ.get("HASI_RESULTS_DIR"))

exists: True
remote: origin	https://github.com/vivekjindal24/spatio-temporal-crime.git (fetch)
origin	https://github.com/vivekjindal24/spatio-temporal-crime.git (push)
 
status: ## master...origin/master
?? data/austin_council_districts.geojson
 
log: e813b57 feat(p1): hotspot reframe -- event verification + top-k Hit@k
3b251a9 fix(p2): import _train_zero_fraction in prob_baselines
0d74957 feat(p2): conditional calibration (calibrated vs conformal, per-bucket/node)
 
HASI_RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results


In [111]:
import os, json, pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]
FIG="/content/spatio-temporal-crime/paper_b/figures"
os.makedirs(FIG, exist_ok=True)

# ---- shared style ----
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
C_CAL="#1f5fa0"; C_CF="#c04546"; C_QR="#2a8c4a"; C_PT="#6c6c6c"

def condcal(city):
    f=f"summary_p2cc_{city}_condcal.json"
    d=json.load(open(os.path.join(R,f)))["conditions"]
    return d
chi=condcal("chicago"); aus=condcal("austin")

def condcal_fig(city, d, fname):
    cal=d["calibrated"]; cf=d["conformal"]
    buckets=["low","med","high","top_decile"]
    labels=["Low","Medium","High","Top-decile"]
    cal_v=[cal[b+"_mean"] for b in buckets]
    cf_v=[cf[b+"_mean"] for b in buckets]
    x=np.arange(len(labels)); w=.38
    fig,ax=plt.subplots(figsize=(6.2,3.4))
    b1=ax.bar(x-w/2, cal_v, w, label="Calibrated (HASI-Net)", color=C_CAL)
    b2=ax.bar(x+w/2, cf_v, w, label="Split conformal", color=C_CF)
    ax.axhline(0.80, ls="--", c="k", lw=1, alpha=.7, label="Nominal 80%")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Empirical coverage"); ax.set_ylim(0,1.0)
    ax.set_title(f"Conditional coverage by crime-level bucket — {city.capitalize()}")
    for b in [b1,b2]:
        for r in b:
            ax.text(r.get_x()+r.get_width()/2, r.get_height()+.012,
                    f"{r.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    ax.legend(fontsize=8, loc="lower left", ncol=3)
    fig.tight_layout(); fig.savefig(os.path.join(FIG,fname)); plt.close(fig)
    print("wrote",fname)

condcal_fig("chicago", chi, "condcal_chicago.png")
condcal_fig("austin", aus, "condcal_austin.png")

wrote condcal_chicago.png
wrote condcal_austin.png


In [112]:
import os, json, pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]; FIG="/content/spatio-temporal-crime/paper_b/figures"
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
C_CAL="#1f5fa0"; C_CF="#c04546"; C_QR="#2a8c4a"; C_PT="#6c6c6c"

def load4(city, tag):
    df=pd.read_csv(os.path.join(R,f"probbase_{city}_{tag}_4way_meanstd.csv"),index_col=0)
    # rows: condition; the row label is at index, first two rows are junk (NaN header)
    df=df.drop(index=["NaN","condition"],errors="ignore")
    return df
chi4=load4("chicago","p2"); aus4=load4("austin","p2austin")

# ---- probbase figure: coverage80 + CRPS for 4 conditions, Chicago ----
def probbase_fig(df, fname, title):
    conds=["calibrated","conformal","quantreg","point"]
    cov=[df.loc[c,"coverage80"] if c in df.index else np.nan for c in conds]
    crps=[df.loc[c,"CRPS"] if c in df.index else np.nan for c in conds]
    mae=[df.loc[c,"MAE"] if c in df.index else np.nan for c in conds]
    x=np.arange(len(conds)); w=.38
    fig,ax=plt.subplots(figsize=(6.2,3.4))
    bars=ax.bar(x-w/2, cov, w, color=[C_CAL,C_CF,C_QR,C_PT], label="Coverage@80")
    ax.axhline(0.80, ls="--", c="k", lw=1, alpha=.7)
    ax.set_ylabel("Coverage@80"); ax.set_ylim(0,1.0)
    ax.set_xticks(x); ax.set_xticklabels(["Calibrated","Conformal","QuantReg","Point"])
    for r in bars:
        if not np.isnan(r.get_height()):
            ax.text(r.get_x()+r.get_width()/2, r.get_height()+.012,
                    f"{r.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    ax2=ax.twinx(); ax2.grid(False)
    ax2.bar(x+w/2, crps, w, color=[C_CAL,C_CF,C_QR,C_PT], alpha=.45, hatch="//", label="CRPS")
    ax2.set_ylabel("CRPS (lower better)")
    ax.set_title(title)
    h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
    ax.legend(h1+h2,l1+l2,fontsize=8,loc="upper right")
    fig.tight_layout(); fig.savefig(os.path.join(FIG,fname)); plt.close(fig); print("wrote",fname)

probbase_fig(chi4,"probbase_chicago.png","Probabilistic baselines — Chicago (5 seeds)")
probbase_fig(aus4,"probbase_austin.png","Probabilistic baselines — Austin (5 seeds)")

TypeError: 'value' must be an instance of str or bytes, not a float

In [113]:
import os, pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]; FIG="/content/spatio-temporal-crime/paper_b/figures"
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
C_CAL="#1f5fa0"; C_CF="#c04546"; C_QR="#2a8c4a"
def load4(city, tag):
    df=pd.read_csv(os.path.join(R,f"probbase_{city}_{tag}_4way_meanstd.csv"),index_col=0)
    df=df.drop(index=["NaN","condition"],errors="ignore")
    return df
chi4=load4("chicago","p2"); aus4=load4("austin","p2austin")

def probbase_fig(df, fname, title):
    conds=["calibrated","conformal","quantreg"]
    labels=["Calibrated\n(HASI-Net)","Split\nconformal","Quantile\nregression"]
    cov=[float(df.loc[c,"coverage80"]) for c in conds]
    crps=[float(df.loc[c,"CRPS"]) for c in conds]
    cols=[C_CAL,C_CF,C_QR]
    x=np.arange(len(conds)); w=.38
    fig,ax=plt.subplots(figsize=(6.2,3.5))
    bars=ax.bar(x-w/2, cov, w, color=cols, label="Coverage@80")
    ax.axhline(0.80, ls="--", c="k", lw=1, alpha=.7)
    ax.set_ylabel("Coverage@80"); ax.set_ylim(0,1.0)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    for r in bars:
        ax.text(r.get_x()+r.get_width()/2, r.get_height()+.012,
                f"{r.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    ax2=ax.twinx(); ax2.grid(False)
    ax2.bar(x+w/2, crps, w, color=cols, alpha=.45, hatch="//", label="CRPS")
    ax2.set_ylabel("CRPS (lower better)")
    ax.set_title(title)
    h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
    ax.legend(h1+h2,l1+l2,fontsize=8,loc="upper right")
    fig.tight_layout(); fig.savefig(os.path.join(FIG,fname)); plt.close(fig); print("wrote",fname)
probbase_fig(chi4,"probbase_chicago.png","Probabilistic baselines — Chicago (5 seeds)")
probbase_fig(aus4,"probbase_austin.png","Probabilistic baselines — Austin (5 seeds)")

wrote probbase_chicago.png
wrote probbase_austin.png


In [114]:
import os, pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]; FIG="/content/spatio-temporal-crime/paper_b/figures"
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
C_SC="#888888"; C_TF="#1f5fa0"

# ---- transfer Chicago->Austin ----
df=pd.read_csv(os.path.join(R,"transfer_p2chi_aus_meanstd.csv"),index_col=0)
df=df.drop(index=["NaN","condition"],errors="ignore")
conds=["scratch","transfer"]; labels=["Scratch\n(Austin only)","Transfer\n(Chicago$\\to$Austin)"]
mae=[float(df.loc[c,"MAE"]) for c in conds]; mae_s=[float(df.loc[c,"MAE.1"]) for c in conds]
cov=[float(df.loc[c,"coverage80"]) for c in conds]; cov_s=[float(df.loc[c,"coverage80.1"]) for c in conds]
x=np.arange(2); w=.38
fig,(axA,axB)=plt.subplots(1,2,figsize=(7.0,3.2))
axA.bar(x-w/2,[mae[0]],[w],color=C_SC,label="Scratch"); axA.bar(x+w/2,[mae[1]],[w],color=C_TF,label="Transfer")
axA.errorbar(x-w/2,[mae[0]],yerr=[mae_s[0]],fmt="none",c="k",capsize=4)
axA.errorbar(x+w/2,[mae[1]],yerr=[mae_s[1]],fmt="none",c="k",capsize=4)
axA.set_xticks(x); axA.set_xticklabels(["Scratch","Transfer"]); axA.set_ylabel("MAE (lower better)")
axA.set_title("Point accuracy")
for i,(m,s) in enumerate(zip(mae,mae_s)):
    axA.text(x[i]+(-w/2 if i==0 else w/2), m+s+.06, f"{m:.2f}", ha="center", fontsize=8)
axA.legend(fontsize=8)
axB.bar(x-w/2,[cov[0]],[w],color=C_SC); axB.bar(x+w/2,[cov[1]],[w],color=C_TF)
axB.errorbar(x-w/2,[cov[0]],yerr=[cov_s[0]],fmt="none",c="k",capsize=4)
axB.errorbar(x+w/2,[cov[1]],yerr=[cov_s[1]],fmt="none",c="k",capsize=4)
axB.axhline(0.80,ls="--",c="k",lw=1,alpha=.7); axB.set_ylim(0,1.0)
axB.set_xticks(x); axB.set_xticklabels(["Scratch","Transfer"]); axB.set_ylabel("Coverage@80")
axB.set_title("Marginal calibration")
for i,(c,s) in enumerate(zip(cov,cov_s)):
    axB.text(x[i]+(-w/2 if i==0 else w/2), c+s+.02, f"{c:.2f}", ha="center", fontsize=8)
fig.suptitle("Cross-region transfer: Chicago $\\to$ Austin",y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG,"transfer_austin.png"),bbox_inches="tight"); plt.close(fig)
print("wrote transfer_austin.png")

# ---- robustness MP ----
df=pd.read_csv(os.path.join(R,"robust_mp_p2_meanstd.csv"),index_col=0)
df=df.drop(index=["NaN","mask_frac"],errors="ignore")
df=df.astype(float)
masks=[0.0,0.1,0.25,0.5]
mae=[df.loc[m,"MAE"] for m in masks]; cov=[df.loc[m,"coverage80"] for m in masks]
fig,axL=plt.subplots(figsize=(6.2,3.4)); axR=axL.twinx(); axR.grid(False)
axL.plot(masks,mae,"-o",color=C_TF,lw=2,label="MAE")
axL.set_xlabel("Missing-data fraction (random mask)"); axL.set_ylabel("MAE",color=C_TF)
axR.plot(masks,cov,"-s",color=C_CF,lw=2,label="Coverage@80")
axR.axhline(0.80,ls="--",c="k",lw=1,alpha=.7); axR.set_ylabel("Coverage@80",color=C_CF); axR.set_ylim(0,1)
axL.set_title("Missing-data robustness — Madhya Pradesh panel")
h1,l1=axL.get_legend_handles_labels(); h2,l2=axR.get_legend_handles_labels()
axL.legend(h1+h2,l1+l2,fontsize=8,loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(FIG,"robustness_mp.png")); plt.close(fig)
print("wrote robustness_mp.png")
import os
print("paper_b/figures now:", sorted(os.listdir(FIG)))

ValueError: 'x' and 'y' must have the same size

In [115]:
import os, pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]; FIG="/content/spatio-temporal-crime/paper_b/figures"
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
C_SC="#888888"; C_TF="#1f5fa0"

# ---- transfer Chicago->Austin (2 bars per panel) ----
df=pd.read_csv(os.path.join(R,"transfer_p2chi_aus_meanstd.csv"),index_col=0)
df=df.drop(index=["NaN","condition"],errors="ignore")
g=lambda c,k: float(df.loc[c,k])
m_s,m_t=g("scratch","MAE"),g("transfer","MAE")
ms_s,ms_t=g("scratch","MAE.1"),g("transfer","MAE.1")
c_s,c_t=g("scratch","coverage80"),g("transfer","coverage80")
cs_s,cs_t=g("scratch","coverage80.1"),g("transfer","coverage80.1")
fig,(axA,axB)=plt.subplots(1,2,figsize=(7.0,3.2))
x0,x1=0,1
axA.bar(x0,m_s,color=C_SC,label="Scratch"); axA.bar(x1,m_t,color=C_TF,label="Transfer")
axA.errorbar([x0,x1],[m_s,m_t],yerr=[ms_s,ms_t],fmt="none",c="k",capsize=4)
axA.set_xticks([x0,x1]); axA.set_xticklabels(["Scratch","Transfer"]); axA.set_ylabel("MAE (lower better)")
axA.set_title("Point accuracy")
axA.text(x0,m_s+ms_s+.06,f"{m_s:.2f}",ha="center",fontsize=8)
axA.text(x1,m_t+ms_t+.06,f"{m_t:.2f}",ha="center",fontsize=8); axA.legend(fontsize=8)
axB.bar(x0,c_s,color=C_SC); axB.bar(x1,c_t,color=C_TF)
axB.errorbar([x0,x1],[c_s,c_t],yerr=[cs_s,cs_t],fmt="none",c="k",capsize=4)
axB.axhline(0.80,ls="--",c="k",lw=1,alpha=.7); axB.set_ylim(0,1.0)
axB.set_xticks([x0,x1]); axB.set_xticklabels(["Scratch","Transfer"]); axB.set_ylabel("Coverage@80")
axB.set_title("Marginal calibration")
axB.text(x0,c_s+cs_s+.02,f"{c_s:.2f}",ha="center",fontsize=8)
axB.text(x1,c_t+cs_t+.02,f"{c_t:.2f}",ha="center",fontsize=8)
fig.suptitle("Cross-region transfer: Chicago $\\to$ Austin",y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG,"transfer_austin.png"),bbox_inches="tight"); plt.close(fig)
print("wrote transfer_austin.png")

# ---- robustness MP ----
df=pd.read_csv(os.path.join(R,"robust_mp_p2_meanstd.csv"),index_col=0)
df=df.drop(index=["NaN","mask_frac"],errors="ignore").astype(float)
masks=[0.0,0.1,0.25,0.5]
mae=[df.loc[m,"MAE"] for m in masks]; cov=[df.loc[m,"coverage80"] for m in masks]
fig,axL=plt.subplots(figsize=(6.2,3.4)); axR=axL.twinx(); axR.grid(False)
axL.plot(masks,mae,"-o",color=C_TF,lw=2,label="MAE")
axL.set_xlabel("Missing-data fraction (random mask)"); axL.set_ylabel("MAE",color=C_TF)
axR.plot(masks,cov,"-s",color="#c04546",lw=2,label="Coverage@80")
axR.axhline(0.80,ls="--",c="k",lw=1,alpha=.7); axR.set_ylabel("Coverage@80",color="#c04546"); axR.set_ylim(0,1)
axL.set_title("Missing-data robustness — Madhya Pradesh panel")
h1,l1=axL.get_legend_handles_labels(); h2,l2=axR.get_legend_handles_labels()
axL.legend(h1+h2,l1+l2,fontsize=8,loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(FIG,"robustness_mp.png")); plt.close(fig)
print("wrote robustness_mp.png")
print("paper_b/figures now:", sorted(os.listdir(FIG)))

wrote transfer_austin.png


ValueError: could not convert string to float: 'mean'

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]; FIG="/content/spatio-temporal-crime/paper_b/figures"
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
# robust: parse safely
df=pd.read_csv(os.path.join(R,"robust_mp_p2_meanstd.csv"),index_col=0)
print("raw index:", list(df.index))
df["MAE_num"]=pd.to_numeric(df["MAE"],errors="coerce")
df=df[df["MAE_num"].notna()]
df["mask_num"]=pd.to_numeric(df.index,errors="coerce")
df=df.sort_values("mask_num")
masks=df["mask_num"].tolist(); mae=df["MAE_num"].tolist()
cov=pd.to_numeric(df["coverage80"],errors="coerce").tolist()
print("parsed:", list(zip(masks,mae,cov)))
fig,axL=plt.subplots(figsize=(6.2,3.4)); axR=axL.twinx(); axR.grid(False)
axL.plot(masks,mae,"-o",color="#1f5fa0",lw=2,label="MAE")
axL.set_xlabel("Missing-data fraction (random mask)"); axL.set_ylabel("MAE",color="#1f5fa0")
axR.plot(masks,cov,"-s",color="#c04546",lw=2,label="Coverage@80")
axR.axhline(0.80,ls="--",c="k",lw=1,alpha=.7); axR.set_ylabel("Coverage@80",color="#c04546"); axR.set_ylim(0,1)
axL.set_title("Missing-data robustness — Madhya Pradesh panel")
h1,l1=axL.get_legend_handles_labels(); h2,l2=axR.get_legend_handles_labels()
axL.legend(h1+h2,l1+l2,fontsize=8,loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(FIG,"robustness_mp.png")); plt.close(fig)
print("wrote robustness_mp.png")
print("paper_b/figures now:", sorted(os.listdir(FIG)))

In [116]:
import os, subprocess
repo="/content/spatio-temporal-crime"
# check gitignore for figures/results
gi=os.path.join(repo,".gitignore")
print("=== .gitignore ===")
print(open(gi).read() if os.path.exists(gi) else "(none)")
print("=== is paper_b/figures tracked? ===")
r=subprocess.run(["git","-C",repo,"ls-files","paper_b/figures"],capture_output=True,text=True)
print(repr(r.stdout), r.stderr)
print("=== existing paper_b tracked files ===")
r=subprocess.run(["git","-C",repo,"ls-files","paper_b"],capture_output=True,text=True)
print(r.stdout)

=== .gitignore ===
# python
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/

# local agent config (not research)
.claude/

# data: commit the public source files (NCRB CSVs, 2011 census, district
# shapefile, Chicago community-areas GeoJSON) so Colab runs are reproducible.
# Keep ignoring the regenerable Chicago Socrata parquet cache (re-fetched live).
data/*.parquet

# results (regenerable by the Colab runs; keep csv/json, drop heavy/regenerable)
results/*.pt
results/*.log
results/*.png

# latex build artifacts (keep .tex, .pdf, figures)
*.aux
*.bbl
*.blg
*.out
*.synctex.gz
*.fdb_latexmk
*.fls

# logs (latex, run logs)
*.log

# os
.DS_Store
=== is paper_b/figures tracked? ===
'paper_b/figures/architecture.png\npaper_b/figures/channel_weights_mp.png\npaper_b/figures/choropleth_mp.png\npaper_b/figures/comparison_CSI_mp.png\npaper_b/figures/comparison_MAE_mp.png\npaper_b/figures/district_risk_mp.png\npaper_b/figures/pred_vs_actual_mp.png\npaper_b/figures/training_curves_mp.png\n' 
=== exis

In [117]:
import os, pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R=os.environ["HASI_RESULTS_DIR"]; FIG="/content/spatio-temporal-crime/paper_b/figures"
plt.rcParams.update({"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "figure.dpi":150,"savefig.dpi":150,"axes.axisbelow":True})
df=pd.read_csv(os.path.join(R,"robust_mp_p2_meanstd.csv"),index_col=0)
df["MAE_num"]=pd.to_numeric(df["MAE"],errors="coerce")
df=df[df["MAE_num"].notna()].copy()
df["mask_num"]=pd.to_numeric(df.index,errors="coerce")
df=df.sort_values("mask_num")
masks=df["mask_num"].tolist(); mae=df["MAE_num"].tolist()
cov=pd.to_numeric(df["coverage80"],errors="coerce").tolist()
print("parsed:", list(zip(masks,[round(m,2) for m in mae],[round(c,3) for c in cov])))
fig,axL=plt.subplots(figsize=(6.2,3.4)); axR=axL.twinx(); axR.grid(False)
axL.plot(masks,mae,"-o",color="#1f5fa0",lw=2,label="MAE")
axL.set_xlabel("Missing-data fraction (random mask)"); axL.set_ylabel("MAE",color="#1f5fa0")
axR.plot(masks,cov,"-s",color="#c04546",lw=2,label="Coverage@80")
axR.axhline(0.80,ls="--",c="k",lw=1,alpha=.7); axR.set_ylabel("Coverage@80",color="#c04546"); axR.set_ylim(0,1)
axL.set_title("Missing-data robustness — Madhya Pradesh panel")
h1,l1=axL.get_legend_handles_labels(); h2,l2=axR.get_legend_handles_labels()
axL.legend(h1+h2,l1+l2,fontsize=8,loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(FIG,"robustness_mp.png")); plt.close(fig)
print("wrote robustness_mp.png")
print("P2 figures present:", sorted(f for f in os.listdir(FIG) if f in
 {"condcal_chicago.png","condcal_austin.png","probbase_chicago.png","probbase_austin.png","transfer_austin.png","robustness_mp.png","architecture.png"}))

parsed: [(0.0, 47.04, 0.158), (0.1, 55.59, 0.091), (0.25, 62.03, 0.229), (0.5, 56.74, 0.723)]
wrote robustness_mp.png
P2 figures present: ['architecture.png', 'condcal_austin.png', 'condcal_chicago.png', 'probbase_austin.png', 'probbase_chicago.png', 'robustness_mp.png', 'transfer_austin.png']


In [118]:
import os, subprocess
repo="/content/spatio-temporal-crime"
FIG=os.path.join(repo,"paper_b","figures")
# stale MP-policy figures from the old paper_b (calibration paper won't use them)
stale=["channel_weights_mp.png","choropleth_mp.png","comparison_CSI_mp.png",
       "comparison_MAE_mp.png","district_risk_mp.png","pred_vs_actual_mp.png",
       "training_curves_mp.png"]
new=["condcal_chicago.png","condcal_austin.png","probbase_chicago.png",
     "probbase_austin.png","transfer_austin.png","robustness_mp.png"]
def run(cmd):
    r=subprocess.run(cmd,capture_output=True,text=True,cwd=repo)
    return r.stdout+r.stderr
print(run(["git","rm"]+[f"paper_b/figures/{s}" for s in stale]))
print(run(["git","add"]+[f"paper_b/figures/{n}" for n in new]))
print(run(["git","status","-s","paper_b/figures"]))

rm 'paper_b/figures/channel_weights_mp.png'
rm 'paper_b/figures/choropleth_mp.png'
rm 'paper_b/figures/comparison_CSI_mp.png'
rm 'paper_b/figures/comparison_MAE_mp.png'
rm 'paper_b/figures/district_risk_mp.png'
rm 'paper_b/figures/pred_vs_actual_mp.png'
rm 'paper_b/figures/training_curves_mp.png'


D  paper_b/figures/channel_weights_mp.png
D  paper_b/figures/choropleth_mp.png
D  paper_b/figures/comparison_CSI_mp.png
D  paper_b/figures/comparison_MAE_mp.png
A  paper_b/figures/condcal_austin.png
A  paper_b/figures/condcal_chicago.png
D  paper_b/figures/district_risk_mp.png
D  paper_b/figures/pred_vs_actual_mp.png
A  paper_b/figures/probbase_austin.png
A  paper_b/figures/probbase_chicago.png
A  paper_b/figures/robustness_mp.png
D  paper_b/figures/training_curves_mp.png
A  paper_b/figures/transfer_austin.png



In [119]:
import subprocess
repo="/content/spatio-temporal-crime"
def run(cmd):
    r=subprocess.run(cmd,capture_output=True,text=True,cwd=repo)
    return (r.stdout+r.stderr).strip()
msg=("docs(paper_b): add real P2 calibration figures; drop stale MP-policy figures\n\n"
 "Generated on Colab from persisted P2 result files (Drive):\n"
 " conditional coverage (condcal_chicago/austin), 4-way probabilistic baselines\n"
 " (probbase_chicago/austin), Chicago->Austin transfer, MP robustness.\n"
 " Removes the old MP case-study figures no longer used by the calibration paper.\n\n"
 "Co-Authored-By: Claude <noreply@anthropic.com>")
print(run(["git","commit","-m",msg]))
print("--- log ---")
print(run(["git","log","--oneline","-2"]))
print("--- push ---")
print(run(["git","push","origin","master"]))

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@d1916256ef6f.(none)')
--- log ---
e813b57 feat(p1): hotspot reframe -- event verification + top-k Hit@k
3b251a9 fix(p2): import _train_zero_fraction in prob_baselines
--- push ---
fatal: could not read Username for 'https://github.com': No such device or address


In [120]:
import shutil, subprocess
for t in ["pdflatex","latexmk","xelatex","lualatex","bibtex"]:
    print(t, "->", shutil.which(t))
# also check apt texlive
r=subprocess.run(["ls","/usr/share/texlive"],capture_output=True,text=True)
print("texlive dir:", r.stdout[:200] or "(none)", r.stderr[:80])

pdflatex -> None
latexmk -> None
xelatex -> None
lualatex -> None
bibtex -> None
texlive dir: (none) ls: cannot access '/usr/share/texlive': No such file or directory



In [121]:
import os
print("kernel alive; HASI_RESULTS_DIR =", os.environ.get("HASI_RESULTS_DIR"))
R=os.environ.get("HASI_RESULTS_DIR","")
print("results dir exists:", os.path.isdir(R) if R else "no env")
if R and os.path.isdir(R):
    print("key P2 files present:",
          all(os.path.exists(os.path.join(R,f)) for f in
              ["summary_p2cc_chicago_condcal.json","probbase_chicago_p2_4way_meanstd.csv",
               "transfer_p2chi_aus_meanstd.csv","robust_mp_p2_meanstd.csv","p2_stats_summary.json"]))
# check if the figures I generated earlier are still in the Colab checkout
FIG="/content/spatio-temporal-crime/paper_b/figures"
print("colab figures:", sorted(os.listdir(FIG)) if os.path.isdir(FIG) else "GONE")

kernel alive; HASI_RESULTS_DIR = /content/drive/MyDrive/spatio-temporal-crime_results
results dir exists: True
key P2 files present: True
colab figures: ['architecture.png', 'condcal_austin.png', 'condcal_chicago.png', 'probbase_austin.png', 'probbase_chicago.png', 'robustness_mp.png', 'transfer_austin.png']


In [122]:
import os
FIG="/content/spatio-temporal-crime/paper_b/figures"
new=["condcal_chicago.png","condcal_austin.png","probbase_chicago.png",
     "probbase_austin.png","transfer_austin.png","robustness_mp.png"]
tot=0
for n in new:
    s=os.path.getsize(os.path.join(FIG,n)); tot+=s
    print(f"{n:28s} {s:7d} B  ({s/1024:.1f} KB)")
import base64
b64=sum(len(base64.b64encode(open(os.path.join(FIG,n),'rb').read())) for n in new)
print(f"\ntotal raw {tot/1024:.1f} KB  | base64 {b64/1024:.1f} KB ({b64} chars)")

condcal_chicago.png            48891 B  (47.7 KB)
condcal_austin.png             47626 B  (46.5 KB)
probbase_chicago.png           65250 B  (63.7 KB)
probbase_austin.png            60034 B  (58.6 KB)
transfer_austin.png            47201 B  (46.1 KB)
robustness_mp.png              72926 B  (71.2 KB)

total raw 333.9 KB  | base64 445.2 KB (455912 chars)


In [123]:
import os, glob
rd = os.environ.get("HASI_RESULTS_DIR","/content/drive/MyDrive/spatio-temporal-crime_results")
print("HASI_RESULTS_DIR:", rd)
need = ["summary_p2cc_chicago_condcal.json","summary_p2cc_austin_condcal.json",
        "probbase_chicago_p2_4way_meanstd.csv","probbase_austin_p2austin_4way_meanstd.csv",
        "transfer_p2chi_aus_meanstd.csv","robust_mp_p2_meanstd.csv"]
for f in need:
    p=os.path.join(rd,f)
    print(f, "EXISTS" if os.path.exists(p) else "MISSING", os.path.getsize(p) if os.path.exists(p) else "")

HASI_RESULTS_DIR: /content/drive/MyDrive/spatio-temporal-crime_results
summary_p2cc_chicago_condcal.json EXISTS 884
summary_p2cc_austin_condcal.json EXISTS 869
probbase_chicago_p2_4way_meanstd.csv EXISTS 1710
probbase_austin_p2austin_4way_meanstd.csv EXISTS 1669
transfer_p2chi_aus_meanstd.csv EXISTS 1001
robust_mp_p2_meanstd.csv EXISTS 1709


In [ ]:
import os, json
rd = os.environ.get("HASI_RESULTS_DIR","/content/drive/MyDrive/spatio-temporal-crime_results")
need = ["summary_p2cc_chicago_condcal.json","summary_p2cc_austin_condcal.json",
        "probbase_chicago_p2_4way_meanstd.csv","probbase_austin_p2austin_4way_meanstd.csv",
        "transfer_p2chi_aus_meanstd.csv","robust_mp_p2_meanstd.csv"]
for f in need:
    p=os.path.join(rd,f)
    print("\n========== FILE:", f, "==========")
    print(open(p).read())

In [124]:
import os
rd = os.environ.get("HASI_RESULTS_DIR","/content/drive/MyDrive/spatio-temporal-crime_results")
need = ["summary_p2cc_chicago_condcal.json","summary_p2cc_austin_condcal.json",
        "probbase_chicago_p2_4way_meanstd.csv","probbase_austin_p2austin_4way_meanstd.csv",
        "transfer_p2chi_aus_meanstd.csv","robust_mp_p2_meanstd.csv"]
for f in need:
    p=os.path.join(rd,f)
    print("\n========== FILE:", f, "==========")
    print(open(p).read())


========== FILE: summary_p2cc_chicago_condcal.json ==========
{
  "tag": "p2cc",
  "dataset": "chicago",
  "seeds": [
    42,
    1,
    2,
    3,
    4
  ],
  "conditions": {
    "calibrated": {
      "coverage80_mean": 0.840702947845805,
      "conditional_gap_mean": 0.11546024326710562,
      "low_mean": 0.9034694661374983,
      "med_mean": 0.7256341275941584,
      "high_mean": 0.8576929096885271,
      "top_decile_mean": 0.9153386454183267,
      "per_node_coverage_std_mean": 0.07187768828462307,
      "sharpness80_mean": 11.668160247802735
    },
    "conformal": {
      "coverage80_mean": 0.7864667078952793,
      "conditional_gap_mean": 0.3363545816733068,
      "low_mean": 0.8346434515426961,
      "med_mean": 0.8448501152959264,
      "high_mean": 0.6800125215213649,
      "top_decile_mean": 0.46364541832669326,
      "per_node_coverage_std_mean": 0.1871731236796727,
      "sharpness80_mean": 7.330486411521852
    }
  }
}

========== FILE: summary_p2cc_austin_condcal.json =